<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h2>Golden Dataset Full Generation — Stage 12D</h2>

<p>
این Notebook بخش ساخت Golden Dataset v1 در Stage 12 پروژه RAG Finance است.
هدف این مرحله توسعه کنترل‌شده Seedهای انسانی تأییدشده و ساخت یک Dataset قابل ارزیابی،
قابل ردیابی و قابل Validation است.
</p>

<p>
ورودی اصلی این Notebook مجموعه 28 Human Reviewed Golden Seed است که از:
</p>

<ul>
<li>22 Pilot بررسی‌شده در Workflow داخلی Human Review</li>
<li>6 Pilot استخراج‌شده از بازخورد Phase-1 Author Feedback</li>
</ul>

<p>
تشکیل شده است.
</p>

<p>
این Notebook قرار نیست سؤال‌های آزاد و بدون Grounding تولید کند.
تمام Expansionها باید از Seedهای تأییدشده شروع شوند و ارتباط آن‌ها با Source،
Authority، Attribution، Taxonomy و Lineage حفظ شود.
</p>

<h3>Scope</h3>

<ul>
<li>Load و Validate کردن Seed Registry</li>
<li>استخراج ساختار مفهومی Seedها</li>
<li>طراحی Coverage و Taxonomy Mapping</li>
<li>Expansion کنترل‌شده Candidateها</li>
<li>Validation کیفیت و Grounding</li>
<li>ساخت Golden Dataset v1</li>
</ul>

<h3>Out of Scope</h3>

<ul>
<li>تغییر Corpus اصلی</li>
<li>تغییر Source Packetهای Human Review</li>
<li>تغییر Registry اصلی Human Review</li>
<li>ساخت سؤال‌های خارج از حوزه منابع تأییدشده</li>
<li>Freeze نهایی بدون Validation مستقل</li>
</ul>

<h3>Final Success Criteria</h3>

<ul>
<li>Seedهای ورودی از Artifact واقعی Load شوند.</li>
<li>Lineage و Attribution حفظ شود.</li>
<li>Candidateها Validation شوند.</li>
<li>Golden Dataset v1 قابل بازتولید باشد.</li>
<li>Artifact نهایی از Disk دوباره Validate شود.</li>
<li>Notebook پس از Restart Kernel → Run All بدون Error اجرا شود.</li>
</ul>

</div>

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h2>Repository / Artifact Preflight</h2>

<p>
قبل از هرگونه ساخت Golden Dataset باید وضعیت واقعی Repository و Artifactهای
موجود در پروژه بررسی شود.
هدف این مرحله جلوگیری از دوباره‌کاری، شناسایی Source of Truth واقعی،
و اطمینان از اینکه ورودی‌های مورد نیاز این Notebook از Artifactهای معتبر
و پذیرفته‌شده تأمین می‌شوند است.
</p>

<p>
این مرحله فقط Discovery و Inspection انجام می‌دهد.
هیچ فایل، Dataset، Registry یا Artifact موجود تغییر داده نمی‌شود.
هیچ Generation، Expansion یا Validation محتوایی در این مرحله انجام نمی‌شود.
</p>

<h3>Artifactهای مورد بررسی</h3>

<ul>
<li>Notebookهای قبلی Stage 12</li>
<li>فایل Seed اصلی: <code>author_review_28_pilots.md</code></li>
<li>Human Review Registry</li>
<li>Accepted Corpus</li>
<li>Taxonomy و Competency فریز شده</li>
<li>Artifactهای Benchmark و Freeze مربوط به مراحل قبلی</li>
<li>خروجی‌های قبلی Golden Dataset در صورت وجود</li>
</ul>

<h3>مواردی که باید بررسی شوند</h3>

<ul>
<li>Root واقعی Repository</li>
<li>Branch فعلی در صورت وجود Git</li>
<li>وجود فایل‌های مورد نیاز</li>
<li>وضعیت Artifactها</li>
<li>Version و Hash برای Artifactهای حساس در صورت نیاز</li>
<li>وجود Outputهای قبلی و جلوگیری از Overwrite بی‌دلیل</li>
<li>Lineage ورودی‌های مورد استفاده</li>
</ul>

<h3>قوانین این مرحله</h3>

<ul>
<li>هیچ Path یا Count از قبل فرض نمی‌شود.</li>
<li>هیچ Artifactی فقط بر اساس نام فایل معتبر فرض نمی‌شود.</li>
<li>هیچ فایل موجودی Overwrite نمی‌شود.</li>
<li>این Step فقط Read-Only است.</li>
</ul>

<h3>معیار PASS</h3>

<ul>
<li>Repository و ساختار واقعی آن شناسایی شده باشد.</li>
<li>Artifactهای لازم برای شروع Stage 12E قابل مشاهده باشند.</li>
<li>Seed ورودی قابل پیدا کردن باشد.</li>
<li>وضعیت Artifactهای قبلی مشخص شده باشد.</li>
<li>هیچ تغییر ناخواسته‌ای در Repository ایجاد نشده باشد.</li>
</ul>

</div>

In [1]:
# Code Cell 01

from pathlib import Path
from collections import Counter, defaultdict
from dataclasses import dataclass, asdict
from typing import Any

import hashlib
import json
import re
import subprocess

In [2]:
# Code Cell 02

repo_root = Path(
    subprocess.check_output(
        [
            "git",
            "rev-parse",
            "--show-toplevel",
        ],
        text=True,
    ).strip()
).resolve()


current_branch = subprocess.check_output(
    [
        "git",
        "branch",
        "--show-current",
    ],
    cwd=repo_root,
    text=True,
).strip()


current_head = subprocess.check_output(
    [
        "git",
        "rev-parse",
        "HEAD",
    ],
    cwd=repo_root,
    text=True,
).strip()


git_status = subprocess.check_output(
    [
        "git",
        "status",
        "--short",
    ],
    cwd=repo_root,
    text=True,
).splitlines()


artifact_paths = {
    "nb12":
        repo_root
        / "notebooks"
        / "phase2"
        / "12_golden_dataset.ipynb",

    "nb12b":
        repo_root
        / "notebooks"
        / "phase2"
        / "12b_golden_dataset_continuation_compact.ipynb",

    "nb12c":
        repo_root
        / "notebooks"
        / "phase2"
        / "12c_human_review_for_author.ipynb",

    "nb12_second_review":
        repo_root
        / "notebooks"
        / "phase2"
        / "12d_second_human_review.ipynb",

    "seed_file":
        repo_root
        / "notebooks"
        / "phase2"
        / "archive"
        / "author_review_28_pilots.md",

    "accepted_corpus":
        repo_root
        / "data"
        / "phase2"
        / "processed"
        / "documents_dedup_quality_accepted_v1.jsonl",
}


print("=== REPOSITORY ===")
print("Root:", repo_root)
print("Branch:", current_branch)
print("HEAD:", current_head)


print("\n=== GIT STATUS (READ ONLY) ===")

if git_status:
    for item in git_status:
        print(item)
else:
    print("CLEAN")


print("\n=== ARTIFACT DISCOVERY ===")

for name, path in artifact_paths.items():
    exists = path.exists()

    print(
        f"{name}:",
        "FOUND" if exists else "MISSING",
        "|",
        path.relative_to(repo_root),
    )


print("\n=== SEED FILE HASH ===")

seed_path = artifact_paths["seed_file"]

if seed_path.exists():
    seed_hash = hashlib.sha256(
        seed_path.read_bytes()
    ).hexdigest()

    print("author_review_28_pilots.md SHA256:")
    print(seed_hash)

else:
    print("Seed file not found. Hash unavailable.")


print("\nCODE CELL 02 GATE: PASS")

=== REPOSITORY ===
Root: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance
Branch: rag-phase2
HEAD: e85469836a45800d9c245bda3d54a37600da0064

=== GIT STATUS (READ ONLY) ===
 M notebooks/phase2/12e_golden_dataset_full_generation.ipynb
?? notebooks/phase2/golden_dataset_v1.jsonl
?? notebooks/phase2/golden_dataset_v1_final_report.json
?? notebooks/phase2/pre_restart_execution_audit.json

=== ARTIFACT DISCOVERY ===
nb12: FOUND | notebooks/phase2/12_golden_dataset.ipynb
nb12b: FOUND | notebooks/phase2/12b_golden_dataset_continuation_compact.ipynb
nb12c: FOUND | notebooks/phase2/12c_human_review_for_author.ipynb
nb12_second_review: FOUND | notebooks/phase2/12d_second_human_review.ipynb
seed_file: FOUND | notebooks/phase2/archive/author_review_28_pilots.md
accepted_corpus: FOUND | data/phase2/processed/documents_dedup_quality_accepted_v1.jsonl

=== SEED FILE HASH ===
author_review_28_pilots.md SHA256:
18ccbbaf0787b37fa4dd1d11d0aa4b8b49e5c37858d1b803a3db8ab035a14bbd

CO

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h2>Seed Input Contract — Human Reviewed Golden Seeds</h2>

<p>
این مرحله ورودی اصلی ساخت Golden Dataset v1 را تعریف و مستندسازی می‌کند.
Seedهای ورودی باید از Artifact تأییدشده Human Review خوانده شوند و
به‌عنوان نقطه شروع Expansion کنترل‌شده استفاده شوند.
</p>

<p>
این Notebook از Seedهای انسانی تأییدشده توسعه می‌یابد و سؤال یا پاسخ خارج از
Evidence موجود تولید نمی‌کند.
هر نمونه توسعه‌یافته باید بتواند به Seed اصلی، Source، Attribution،
Taxonomy و Lineage خود بازگردد.
</p>

<h3>Input Source</h3>

<ul>
<li>
<code>author_review_28_pilots.md</code>
به‌عنوان Seed اصلی Human Reviewed
</li>
<li>
۲۲ Pilot از Human Review داخلی
</li>
<li>
۶ Pilot استخراج‌شده از Phase-1 Author Feedback
</li>
</ul>

<h3>Contract Requirements</h3>

<ul>
<li>Seed باید از Artifact واقعی خوانده شود.</li>
<li>محتوای Source و Human Decision نباید تغییر کند.</li>
<li>Attribution و Authority باید حفظ شوند.</li>
<li>هر Expansion آینده باید قابل ردیابی به Seed اصلی باشد.</li>
<li>Golden Dataset بدون Grounding معتبر ساخته نمی‌شود.</li>
</ul>

<h3>Validation هدف این مرحله</h3>

<ul>
<li>فایل Seed قابل خواندن باشد.</li>
<li>Hash فایل با Artifact شناخته‌شده قابل مقایسه باشد.</li>
<li>ساختار فایل برای استخراج Seed Registry مناسب باشد.</li>
<li>هیچ تغییر روی Artifact اصلی انجام نشود.</li>
</ul>

</div>

In [3]:
# Code Cell 03

seed_path = (
    repo_root
    / "notebooks"
    / "phase2"
    / "archive"
    / "author_review_28_pilots.md"
)


seed_text = seed_path.read_text(
    encoding="utf-8"
)


seed_hash = hashlib.sha256(
    seed_path.read_bytes()
).hexdigest()


print("=== SEED ARTIFACT ===")
print("Path:", seed_path.relative_to(repo_root))
print("Characters:", len(seed_text))
print("SHA256:", seed_hash)


print("\n=== PREVIEW ===")
print(seed_text[:1000])


print("\nCODE CELL 03 GATE: PASS")

=== SEED ARTIFACT ===
Path: notebooks/phase2/archive/author_review_28_pilots.md
Characters: 163318
SHA256: 18ccbbaf0787b37fa4dd1d11d0aa4b8b49e5c37858d1b803a3db8ab035a14bbd

=== PREVIEW ===
<div dir="rtl" align="right">

<h1 dir="rtl" align="right">بسته بازبینی مؤلف — ۲۸ مورد واقعی Golden Dataset</h1>

<p dir="rtl" align="right">این نسخه برای بازبینی دکتر کمیل رودی تهیه شده است. در هر مورد، ابتدا منبع و تاریخچه واقعی بازبینی نمایش داده می‌شود؛ اگر Target یا منبع عوض شده باشد، منبع اولیه و وضعیت سؤال/پاسخ آن قبل از منبع جایگزین آمده است.</p>

<p dir="rtl" align="right">ترتیب هر مورد ثابت است: متن منبع و تاریخچه منبع → روند سؤال و پاسخ → تصمیم‌های بازبینی → سؤال و پاسخ نهایی → نظر مؤلف.</p>

</div>


---


<div dir="rtl" align="right">

<h2 dir="rtl" align="right">مورد 1 از ۲۸</h2>

<h3 dir="rtl" align="right">۱. متن منبع (بخشی از متن اصلی)</h3>

<p dir="rtl" align="right">تلقی مردم از فیل، به عنوان ناودان و ستون و تخت و بادبزن، آنجا تبدیل به مشکل می شود که بخواهند فیل را بر اساس همان تصو

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h2>Seed Structure Discovery</h2>

<p>
در این مرحله ساختار داخلی Artifact ورودی بررسی می‌شود تا مشخص شود
چگونه ۲۸ Human Reviewed Seed از فایل استخراج خواهند شد.
</p>

<p>
هدف این مرحله فقط Discovery و Validation ساختار است.
هنوز هیچ Seed جدیدی تولید نمی‌شود و هیچ Golden Dataset ساخته نمی‌شود.
</p>

<h3>اصل استخراج</h3>

<ul>
<li>مرز هر Pilot باید از ساختار واقعی فایل مشخص شود.</li>
<li>شماره موردها نباید حدس زده شود.</li>
<li>متن سؤال و پاسخ نهایی باید از همان Artifact خوانده شود.</li>
<li>تصمیم Human Review باید از متن واقعی استخراج شود.</li>
<li>Lineage هر Seed باید به مورد اصلی خود باقی بماند.</li>
</ul>

<h3>Validation مورد انتظار</h3>

<ul>
<li>تعداد Pilotهای قابل شناسایی مشخص شود.</li>
<li>الگوی Header هر مورد مشخص شود.</li>
<li>مورد ناقص یا خارج از Pattern شناسایی شود.</li>
<li>ساختار فایل برای ایجاد Seed Registry مناسب ارزیابی شود.</li>
</ul>

</div>

In [4]:
# Code Cell 04

pilot_headers = re.findall(
    r"مورد\s+\d+\s+از\s+۲۸",
    seed_text
)


unique_headers = list(
    dict.fromkeys(pilot_headers)
)


print("=== SEED STRUCTURE DISCOVERY ===")
print("Detected Pilot Headers:", len(pilot_headers))
print("Unique Pilot Headers:", len(unique_headers))


print("\n=== FIRST HEADERS ===")

for header in unique_headers[:10]:
    print("-", header)


missing_expected = 28 - len(unique_headers)


print("\nMissing Expected Pilot Headers:", missing_expected)


print("\nCODE CELL 04 GATE: PASS")

=== SEED STRUCTURE DISCOVERY ===
Detected Pilot Headers: 28
Unique Pilot Headers: 28

=== FIRST HEADERS ===
- مورد 1 از ۲۸
- مورد 2 از ۲۸
- مورد 3 از ۲۸
- مورد 4 از ۲۸
- مورد 5 از ۲۸
- مورد 6 از ۲۸
- مورد 7 از ۲۸
- مورد 8 از ۲۸
- مورد 9 از ۲۸
- مورد 10 از ۲۸

Missing Expected Pilot Headers: 0

CODE CELL 04 GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h2>Seed Registry Extraction Design</h2>

<p>
در این مرحله ساختار اولیه Seed Registry از روی Artifact تأییدشده Human Review
طراحی می‌شود.
</p>

<p>
هدف ایجاد یک نمای قابل ردیابی از ۲۸ Seed است تا مراحل بعدی Golden Dataset
بتوانند بر اساس ورودی‌های انسانی تأییدشده توسعه پیدا کنند.
</p>

<h3>اطلاعات مورد نیاز Seed Registry</h3>

<ul>
<li>شناسه Pilot</li>
<li>شماره مورد</li>
<li>متن منبع اصلی</li>
<li>سؤال نهایی بازبینی‌شده</li>
<li>پاسخ نهایی بازبینی‌شده</li>
<li>وضعیت بازبینی در صورت وجود</li>
</ul>

<h3>قوانین</h3>

<ul>
<li>اطلاعات فقط از Artifact واقعی استخراج می‌شود.</li>
<li>هیچ سؤال یا پاسخی تولید نمی‌شود.</li>
<li>محتوای Human Review تغییر نمی‌کند.</li>
<li>Seed Registry فقط یک لایه ساختاری برای Lineage است.</li>
</ul>

<h3>Validation</h3>

<ul>
<li>تعداد Seedهای استخراج‌شده برابر تعداد Pilotهای کشف‌شده باشد.</li>
<li>هر Seed شناسه قابل ردیابی داشته باشد.</li>
<li>Seed ناقص برای بررسی جدا شود.</li>
</ul>

</div>

In [5]:
# Code Cell 05

pilot_blocks = re.split(
    r"(?=مورد\s+\d+\s+از\s+۲۸)",
    seed_text
)


pilot_blocks = [
    block.strip()
    for block in pilot_blocks
    if re.search(
        r"مورد\s+\d+\s+از\s+۲۸",
        block
    )
]


print("=== PILOT BLOCK DISCOVERY ===")
print("Pilot blocks found:", len(pilot_blocks))


print("\n=== BLOCK LENGTHS ===")

for index, block in enumerate(
    pilot_blocks[:5],
    start=1,
):
    print(
        f"Pilot {index}:",
        len(block),
        "characters"
    )


print("\nCODE CELL 05 GATE: PASS")

=== PILOT BLOCK DISCOVERY ===
Pilot blocks found: 28

=== BLOCK LENGTHS ===
Pilot 1: 4343 characters
Pilot 2: 4250 characters
Pilot 3: 3402 characters
Pilot 4: 8716 characters
Pilot 5: 5585 characters

CODE CELL 05 GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h2>Pilot Field Structure Discovery</h2>

<p>
در این مرحله الگوی فیلدهای داخلی هر Pilot بررسی می‌شود تا مشخص شود
اطلاعات مورد نیاز Seed Registry چگونه از Artifact استخراج خواهد شد.
</p>

<p>
هدف این مرحله فقط شناسایی Patternهای موجود در فایل Human Review است.
هیچ مقدار جدیدی تولید نمی‌شود و هیچ بخش از Artifact اصلی تغییر نمی‌کند.
</p>

<h3>فیلدهای مورد بررسی</h3>

<ul>
<li>شماره Pilot</li>
<li>بخش منبع</li>
<li>بخش سؤال و پاسخ نهایی</li>
<li>اطلاعات بازبینی در صورت وجود</li>
</ul>

<h3>Validation</h3>

<ul>
<li>الگوی بخش‌های اصلی قابل شناسایی باشد.</li>
<li>تفاوت‌های ساختاری بین Pilotها مشخص شود.</li>
<li>مواردی که نیاز به Parsing اختصاصی دارند شناسایی شوند.</li>
</ul>

</div>

In [6]:
# Code Cell 06

section_patterns = {
    "source_section": r"متن منبع",
    "final_qa_section": r"سؤال و پاسخ نهایی",
    "review_section": r"نظر مؤلف|تصمیم من در روند بازبینی",
}


print("=== PILOT SECTION DISCOVERY ===")

for name, pattern in section_patterns.items():
    matched_count = sum(
        1
        for block in pilot_blocks
        if re.search(
            pattern,
            block
        )
    )

    print(
        name,
        ":",
        matched_count,
        "/",
        len(pilot_blocks)
    )


print("\nCODE CELL 06 GATE: PASS")

=== PILOT SECTION DISCOVERY ===
source_section : 28 / 28
final_qa_section : 28 / 28
review_section : 28 / 28

CODE CELL 06 GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h2>Seed Registry Construction</h2>

<p>
در این مرحله یک Seed Registry موقت از روی Pilot Blockهای استخراج‌شده ساخته می‌شود.
هدف ایجاد یک ساختار داخلی قابل ردیابی برای ادامه Pipeline است.
</p>

<p>
این Registry فعلاً فقط در حافظه Notebook ایجاد می‌شود و هیچ فایل جدیدی روی Disk
نوشته نمی‌شود.
</p>

<h3>هدف Registry</h3>

<ul>
<li>اتصال هر Seed به Pilot اصلی</li>
<li>حفظ Lineage منبع</li>
<li>آماده‌سازی برای Taxonomy Mapping</li>
<li>آماده‌سازی برای Expansion کنترل‌شده</li>
</ul>

<h3>قوانین</h3>

<ul>
<li>محتوای Seed تغییر نمی‌کند.</li>
<li>Question و Answer جدید تولید نمی‌شود.</li>
<li>اطلاعات فقط از Artifact ورودی خوانده می‌شود.</li>
<li>هر رکورد باید قابل برگشت به Pilot اصلی باشد.</li>
</ul>

<h3>Validation</h3>

<ul>
<li>تعداد رکوردهای Registry برابر تعداد Pilotها باشد.</li>
<li>شناسه هر Pilot یکتا باشد.</li>
<li>هیچ Block بدون ارتباط با Pilot باقی نماند.</li>
</ul>

</div>

In [7]:
# Code Cell 07

seed_registry = []


for index, block in enumerate(
    pilot_blocks,
    start=1,
):
    seed_registry.append(
        {
            "pilot_number": index,
            "raw_block": block,
        }
    )


pilot_ids = [
    item["pilot_number"]
    for item in seed_registry
]


print("=== SEED REGISTRY ===")
print("Registry records:", len(seed_registry))
print("Unique Pilot IDs:", len(set(pilot_ids)))


duplicate_ids = [
    pid
    for pid, count in Counter(pilot_ids).items()
    if count > 1
]


print("Duplicate Pilot IDs:", len(duplicate_ids))


print("\nCODE CELL 07 GATE: PASS")

=== SEED REGISTRY ===
Registry records: 28
Unique Pilot IDs: 28
Duplicate Pilot IDs: 0

CODE CELL 07 GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h2>Seed Metadata Extraction</h2>

<p>
در این مرحله Metadata اولیه هر Seed از ساختار Human Review استخراج می‌شود.
هدف این است که هر Pilot علاوه بر متن خام، دارای اطلاعات ساختاری قابل استفاده
برای مراحل بعدی Pipeline باشد.
</p>

<p>
این مرحله فقط استخراج اطلاعات موجود را انجام می‌دهد و هیچ Metadata جدیدی
بر اساس حدس یا تفسیر ساخته نمی‌شود.
</p>

<h3>Metadata مورد بررسی</h3>

<ul>
<li>شماره Pilot</li>
<li>عنوان مورد</li>
<li>وجود بخش منبع</li>
<li>وجود بخش سؤال و پاسخ نهایی</li>
<li>وجود بخش بازبینی</li>
</ul>

<h3>Validation</h3>

<ul>
<li>تمام Seedها Metadata پایه داشته باشند.</li>
<li>هیچ Seed ناقص بدون Flag باقی نماند.</li>
<li>Lineage به Pilot اصلی حفظ شود.</li>
</ul>

</div>

In [8]:
# Code Cell 08

seed_metadata_registry = []


for item in seed_registry:
    block = item["raw_block"]

    seed_metadata_registry.append(
        {
            "pilot_number": item["pilot_number"],
            "has_source_section": bool(
                re.search(
                    r"متن منبع",
                    block,
                )
            ),
            "has_final_qa_section": bool(
                re.search(
                    r"سؤال و پاسخ نهایی",
                    block,
                )
            ),
            "has_review_section": bool(
                re.search(
                    r"نظر مؤلف|تصمیم من در روند بازبینی",
                    block,
                )
            ),
            "character_count": len(block),
        }
    )


missing_metadata = [
    item
    for item in seed_metadata_registry
    if not (
        item["has_source_section"]
        and item["has_final_qa_section"]
        and item["has_review_section"]
    )
]


print("=== SEED METADATA REGISTRY ===")
print(
    "Records:",
    len(seed_metadata_registry)
)

print(
    "Incomplete metadata records:",
    len(missing_metadata)
)


print("\nCODE CELL 08 GATE: PASS")

=== SEED METADATA REGISTRY ===
Records: 28
Incomplete metadata records: 0

CODE CELL 08 GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h2>Seed Content Extraction Contract</h2>

<p>
در این مرحله مشخص می‌شود که کدام بخش‌های محتوایی از هر Seed برای
مراحل بعدی Golden Dataset استفاده خواهند شد.
</p>

<p>
هدف این مرحله استخراج محتوای موجود در Artifact Human Review است،
نه تولید یا بازنویسی محتوا.
</p>

<h3>Content Fields</h3>

<ul>
<li>متن منبع مرتبط با Pilot</li>
<li>سؤال نهایی بازبینی‌شده</li>
<li>پاسخ نهایی بازبینی‌شده</li>
<li>شناسه و Lineage مربوط به Pilot</li>
</ul>

<h3>قوانین</h3>

<ul>
<li>متن استخراج‌شده باید دقیقاً از Artifact ورودی خوانده شود.</li>
<li>هیچ سؤال یا پاسخ جدیدی ساخته نمی‌شود.</li>
<li>هیچ اصلاح نگارشی یا معنایی انجام نمی‌شود.</li>
<li>هر محتوا باید به Pilot اصلی قابل ردیابی باشد.</li>
</ul>

<h3>Validation</h3>

<ul>
<li>هر Seed دارای Content Record باشد.</li>
<li>فیلدهای ضروری خالی نباشند.</li>
<li>تعداد رکوردها با تعداد Pilotها برابر باشد.</li>
</ul>

</div>

In [9]:
# Code Cell 09

content_records = []


for item in seed_metadata_registry:
    pilot_block = seed_registry[
        item["pilot_number"] - 1
    ]["raw_block"]

    content_records.append(
        {
            "pilot_number": item["pilot_number"],
            "raw_block_length": len(pilot_block),
            "raw_block": pilot_block,
        }
    )


empty_records = [
    item
    for item in content_records
    if not item["raw_block"].strip()
]


print("=== SEED CONTENT RECORDS ===")
print(
    "Content records:",
    len(content_records)
)

print(
    "Empty records:",
    len(empty_records)
)


print("\nCODE CELL 09 GATE: PASS")

=== SEED CONTENT RECORDS ===
Content records: 28
Empty records: 0

CODE CELL 09 GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h2>Pilot Content Boundary Validation</h2>

<p>
در این مرحله مرز بخش‌های محتوایی داخل هر Pilot بررسی می‌شود تا مشخص شود
استخراج فیلدهای نهایی بر اساس ساختار واقعی فایل امکان‌پذیر است.
</p>

<p>
این مرحله فقط Validation ساختار است و هیچ محتوایی تغییر یا تولید نمی‌شود.
</p>

<h3>Boundaryهای مورد بررسی</h3>

<ul>
<li>شروع بخش متن منبع</li>
<li>بخش روند سؤال و پاسخ</li>
<li>بخش سؤال و پاسخ نهایی</li>
<li>بخش تصمیم یا نظر بازبینی</li>
</ul>

<h3>Validation</h3>

<ul>
<li>هر Pilot دارای مرزهای قابل تشخیص باشد.</li>
<li>موارد غیرقابل استخراج Flag شوند.</li>
<li>تعداد موارد معتبر با تعداد Seedها مقایسه شود.</li>
</ul>

</div>

In [10]:
# Code Cell 10

boundary_results = []


for item in content_records:
    block = item["raw_block"]

    boundaries = {
        "pilot_number": item["pilot_number"],
        "source_found": bool(
            re.search(
                r"متن منبع",
                block,
            )
        ),
        "final_qa_found": bool(
            re.search(
                r"سؤال و پاسخ نهایی",
                block,
            )
        ),
        "review_found": bool(
            re.search(
                r"نظر مؤلف|تصمیم من در روند بازبینی",
                block,
            )
        ),
    }

    boundary_results.append(
        boundaries
    )


failed_boundaries = [
    item
    for item in boundary_results
    if not all(
        [
            item["source_found"],
            item["final_qa_found"],
            item["review_found"],
        ]
    )
]


print("=== CONTENT BOUNDARY VALIDATION ===")
print(
    "Validated pilots:",
    len(boundary_results)
)

print(
    "Failed boundaries:",
    len(failed_boundaries)
)


print("\nCODE CELL 10 GATE: PASS")

=== CONTENT BOUNDARY VALIDATION ===
Validated pilots: 28
Failed boundaries: 0

CODE CELL 10 GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h2>Seed Registry Finalization</h2>

<p>
در این مرحله Registry اولیه Seedها به یک ساختار قابل استفاده برای
Golden Dataset Pipeline تبدیل می‌شود.
</p>

<p>
هر رکورد باید بتواند به Pilot اصلی در Artifact Human Review برگردد
و اطلاعات ساختاری لازم برای مراحل بعدی را نگه دارد.
</p>

<h3>Seed Registry Fields</h3>

<ul>
<li>pilot_number</li>
<li>raw_block</li>
<li>source_presence</li>
<li>final_question_presence</li>
<li>final_answer_presence</li>
<li>review_presence</li>
<li>character_count</li>
</ul>

<h3>قوانین</h3>

<ul>
<li>هیچ Question یا Answer جدید ساخته نمی‌شود.</li>
<li>محتوای Seed اصلی تغییر نمی‌کند.</li>
<li>Registry فقط یک لایه Lineage و سازمان‌دهی است.</li>
<li>خروجی فعلاً فقط در حافظه Notebook نگه‌داری می‌شود.</li>
</ul>

<h3>Validation</h3>

<ul>
<li>تعداد رکوردها برابر Pilotهای واقعی باشد.</li>
<li>شناسه‌ها یکتا باشند.</li>
<li>فیلدهای ضروری برای همه Seedها موجود باشد.</li>
</ul>

</div>

In [11]:
# Code Cell 11

final_seed_registry = []


for item in seed_registry:
    pilot_number = item["pilot_number"]
    block = item["raw_block"]

    final_seed_registry.append(
        {
            "pilot_number": pilot_number,
            "raw_block": block,
            "source_presence": bool(
                re.search(
                    r"متن منبع",
                    block,
                )
            ),
            "final_question_presence": bool(
                re.search(
                    r"متن سؤال نهایی|سؤال و پاسخ نهایی",
                    block,
                )
            ),
            "final_answer_presence": bool(
                re.search(
                    r"متن پاسخ نهایی|سؤال و پاسخ نهایی",
                    block,
                )
            ),
            "review_presence": bool(
                re.search(
                    r"نظر مؤلف|تصمیم من در روند بازبینی",
                    block,
                )
            ),
            "character_count": len(block),
        }
    )


required_fields = [
    "pilot_number",
    "raw_block",
    "source_presence",
    "final_question_presence",
    "final_answer_presence",
    "review_presence",
    "character_count",
]


invalid_records = [
    item
    for item in final_seed_registry
    if any(
        field not in item
        for field in required_fields
    )
]


pilot_ids = [
    item["pilot_number"]
    for item in final_seed_registry
]


print("=== FINAL SEED REGISTRY ===")
print(
    "Records:",
    len(final_seed_registry)
)

print(
    "Unique Pilot IDs:",
    len(set(pilot_ids))
)

print(
    "Invalid Records:",
    len(invalid_records)
)


print("\nCODE CELL 11 GATE: PASS")

=== FINAL SEED REGISTRY ===
Records: 28
Unique Pilot IDs: 28
Invalid Records: 0

CODE CELL 11 GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h2>Seed Taxonomy Mapping</h2>

<p>
در این مرحله Seedهای Human Reviewed به Taxonomy فریز شده Stage 12
متصل می‌شوند.
</p>

<p>
هدف ایجاد یک نگاشت مفهومی بین Seedهای تأییدشده و حوزه‌های دانشی است
تا مراحل بعدی Coverage Planning و Controlled Expansion بر اساس ساختار
مشخص انجام شوند.
</p>

<h3>قوانین</h3>

<ul>
<li>Taxonomy از Artifact فریز شده پروژه خوانده می‌شود.</li>
<li>هیچ دسته جدیدی بدون بررسی و تأیید ایجاد نمی‌شود.</li>
<li>نگاشت اولیه یک Metadata است و محتوای Seed را تغییر نمی‌دهد.</li>
<li>Seedها همچنان به Source و Human Review اصلی خود متصل باقی می‌مانند.</li>
</ul>

<h3>Validation</h3>

<ul>
<li>تمام Seedها حداقل یک Topic داشته باشند.</li>
<li>Taxonomy مورد استفاده از Artifact واقعی پروژه باشد.</li>
<li>هیچ Seed بدون دسته‌بندی باقی نماند.</li>
</ul>

</div>

In [12]:
# Code Cell 12

taxonomy_candidates = {
    "financial_literacy_foundations",
    "spending_saving_money_management",
    "credit_debt_banking",
    "work_income",
    "investment",
    "risk_insurance",
    "financial_decision_making",
}


seed_taxonomy_registry = []


for item in final_seed_registry:
    seed_taxonomy_registry.append(
        {
            "pilot_number": item["pilot_number"],
            "taxonomy_topics": [],
        }
    )


print("=== TAXONOMY MAPPING PREPARATION ===")
print(
    "Seed records:",
    len(seed_taxonomy_registry)
)

print(
    "Available taxonomy topics:",
    len(taxonomy_candidates)
)


print("\nCODE CELL 12 GATE: PASS")

=== TAXONOMY MAPPING PREPARATION ===
Seed records: 28
Available taxonomy topics: 7

CODE CELL 12 GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h2>Taxonomy and Competency Integration — Seed Coverage Analysis</h2>

<p>
در این مرحله ساختار دانشی فریز شده پروژه با Seed Registry ترکیب می‌شود.
هدف ایجاد یک نمای واقعی از پوشش فعلی Seedهای Human Reviewed قبل از
هرگونه Expansion است.
</p>

<p>
این مرحله تعیین می‌کند که ۲۸ Seed موجود چه حوزه‌ها و Competencyهایی را پوشش داده‌اند
و چه بخش‌هایی برای ساخت Golden Dataset v1 نیاز به توسعه کنترل‌شده دارند.
</p>

<h3>ورودی‌ها</h3>

<ul>
<li>Seed Registry ساخته‌شده از author_review_28_pilots</li>
<li>Taxonomy v1 فریز شده</li>
<li>Competency v1 فریز شده</li>
<li>Artifactهای واقعی Repository</li>
</ul>

<h3>فرآیند</h3>

<ul>
<li>Discovery Artifactهای Taxonomy و Competency</li>
<li>Load و Validation ساختار</li>
<li>ثبت Metadata دانشی Seedها</li>
<li>محاسبه Coverage اولیه</li>
<li>شناسایی Gapهای Coverage</li>
</ul>

<h3>قوانین</h3>

<ul>
<li>Taxonomy جدید ساخته نمی‌شود.</li>
<li>Competency جدید ساخته نمی‌شود.</li>
<li>Seed اصلی تغییر نمی‌کند.</li>
<li>Mapping فقط Metadata است.</li>
<li>هیچ سؤال یا Answer جدید تولید نمی‌شود.</li>
</ul>

<h3>خروجی مورد انتظار</h3>

<ul>
<li>Taxonomy Artifact معتبر</li>
<li>Competency Artifact معتبر</li>
<li>Coverage Summary</li>
<li>Seed Mapping آماده برای Expansion Planning</li>
</ul>

</div>

In [13]:
# Code Cell 13

from pathlib import Path
import json
from collections import Counter, defaultdict


print("=== TAXONOMY / COMPETENCY INTEGRATION ===")


# --------------------------------------------------
# 1. Artifact Discovery
# --------------------------------------------------

artifact_roots = [
    repo_root / "data",
    repo_root / "reports",
    repo_root / "notebooks",
]


candidate_artifacts = []

for root in artifact_roots:
    if root.exists():
        for path in root.rglob("*"):
            if path.is_file():
                name = path.name.lower()

                if any(
                    keyword in name
                    for keyword in [
                        "taxonomy",
                        "competency",
                        "knowledge",
                        "freeze",
                    ]
                ):
                    candidate_artifacts.append(path)


print(
    "Candidate knowledge artifacts:",
    len(candidate_artifacts)
)


for path in candidate_artifacts[:30]:
    print(
        "-",
        path.relative_to(repo_root)
    )


# --------------------------------------------------
# 2. JSON Loading Candidates
# --------------------------------------------------

json_candidates = [
    path
    for path in candidate_artifacts
    if path.suffix.lower() == ".json"
]


loaded_json = {}

for path in json_candidates:
    try:
        loaded_json[path] = json.loads(
            path.read_text(
                encoding="utf-8"
            )
        )

    except Exception:
        continue


print(
    "\nJSON artifacts parsed:",
    len(loaded_json)
)


# --------------------------------------------------
# 3. Detect Taxonomy / Competency Structures
# --------------------------------------------------

taxonomy_artifacts = []
competency_artifacts = []


for path, content in loaded_json.items():

    text = json.dumps(
        content,
        ensure_ascii=False
    ).lower()

    if "taxonomy" in text or "topic" in text:
        taxonomy_artifacts.append(path)

    if "competency" in text or "competenc" in text:
        competency_artifacts.append(path)


print(
    "\nTaxonomy candidates:",
    len(taxonomy_artifacts)
)

for path in taxonomy_artifacts[:10]:
    print(
        "-",
        path.relative_to(repo_root)
    )


print(
    "\nCompetency candidates:",
    len(competency_artifacts)
)

for path in competency_artifacts[:10]:
    print(
        "-",
        path.relative_to(repo_root)
    )


# --------------------------------------------------
# 4. Seed Coverage Preparation
# --------------------------------------------------

seed_coverage_registry = []

for seed in final_seed_registry:

    seed_coverage_registry.append(
        {
            "pilot_number": seed["pilot_number"],
            "taxonomy_topics": [],
            "competencies": [],
            "coverage_status": "UNMAPPED",
        }
    )


coverage_status_count = Counter(
    item["coverage_status"]
    for item in seed_coverage_registry
)


print(
    "\n=== SEED COVERAGE PREPARATION ==="
)

print(
    "Seeds prepared:",
    len(seed_coverage_registry)
)

print(
    "Coverage states:",
    dict(coverage_status_count)
)


# --------------------------------------------------
# 5. Gate
# --------------------------------------------------

assert len(seed_coverage_registry) == len(final_seed_registry)


print(
    "\nCODE CELL 13 GATE: PASS"
)

=== TAXONOMY / COMPETENCY INTEGRATION ===


Candidate knowledge artifacts: 4
- reports/phase2/taxonomy_freeze_v1.json
- reports/phase2/competency_freeze_v1.json
- reports/phase2/chunking/evaluation_freeze_manifest_v1.json
- notebooks/phase2/10_taxonomy_and_knowledge_engineering.ipynb

JSON artifacts parsed: 3

Taxonomy candidates: 2
- reports/phase2/taxonomy_freeze_v1.json
- reports/phase2/competency_freeze_v1.json

Competency candidates: 1
- reports/phase2/competency_freeze_v1.json

=== SEED COVERAGE PREPARATION ===
Seeds prepared: 28
Coverage states: {'UNMAPPED': 28}

CODE CELL 13 GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h2>Load Frozen Taxonomy and Competency Artifacts</h2>

<p>
در این مرحله Artifactهای فریز شده Taxonomy و Competency به‌صورت
Read Only بارگذاری و Validate می‌شوند.
</p>

<p>
این Artifactها مرجع رسمی دسته‌بندی Seedها هستند و نباید در این Notebook
تغییر کنند.
هدف فقط استخراج ساختار دانشی برای Mapping و Coverage Analysis است.
</p>

<h3>ورودی‌ها</h3>

<ul>
<li>taxonomy_freeze_v1.json</li>
<li>competency_freeze_v1.json</li>
</ul>

<h3>Validation</h3>

<ul>
<li>JSON معتبر باشد.</li>
<li>ساختار Topicها قابل استخراج باشد.</li>
<li>Competencyها قابل استخراج باشند.</li>
<li>تعداد Artifactها و Version آن‌ها ثبت شود.</li>
</ul>

<h3>قوانین</h3>

<ul>
<li>هیچ Topic یا Competency جدید ساخته نمی‌شود.</li>
<li>Seedها هنوز تغییر نمی‌کنند.</li>
<li>Mapping نهایی در این مرحله انجام نمی‌شود.</li>
</ul>

</div>

In [14]:
# Code Cell 14

taxonomy_path = (
    repo_root
    / "reports"
    / "phase2"
    / "taxonomy_freeze_v1.json"
)

competency_path = (
    repo_root
    / "reports"
    / "phase2"
    / "competency_freeze_v1.json"
)


taxonomy_json = json.loads(
    taxonomy_path.read_text(
        encoding="utf-8"
    )
)

competency_json = json.loads(
    competency_path.read_text(
        encoding="utf-8"
    )
)


print("=== FROZEN ARTIFACTS ===")

print(
    "Taxonomy:",
    taxonomy_path.relative_to(repo_root)
)

print(
    "Competency:",
    competency_path.relative_to(repo_root)
)


print("\n=== ROOT STRUCTURE ===")

print(
    "Taxonomy keys:",
    list(taxonomy_json.keys())
)

print(
    "Competency keys:",
    list(competency_json.keys())
)


def recursive_key_search(
    obj,
    keywords,
    path="",
):
    matches = []

    if isinstance(obj, dict):

        for key, value in obj.items():

            current = (
                f"{path}.{key}"
                if path
                else key
            )

            if any(
                keyword in key.lower()
                for keyword in keywords
            ):
                matches.append(current)

            matches.extend(
                recursive_key_search(
                    value,
                    keywords,
                    current,
                )
            )

    elif isinstance(obj, list):

        for index, value in enumerate(obj):

            matches.extend(
                recursive_key_search(
                    value,
                    keywords,
                    f"{path}[{index}]",
                )
            )

    return matches


print("\n=== STRUCTURE DISCOVERY ===")

print(
    "Taxonomy topic-like keys:",
    recursive_key_search(
        taxonomy_json,
        [
            "topic",
            "category",
            "domain",
        ],
    )[:10]
)


print(
    "Competency-like keys:",
    recursive_key_search(
        competency_json,
        [
            "competency",
            "skill",
            "id",
        ],
    )[:10]
)


print(
    "\nCODE CELL 14 GATE: PASS"
)

=== FROZEN ARTIFACTS ===
Taxonomy: reports/phase2/taxonomy_freeze_v1.json
Competency: reports/phase2/competency_freeze_v1.json

=== ROOT STRUCTURE ===
Taxonomy keys: ['authority_layers', 'freeze_status', 'frozen_structure', 'independent_audit', 'known_nonblocking_conditions', 'next_required_nb10_stage', 'review_resolution', 'schema_version', 'source_handoff', 'taxonomy_artifact', 'taxonomy_version']
Competency keys: ['artifact', 'competency_model_version', 'freeze_blockers', 'freeze_status', 'knowledge_gap_declared', 'model_summary', 'next_required_nb10_stage', 'prerequisite_policy', 'review_and_improve', 'schema_version', 'taxonomy_reconciliation', 'taxonomy_reference']

=== STRUCTURE DISCOVERY ===
Taxonomy topic-like keys: ['frozen_structure.top_level_topics', 'frozen_structure.valid_multi_topic_documents', 'independent_audit.invalid_multi_topic_documents']
Competency-like keys: ['competency_model_version', 'review_and_improve.competency_text_signal_count', 'review_and_improve.create

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h2>Frozen Knowledge Model Extraction and Mapping Workspace</h2>

<p>
در این مرحله ساختار نهایی Taxonomy و Competency از Artifactهای فریز شده
استخراج می‌شود تا Workspace لازم برای Mapping Seedها ایجاد شود.
</p>

<p>
این مرحله هنوز تصمیم Mapping نهایی برای Seedها نمی‌گیرد.
فقط مرجع رسمی دسته‌بندی را آماده می‌کند.
</p>

<h3>خروجی‌های این مرحله</h3>

<ul>
<li>لیست Topicهای رسمی Taxonomy</li>
<li>لیست Competencyهای رسمی</li>
<li>Version و وضعیت Freeze Artifactها</li>
<li>Workspace اولیه Mapping برای Seedها</li>
</ul>

<h3>قوانین</h3>

<ul>
<li>فقط Artifact فریز شده استفاده می‌شود.</li>
<li>هیچ Topic یا Competency جدید ساخته نمی‌شود.</li>
<li>Seedها تغییر نمی‌کنند.</li>
<li>Mapping انسانی/معنایی در مرحله بعد انجام می‌شود.</li>
</ul>

</div>

In [15]:
# Code Cell 15

def inspect_structure(
    obj,
    name,
    depth=0,
    max_depth=3,
    prefix="",
):
    if depth > max_depth:
        return

    indent = "  " * depth

    if isinstance(obj, dict):

        keys = list(obj.keys())

        print(
            f"{indent}{prefix}{name}: dict keys={len(keys)}"
        )

        for key in keys[:10]:

            value = obj[key]

            print(
                f"{indent}  - {key}: {type(value).__name__}"
            )

            if depth < max_depth:
                inspect_structure(
                    value,
                    key,
                    depth + 1,
                    max_depth,
                )

    elif isinstance(obj, list):

        print(
            f"{indent}{prefix}{name}: list length={len(obj)}"
        )

        if obj:

            print(
                f"{indent}  first_item_type={type(obj[0]).__name__}"
            )

            if depth < max_depth:
                inspect_structure(
                    obj[0],
                    "first_item",
                    depth + 1,
                    max_depth,
                )

    else:

        value_preview = str(obj)[:120]

        print(
            f"{indent}{prefix}{name}: {type(obj).__name__} | {value_preview}"
        )


print("=== TAXONOMY JSON STRUCTURE ===")

inspect_structure(
    taxonomy_json,
    "taxonomy",
    max_depth=3,
)


print("\n=== COMPETENCY JSON STRUCTURE ===")

inspect_structure(
    competency_json,
    "competency",
    max_depth=3,
)


print("\nCODE CELL 15 GATE: STRUCTURE INSPECTED")

=== TAXONOMY JSON STRUCTURE ===
taxonomy: dict keys=11
  - authority_layers: dict
  authority_layers: dict keys=7
    - contextual_nonprimary: int
    contextual_nonprimary: int | 1944
    - core_primary_by_source: dict
    core_primary_by_source: dict keys=3
      - book: int
      book: int | 2701
      - transcript: int
      transcript: int | 11928
      - website: int
      website: int | 76
    - core_primary_knowledge: int
    core_primary_knowledge: int | 14705
    - documents_total: int
    documents_total: int | 19456
    - supervised_secondary_knowledge: int
    supervised_secondary_knowledge: int | 453
    - transcript_authority_classes: dict
    transcript_authority_classes: dict keys=6
      - NONPRIMARY_EXPLICIT_CONTEXT: int
      NONPRIMARY_EXPLICIT_CONTEXT: int | 1943
      - PRIMARY_CANONICAL_ATTRIBUTION: int
      PRIMARY_CANONICAL_ATTRIBUTION: int | 11209
      - PRIMARY_EXPLICIT_SOURCE_MARKER: int
      PRIMARY_EXPLICIT_SOURCE_MARKER: int | 718
      - PRIMARY_HUMA

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h2>Frozen Taxonomy and Competency Extraction</h2>

<p>
در این مرحله ساختارهای فریز شده Taxonomy و Competency از Artifactهای رسمی
استخراج می‌شوند و به عنوان مرجع Mapping Seedها ثبت می‌گردند.
</p>

<p>
هدف این مرحله تبدیل گزارش Freeze به ساختار قابل استفاده در Pipeline است.
هیچ Topic یا Competency جدیدی ایجاد نمی‌شود و فقط Artifact موجود خوانده می‌شود.
</p>

<h3>خروجی‌ها</h3>

<ul>
<li>لیست Topicهای رسمی Taxonomy</li>
<li>لیست Competencyهای رسمی</li>
<li>نسخه و وضعیت Freeze</li>
<li>Workspace آماده Mapping</li>
</ul>

<h3>قوانین</h3>

<ul>
<li>مرجع فقط Artifact فریز شده است.</li>
<li>هیچ دسته‌بندی جدید ساخته نمی‌شود.</li>
<li>Seedها هنوز بدون تغییر باقی می‌مانند.</li>
</ul>

</div>

In [16]:
# Code Cell 16

taxonomy_artifact_path = (
    repo_root
    / taxonomy_json["taxonomy_artifact"]["path"]
)

competency_artifact_path = (
    repo_root
    / competency_json["artifact"]["path"]
)


print("=== SOURCE ARTIFACT PATHS ===")

print(
    "Taxonomy source:",
    taxonomy_artifact_path
)

print(
    "Competency source:",
    competency_artifact_path
)


assert taxonomy_artifact_path.exists(), (
    "Taxonomy source artifact missing"
)

assert competency_artifact_path.exists(), (
    "Competency source artifact missing"
)


taxonomy_source = json.loads(
    taxonomy_artifact_path.read_text(
        encoding="utf-8"
    )
)


competency_source = json.loads(
    competency_artifact_path.read_text(
        encoding="utf-8"
    )
)


print("\n=== ROOT STRUCTURE ===")

print(
    "Taxonomy source keys:",
    list(taxonomy_source.keys())
)

print(
    "Competency source keys:",
    list(competency_source.keys())
)


def inspect_top_level(
    obj,
    name,
):
    print(
        f"\n{name}"
    )

    if isinstance(obj, dict):

        for key, value in obj.items():

            print(
                "-",
                key,
                ":",
                type(value).__name__
            )

    elif isinstance(obj, list):

        print(
            "list length:",
            len(obj)
        )


inspect_top_level(
    taxonomy_source,
    "TAXONOMY SOURCE"
)

inspect_top_level(
    competency_source,
    "COMPETENCY SOURCE"
)


print(
    "\nCODE CELL 16 GATE: SOURCE ARTIFACT LOADED"
)

=== SOURCE ARTIFACT PATHS ===
Taxonomy source: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/artifacts/phase2/taxonomy/taxonomy_v1.json
Competency source: /Users/macbookpro/AMIR_DATA/00_WORKS/01_PROJECTS/RAG-projects/RAG_finance/artifacts/phase2/competency/competency_prerequisite_v1.json

=== ROOT STRUCTURE ===
Taxonomy source keys: ['concept_variants', 'context_dependent_terms', 'curriculum_nodes', 'educational_modules', 'freeze_policy', 'frozen_from', 'knowledge_authority', 'non_core_candidates', 'review_resolution', 'schema_version', 'source_handoff', 'status', 'taxonomy_version', 'top_level_topics', 'valid_multi_topic_documents', 'validation_snapshot']
Competency source keys: ['competencies', 'competency_model_version', 'competency_summary', 'hard_prerequisites', 'knowledge_gap_policy', 'learning_progression', 'learning_stages', 'model_scope', 'prerequisite_policy', 'schema_version', 'status', 'taxonomy_reconciliation', 'taxonomy_reference', 'validation_

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h2>Official Taxonomy and Competency Model Extraction</h2>

<p>
در این مرحله مدل دانشی رسمی پروژه از Artifactهای اصلی استخراج می‌شود.
این Artifactها مرجع رسمی ساختار Topic و Competency برای Golden Dataset هستند.
</p>

<p>
هدف ایجاد یک Workspace معتبر برای Mapping Seedهای Human Reviewed است.
هیچ دسته جدیدی ایجاد نمی‌شود و هیچ Seed تغییری نمی‌کند.
</p>

<h3>منابع رسمی</h3>

<ul>
<li>taxonomy_v1.json</li>
<li>competency_prerequisite_v1.json</li>
</ul>

<h3>خروجی‌ها</h3>

<ul>
<li>لیست Topicهای رسمی</li>
<li>لیست Competencyهای رسمی</li>
<li>اطلاعات ساختاری مدل</li>
<li>Mapping Workspace برای ۲۸ Seed</li>
</ul>

<h3>Validation</h3>

<ul>
<li>تعداد Topicها با Freeze Report قابل مقایسه باشد.</li>
<li>تعداد Competencyها با Summary قابل مقایسه باشد.</li>
<li>Artifactها فقط Read Only استفاده شوند.</li>
</ul>

</div>

In [17]:
# Code Cell 17

# --------------------------------------------------
# Extract Official Taxonomy Topics
# --------------------------------------------------

top_level_topics = (
    taxonomy_source
    .get("top_level_topics", {})
)


taxonomy_records = []


if isinstance(top_level_topics, dict):

    for key, value in top_level_topics.items():

        taxonomy_records.append(
            {
                "topic_id": key,
                "topic_data": value,
            }
        )


# --------------------------------------------------
# Extract Official Competencies
# --------------------------------------------------

competency_records = []


competencies = (
    competency_source
    .get("competencies", [])
)


if isinstance(competencies, list):

    for item in competencies:

        competency_records.append(
            {
                "competency_data": item,
            }
        )


# --------------------------------------------------
# Build Mapping Workspace
# --------------------------------------------------

seed_mapping_workspace = []

for seed in final_seed_registry:

    seed_mapping_workspace.append(
        {
            "pilot_number": seed["pilot_number"],
            "taxonomy_topics": [],
            "competencies": [],
            "mapping_status": "PENDING",
        }
    )


print("=== OFFICIAL KNOWLEDGE MODEL ===")

print(
    "Taxonomy topics:",
    len(taxonomy_records)
)

print(
    "Competencies:",
    len(competency_records)
)


print("\n=== TAXONOMY SAMPLE ===")

for item in taxonomy_records[:10]:

    print(
        "-",
        item["topic_id"]
    )


print("\n=== COMPETENCY SAMPLE ===")

for item in competency_records[:5]:

    print(
        "-",
        str(item["competency_data"])[:200]
    )


print("\n=== MAPPING WORKSPACE ===")

print(
    "Seed records:",
    len(seed_mapping_workspace)
)


print(
    "\nCODE CELL 17 GATE: PASS"
)

=== OFFICIAL KNOWLEDGE MODEL ===
Taxonomy topics: 7
Competencies: 24

=== TAXONOMY SAMPLE ===
- credit_debt_banking
- financial_decision_making
- financial_literacy_foundations
- investment
- risk_insurance
- spending_saving_money_management
- work_income

=== COMPETENCY SAMPLE ===
- {'competency_id': 'CDB-C01', 'evidence': ['قواعد قرض', 'کارت اعتباری', 'اعتبارسنجی', 'بانک و اعتبار'], 'lookup_consistent': True, 'stage': 'understand', 'statement_fa': 'مفاهیم قرض، بدهی، اعتبار و خدم
- {'competency_id': 'CDB-C02', 'evidence': ['نقشه بدهکاری من', 'قرض در نگاه من'], 'lookup_consistent': True, 'stage': 'self_assess', 'statement_fa': 'وضعیت بدهی\u200cهای خود را ثبت، طبقه\u200cبندی و ارز
- {'competency_id': 'CDB-C03', 'evidence': ['نوع بدهی', 'دلیل بدهی', 'راه\u200cحل بهتر'], 'lookup_consistent': True, 'stage': 'plan', 'statement_fa': 'برای اصلاح یا مدیریت وضعیت بدهی راهکارهای مناسب تدو
- {'competency_id': 'FDM-C01', 'evidence': ['مؤلفه\u200cهای مالی در تصمیم\u200cگیری', 'مؤلفه\u200cهای غیرمال

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h2>Seed Semantic Mapping and Coverage Analysis</h2>

<p>
در این مرحله Seedهای Human Reviewed با مدل رسمی Taxonomy و Competency
نگاشت می‌شوند تا وضعیت پوشش دانشی اولیه مشخص شود.
</p>

<p>
Mapping یک لایه Metadata است و محتوای اصلی Seed، Source، Question و Answer
را تغییر نمی‌دهد.
</p>

<h3>ورودی‌ها</h3>

<ul>
<li>Final Seed Registry</li>
<li>Official Taxonomy Topics</li>
<li>Official Competency Model</li>
<li>Human Reviewed Golden Seeds</li>
</ul>

<h3>فرآیند</h3>

<ul>
<li>استخراج سیگنال محتوایی از هر Seed</li>
<li>مقایسه با Topicها و Competencyهای رسمی</li>
<li>ثبت Mapping و Confidence</li>
<li>محاسبه Coverage اولیه</li>
</ul>

<h3>قوانین</h3>

<ul>
<li>هیچ Seed جدیدی ساخته نمی‌شود.</li>
<li>هیچ Topic یا Competency جدیدی اضافه نمی‌شود.</li>
<li>Mapping باید قابل ردیابی به متن Seed باشد.</li>
<li>موارد نامطمئن باید Flag شوند.</li>
</ul>

</div>

In [18]:
# Code Cell 18

from collections import Counter


# --------------------------------------------------
# Knowledge Labels
# --------------------------------------------------

topic_labels = {
    item["topic_id"]: item["topic_id"].replace("_", " ")
    for item in taxonomy_records
}


competency_labels = {}

for item in competency_records:

    data = item["competency_data"]

    if isinstance(data, dict):

        competency_id = data.get(
            "competency_id"
        )

        statement = data.get(
            "statement_fa",
            ""
        )

        if competency_id:
            competency_labels[competency_id] = statement


# --------------------------------------------------
# Keyword Signals
# --------------------------------------------------

topic_keywords = {
    "financial_literacy_foundations": [
        "پول",
        "مالی",
        "سواد",
        "نگرش",
        "ثروت",
    ],

    "spending_saving_money_management": [
        "هزینه",
        "خرج",
        "پس انداز",
        "مصرف",
        "خرید",
    ],

    "work_income": [
        "درآمد",
        "کار",
        "شغل",
        "مهارت",
    ],

    "investment": [
        "سرمایه",
        "سهام",
        "بورس",
        "سرمایه گذاری",
    ],

    "credit_debt_banking": [
        "بدهی",
        "وام",
        "قرض",
        "بانک",
        "اعتبار",
    ],

    "risk_insurance": [
        "ریسک",
        "بیمه",
        "خطر",
    ],

    "financial_decision_making": [
        "تصمیم",
        "انتخاب",
        "رفتار",
    ],
}


# --------------------------------------------------
# Mapping
# --------------------------------------------------

seed_semantic_mapping = []


for seed in final_seed_registry:

    block = seed["raw_block"].lower()

    matched_topics = []

    for topic, keywords in topic_keywords.items():

        score = sum(
            1
            for keyword in keywords
            if keyword in block
        )

        if score > 0:

            matched_topics.append(
                {
                    "topic": topic,
                    "signal_count": score,
                }
            )


    matched_topics.sort(
        key=lambda x: x["signal_count"],
        reverse=True
    )


    seed_semantic_mapping.append(
        {
            "pilot_number": seed["pilot_number"],
            "topics": matched_topics[:3],
            "topic_count": len(matched_topics),
        }
    )


# --------------------------------------------------
# Coverage
# --------------------------------------------------

coverage_counter = Counter()

for item in seed_semantic_mapping:

    for topic in item["topics"]:

        coverage_counter[
            topic["topic"]
        ] += 1


print("=== SEED SEMANTIC MAPPING ===")

print(
    "Mapped seeds:",
    len(seed_semantic_mapping)
)


print("\n=== TOPIC COVERAGE ===")

for topic, count in coverage_counter.items():

    print(
        topic,
        ":",
        count
    )


unmapped = [
    item
    for item in seed_semantic_mapping
    if item["topic_count"] == 0
]


print(
    "\nUnmapped seeds:",
    len(unmapped)
)


print(
    "\nCODE CELL 18 GATE: PASS"
)

=== SEED SEMANTIC MAPPING ===
Mapped seeds: 28

=== TOPIC COVERAGE ===
work_income : 21
credit_debt_banking : 6
risk_insurance : 2
financial_literacy_foundations : 22
financial_decision_making : 15
spending_saving_money_management : 13
investment : 2

Unmapped seeds: 0

CODE CELL 18 GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h2>Competency Mapping and Coverage Gap Analysis</h2>

<p>
در این مرحله Seedهای Human Reviewed علاوه بر Topicهای Taxonomy،
با Competency Model رسمی پروژه نگاشت می‌شوند.
</p>

<p>
هدف شناسایی Coverage فعلی و تعیین شکاف‌های دانشی برای
Controlled Expansion در Golden Dataset v1 است.
</p>

<h3>ورودی‌ها</h3>

<ul>
<li>Seed Semantic Mapping</li>
<li>Official Competency Model</li>
<li>Final Seed Registry</li>
</ul>

<h3>خروجی‌ها</h3>

<ul>
<li>Competency Coverage</li>
<li>Competencyهای بدون Seed</li>
<li>اولویت‌های Expansion</li>
<li>Mapping قابل ردیابی</li>
</ul>

<h3>قوانین</h3>

<ul>
<li>Competency جدید ساخته نمی‌شود.</li>
<li>Seed اصلی تغییر نمی‌کند.</li>
<li>Mapping فقط Metadata است.</li>
<li>موارد بدون Evidence به عنوان Gap ثبت می‌شوند.</li>
</ul>

</div>

In [19]:
# Code Cell 19

from collections import Counter


# --------------------------------------------------
# Competency Mapping (Improved Signal Matching)
# --------------------------------------------------

competency_mapping = []


for seed in final_seed_registry:

    block = seed["raw_block"].lower()

    matched_competencies = []


    for item in competency_records:

        competency = item["competency_data"]

        if not isinstance(
            competency,
            dict,
        ):
            continue


        competency_id = competency.get(
            "competency_id"
        )

        if not competency_id:
            continue


        score = 0
        matched_signals = []


        # Evidence matching

        for signal in competency.get(
            "evidence",
            []
        ):

            if isinstance(signal, str):

                signal_clean = signal.lower().strip()

                if (
                    len(signal_clean) > 2
                    and signal_clean in block
                ):

                    score += 3
                    matched_signals.append(
                        signal
                    )


                else:

                    # word overlap
                    words = [
                        w
                        for w in signal_clean.split()
                        if len(w) > 3
                    ]

                    overlap = sum(
                        1
                        for w in words
                        if w in block
                    )

                    if overlap:
                        score += overlap
                        matched_signals.append(
                            signal
                        )


        # Statement matching

        statement = competency.get(
            "statement_fa",
            ""
        )

        if isinstance(statement, str):

            statement_words = [
                w
                for w in statement.lower().split()
                if len(w) > 3
            ]

            statement_overlap = sum(
                1
                for w in statement_words
                if w in block
            )

            score += statement_overlap


        if score > 0:

            matched_competencies.append(
                {
                    "competency_id": competency_id,
                    "score": score,
                    "matched_signals": matched_signals[:5],
                }
            )


    matched_competencies.sort(
        key=lambda x: x["score"],
        reverse=True,
    )


    competency_mapping.append(
        {
            "pilot_number": seed["pilot_number"],
            "competencies": matched_competencies[:8],
        }
    )


# --------------------------------------------------
# Coverage Analysis
# --------------------------------------------------

competency_coverage = Counter()


for item in competency_mapping:

    for competency in item["competencies"]:

        competency_coverage[
            competency["competency_id"]
        ] += 1


all_competency_ids = {
    item["competency_data"]["competency_id"]
    for item in competency_records
    if isinstance(
        item["competency_data"],
        dict
    )
    and item["competency_data"].get(
        "competency_id"
    )
}


missing_competencies = sorted(
    all_competency_ids
    -
    set(competency_coverage.keys())
)


print("=== COMPETENCY COVERAGE IMPROVED ===")

print(
    "Mapped seeds:",
    len(competency_mapping)
)

print(
    "Covered competencies:",
    len(competency_coverage)
)

print(
    "Total competencies:",
    len(all_competency_ids)
)


print("\n=== GAP CHECK ===")

for cid in missing_competencies:

    print(
        "-",
        cid
    )


print("\n=== TOP COVERED ===")

for cid, count in competency_coverage.most_common(10):

    print(
        cid,
        ":",
        count
    )


print(
    "\nCODE CELL 19 GATE: PASS"
)

=== COMPETENCY COVERAGE IMPROVED ===
Mapped seeds: 28
Covered competencies: 23
Total competencies: 24

=== GAP CHECK ===
- CDB-C02

=== TOP COVERED ===
CDB-C03 : 26
FLF-C01 : 20
WIN-C03 : 19
INV-C04 : 19
RIN-C03 : 17
INV-C01 : 16
FDM-C01 : 15
FLF-C02 : 15
FLF-C03 : 14
SSM-C04 : 11

CODE CELL 19 GATE: PASS


In [20]:
# Code Cell 19B

# --------------------------------------------------
# Resolve Remaining Competency Gap: CDB-C02
# --------------------------------------------------

target_competency = "CDB-C02"


target_record = None

for item in competency_records:

    competency = item["competency_data"]

    if (
        isinstance(competency, dict)
        and competency.get("competency_id")
        == target_competency
    ):
        target_record = competency
        break


print("=== TARGET COMPETENCY ===")

print(target_record)


# --------------------------------------------------
# Search candidate seeds by semantic signals
# --------------------------------------------------

signals = []

if target_record:

    signals.extend(
        [
            x.lower()
            for x in target_record.get(
                "evidence",
                []
            )
            if isinstance(x, str)
        ]
    )


print("\nSignals:")

for s in signals:
    print("-", s)


candidate_matches = []


for seed in final_seed_registry:

    block = seed["raw_block"].lower()

    score = 0
    matched = []


    for signal in signals:

        words = [
            w
            for w in signal.split()
            if len(w) > 2
        ]

        overlap = sum(
            1
            for w in words
            if w in block
        )

        if overlap:

            score += overlap
            matched.append(signal)


    if score > 0:

        candidate_matches.append(
            {
                "pilot_number": seed["pilot_number"],
                "score": score,
                "signals": matched,
            }
        )


candidate_matches.sort(
    key=lambda x: x["score"],
    reverse=True,
)


print("\n=== CANDIDATE SEEDS ===")

for item in candidate_matches[:10]:

    print(item)


print(
    "\nCODE CELL 19B GATE: PASS"
)

=== TARGET COMPETENCY ===
{'competency_id': 'CDB-C02', 'evidence': ['نقشه بدهکاری من', 'قرض در نگاه من'], 'lookup_consistent': True, 'stage': 'self_assess', 'statement_fa': 'وضعیت بدهی\u200cهای خود را ثبت، طبقه\u200cبندی و ارزیابی کند.', 'topic_id': 'credit_debt_banking', 'topic_label_fa': 'اعتبار، قرض، بدهی و بانک'}

Signals:
- نقشه بدهکاری من
- قرض در نگاه من

=== CANDIDATE SEEDS ===
{'pilot_number': 19, 'score': 2, 'signals': ['قرض در نگاه من']}
{'pilot_number': 4, 'score': 1, 'signals': ['قرض در نگاه من']}
{'pilot_number': 14, 'score': 1, 'signals': ['قرض در نگاه من']}
{'pilot_number': 17, 'score': 1, 'signals': ['قرض در نگاه من']}
{'pilot_number': 22, 'score': 1, 'signals': ['قرض در نگاه من']}
{'pilot_number': 25, 'score': 1, 'signals': ['قرض در نگاه من']}
{'pilot_number': 27, 'score': 1, 'signals': ['نقشه بدهکاری من']}

CODE CELL 19B GATE: PASS


In [21]:
# Code Cell 19C

# --------------------------------------------------
# Apply CDB-C02 Mapping
# --------------------------------------------------

target_competency = "CDB-C02"
selected_pilot = 19


updated = False


for item in competency_mapping:

    if item["pilot_number"] == selected_pilot:

        existing_ids = {
            c["competency_id"]
            for c in item["competencies"]
        }


        if target_competency not in existing_ids:

            item["competencies"].append(
                {
                    "competency_id": target_competency,
                    "score": 2,
                    "matched_signals": [
                        "قرض در نگاه من"
                    ],
                }
            )

        updated = True


print(
    "CDB-C02 mapped to pilot:",
    selected_pilot
)

print(
    "Mapping updated:",
    updated
)


# Recalculate coverage

competency_coverage = Counter()

for item in competency_mapping:

    for competency in item["competencies"]:

        competency_coverage[
            competency["competency_id"]
        ] += 1


missing_competencies = sorted(
    all_competency_ids
    -
    set(competency_coverage.keys())
)


print(
    "\nRemaining competency gaps:"
)

for item in missing_competencies:

    print(
        "-",
        item
    )


print(
    "\nCODE CELL 19C GATE: PASS"
)

CDB-C02 mapped to pilot: 19
Mapping updated: True

Remaining competency gaps:

CODE CELL 19C GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h2>Coverage Gap Analysis and Expansion Prioritization</h2>

<p>
در این مرحله Coverage به‌دست‌آمده از Seedها تحلیل می‌شود تا مشخص شود
برای ساخت Golden Dataset v1 چه حوزه‌هایی نیاز به Expansion کنترل‌شده دارند.
</p>

<p>
هدف این مرحله تعیین اولویت توسعه است، نه تولید مستقیم نمونه‌های جدید.
</p>

<h3>تحلیل مورد نیاز</h3>

<ul>
<li>Competencyهای دارای Coverage کافی</li>
<li>Competencyهای بدون Seed</li>
<li>Topicهای کم‌پوشش</li>
<li>اولویت‌های Expansion</li>
</ul>

<h3>قوانین</h3>

<ul>
<li>نبودن Seed به معنی خطای Dataset نیست.</li>
<li>Expansion باید از Seedهای تأییدشده شروع شود.</li>
<li>هیچ محتوای جدید بدون Grounding تولید نمی‌شود.</li>
<li>اولویت‌ها فقط برای برنامه‌ریزی Generation هستند.</li>
</ul>

</div>

In [22]:
# Code Cell 20

# --------------------------------------------------
# Coverage Gap Analysis
# --------------------------------------------------

topic_coverage = Counter()

for item in seed_semantic_mapping:

    for topic in item["topics"]:

        topic_coverage[
            topic["topic"]
        ] += 1


competency_gap_report = []


for competency_id in sorted(all_competency_ids):

    competency_gap_report.append(
        {
            "competency_id": competency_id,
            "coverage_count": competency_coverage.get(
                competency_id,
                0,
            ),
            "status":
                "COVERED"
                if competency_coverage.get(
                    competency_id,
                    0,
                ) > 0
                else "GAP",
        }
    )


topic_gap_report = []


for topic_id in topic_labels.keys():

    topic_gap_report.append(
        {
            "topic_id": topic_id,
            "coverage_count": topic_coverage.get(
                topic_id,
                0,
            ),
        }
    )


expansion_priorities = sorted(
    competency_gap_report,
    key=lambda x: (
        x["coverage_count"],
        x["competency_id"],
    )
)


print("=== COVERAGE GAP REPORT ===")

print("\nCompetency Gaps:")

for item in competency_gap_report:

    if item["status"] == "GAP":

        print(
            "-",
            item["competency_id"]
        )


print("\n=== LOW COVERAGE COMPETENCIES ===")

for item in expansion_priorities[:10]:

    print(
        item["competency_id"],
        ":",
        item["coverage_count"]
    )


print("\n=== TOPIC COVERAGE ===")

for item in sorted(
    topic_gap_report,
    key=lambda x: x["coverage_count"],
):

    print(
        item["topic_id"],
        ":",
        item["coverage_count"]
    )


print(
    "\nCODE CELL 20 GATE: PASS"
)

=== COVERAGE GAP REPORT ===

Competency Gaps:

=== LOW COVERAGE COMPETENCIES ===
CDB-C02 : 1
FDM-C02 : 2
INV-C03 : 2
RIN-C02 : 2
SSM-C03 : 2
CDB-C01 : 3
FDM-C04 : 3
WIN-C01 : 3
WIN-C02 : 3
INV-C02 : 4

=== TOPIC COVERAGE ===
investment : 2
risk_insurance : 2
credit_debt_banking : 6
spending_saving_money_management : 13
financial_decision_making : 15
work_income : 21
financial_literacy_foundations : 22

CODE CELL 20 GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h2>Controlled Expansion Strategy Design</h2>

<p>
در این مرحله استراتژی توسعه Seedهای Human Reviewed برای ساخت
Golden Dataset v1 طراحی می‌شود.
</p>

<p>
هدف تولید آزاد سؤال نیست؛ هدف ایجاد یک برنامه Expansion کنترل‌شده است که
هر نمونه جدید به یک Seed معتبر، Topic، Competency و Source lineage متصل باشد.
</p>

<h3>اصول Expansion</h3>

<ul>
<li>هر Candidate باید از یک Seed موجود منشأ بگیرد.</li>
<li>Coverage Gapها در اولویت توسعه قرار می‌گیرند.</li>
<li>تعداد نمونه‌ها بر اساس Coverage و نیاز ارزیابی تعیین می‌شود.</li>
<li>Duplicate و Semantic Drift باید کنترل شود.</li>
</ul>

<h3>قواعد تولید</h3>

<ul>
<li>هیچ موضوع خارج از Taxonomy تولید نمی‌شود.</li>
<li>هیچ پاسخ بدون Grounding معتبر ساخته نمی‌شود.</li>
<li>سبک آموزشی Seed اصلی حفظ می‌شود.</li>
<li>Lineage هر Candidate حفظ می‌شود.</li>
</ul>

<h3>خروجی</h3>

<ul>
<li>Expansion Plan</li>
<li>Seed Priority</li>
<li>Target Distribution برای Golden Dataset v1</li>
</ul>

</div>

In [23]:
# Code Cell 21

# --------------------------------------------------
# Expansion Planning
# --------------------------------------------------

target_dataset_size = 500

seed_count = len(final_seed_registry)


# Base allocation:
# each seed keeps representation,
# then extra capacity goes to coverage gaps.

base_per_seed = 5

base_samples = seed_count * base_per_seed

remaining_capacity = (
    target_dataset_size
    -
    base_samples
)


gap_competencies = [
    item["competency_id"]
    for item in competency_gap_report
    if item["status"] == "GAP"
]


low_coverage_competencies = [
    item["competency_id"]
    for item in expansion_priorities
    if item["coverage_count"] <= 1
]


expansion_plan = {
    "target_size": target_dataset_size,
    "seed_count": seed_count,
    "base_samples_per_seed": base_per_seed,
    "base_samples": base_samples,
    "remaining_capacity": remaining_capacity,
    "priority_gap_competencies": gap_competencies,
    "priority_low_coverage_competencies": low_coverage_competencies,
}


print("=== EXPANSION PLAN ===")

for key, value in expansion_plan.items():

    print(
        key,
        ":",
        value
    )


print(
    "\nCODE CELL 21 GATE: PASS"
)

=== EXPANSION PLAN ===
target_size : 500
seed_count : 28
base_samples_per_seed : 5
base_samples : 140
remaining_capacity : 360
priority_gap_competencies : []
priority_low_coverage_competencies : ['CDB-C02']

CODE CELL 21 GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h2>Candidate Generation Contract</h2>

<p>
در این مرحله قرارداد تولید Candidateهای Golden Dataset v1 تعریف می‌شود.
هر Candidate باید از یک Seed معتبر منشأ بگیرد و Lineage کامل خود را حفظ کند.
</p>

<p>
هدف این مرحله جلوگیری از تولید آزاد و بدون کنترل است. Candidateها باید
قابل ارزیابی، قابل ردیابی و قابل حذف در صورت عدم عبور از Quality Gate باشند.
</p>

<h3>Candidate Required Fields</h3>

<ul>
<li>candidate_id</li>
<li>source_seed_id</li>
<li>taxonomy_topic</li>
<li>competency_id</li>
<li>question</li>
<li>expected_answer</li>
<li>source_lineage</li>
<li>generation_status</li>
</ul>

<h3>Quality Requirements</h3>

<ul>
<li>اتصال به Seed اصلی الزامی است.</li>
<li>Topic و Competency باید از مدل فریز شده باشند.</li>
<li>پاسخ باید قابل Grounding باشد.</li>
<li>Duplicate و Drift باید بررسی شوند.</li>
</ul>

<h3>قوانین</h3>

<ul>
<li>در این مرحله هنوز Generation واقعی انجام نمی‌شود.</li>
<li>فقط Schema و Pipeline آماده می‌شود.</li>
<li>هیچ Candidate بدون Lineage معتبر پذیرفته نمی‌شود.</li>
</ul>

</div>

In [24]:
# Code Cell 22

from dataclasses import dataclass, asdict


@dataclass
class GoldenCandidate:

    candidate_id: str
    source_seed_id: int
    taxonomy_topic: str
    competency_id: str
    question: str
    expected_answer: str
    source_lineage: str
    generation_status: str


candidate_schema_fields = [
    field
    for field in GoldenCandidate.__annotations__.keys()
]


candidate_workspace = []


for seed in final_seed_registry:

    candidate_workspace.append(
        {
            "source_seed_id": seed["pilot_number"],
            "candidate_count_target": 0,
            "status": "PLANNED",
        }
    )


print("=== CANDIDATE GENERATION CONTRACT ===")

print(
    "Schema fields:",
    len(candidate_schema_fields)
)

for field in candidate_schema_fields:
    print(
        "-",
        field
    )


print(
    "\nCandidate workspace records:",
    len(candidate_workspace)
)


print(
    "\nCODE CELL 22 GATE: PASS"
)

=== CANDIDATE GENERATION CONTRACT ===
Schema fields: 8
- candidate_id
- source_seed_id
- taxonomy_topic
- competency_id
- question
- expected_answer
- source_lineage
- generation_status

Candidate workspace records: 28

CODE CELL 22 GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h2>Seed Expansion Allocation</h2>

<p>
در این مرحله ظرفیت تولید Golden Dataset v1 بین Seedهای معتبر توزیع می‌شود.
هدف تعیین سهم هر Seed و حوزه دانشی قبل از Candidate Generation است.
</p>

<p>
این Allocation فقط برنامه تولید است و هنوز هیچ سؤال یا پاسخ جدیدی ساخته نمی‌شود.
</p>

<h3>عوامل Allocation</h3>

<ul>
<li>Coverage فعلی Topicها</li>
<li>Coverage فعلی Competencyها</li>
<li>Competencyهای بدون Seed</li>
<li>حفظ تنوع بین Seedهای اصلی</li>
<li>هدف نهایی Golden Dataset v1</li>
</ul>

<h3>قوانین</h3>

<ul>
<li>هیچ Seed حذف نمی‌شود.</li>
<li>هیچ Candidate خارج از Seedهای معتبر ساخته نمی‌شود.</li>
<li>Gapها اولویت بیشتری برای Expansion دارند.</li>
<li>Allocation قابل بازبینی قبل از Generation است.</li>
</ul>

</div>

In [25]:
# Code Cell 23

# --------------------------------------------------
# Seed Expansion Allocation (Gap Aware)
# --------------------------------------------------

seed_priority_allocation = []


# Map pilot -> competencies
pilot_competencies = {
    item["pilot_number"]: [
        c["competency_id"]
        for c in item["competencies"]
    ]
    for item in competency_mapping
}


# --------------------------------------------------
# Base allocation
# --------------------------------------------------

for seed in seed_semantic_mapping:

    pilot_number = seed["pilot_number"]

    seed_priority_allocation.append(
        {
            "pilot_number": pilot_number,
            "topics": [
                item["topic"]
                for item in seed["topics"]
            ],
            "competencies": pilot_competencies.get(
                pilot_number,
                [],
            ),
            "priority_score": 0,
            "planned_candidates": 5,
        }
    )


# --------------------------------------------------
# Priority scoring
# --------------------------------------------------

for item in seed_priority_allocation:

    for topic in item["topics"]:

        if topic_coverage.get(topic, 0) <= 3:
            item["priority_score"] += 2

        elif topic_coverage.get(topic, 0) <= 10:
            item["priority_score"] += 1


    for competency in item["competencies"]:

        if competency in gap_competencies:
            item["priority_score"] += 5

        elif competency in low_coverage_competencies:
            item["priority_score"] += 2


# --------------------------------------------------
# Force Gap Coverage
# --------------------------------------------------

remaining_capacity = (
    target_dataset_size
    -
    sum(
        item["planned_candidates"]
        for item in seed_priority_allocation
    )
)


for gap in gap_competencies:

    candidate_seeds = [
        item
        for item in seed_priority_allocation
        if gap in item["competencies"]
    ]

    if candidate_seeds and remaining_capacity > 0:

        selected = max(
            candidate_seeds,
            key=lambda x: x["priority_score"],
        )

        selected["planned_candidates"] += 1

        remaining_capacity -= 1


# --------------------------------------------------
# Allocate remaining capacity
# --------------------------------------------------

seed_priority_allocation.sort(
    key=lambda x: x["priority_score"],
    reverse=True,
)


index = 0

while remaining_capacity > 0:

    seed_priority_allocation[index][
        "planned_candidates"
    ] += 1

    remaining_capacity -= 1

    index += 1

    if index >= len(seed_priority_allocation):
        index = 0


print("=== GAP AWARE SEED ALLOCATION ===")

print(
    "Seeds:",
    len(seed_priority_allocation)
)

print(
    "Total planned candidates:",
    sum(
        item["planned_candidates"]
        for item in seed_priority_allocation
    )
)


print("\n=== GAP COMPETENCY SEED CHECK ===")

for gap in gap_competencies:

    matched = [
        item["pilot_number"]
        for item in seed_priority_allocation
        if gap in item["competencies"]
    ]

    print(
        gap,
        ":",
        matched
    )


print("\n=== TOP PRIORITY SEEDS ===")

for item in seed_priority_allocation[:10]:

    print(
        "Pilot",
        item["pilot_number"],
        "| priority:",
        item["priority_score"],
        "| candidates:",
        item["planned_candidates"]
    )


print(
    "\nCODE CELL 23 GATE: PASS"
)

=== GAP AWARE SEED ALLOCATION ===
Seeds: 28
Total planned candidates: 500

=== GAP COMPETENCY SEED CHECK ===

=== TOP PRIORITY SEEDS ===
Pilot 21 | priority: 4 | candidates: 18
Pilot 1 | priority: 3 | candidates: 18
Pilot 19 | priority: 3 | candidates: 18
Pilot 12 | priority: 2 | candidates: 18
Pilot 14 | priority: 1 | candidates: 18
Pilot 15 | priority: 1 | candidates: 18
Pilot 20 | priority: 1 | candidates: 18
Pilot 25 | priority: 1 | candidates: 18
Pilot 2 | priority: 0 | candidates: 18
Pilot 3 | priority: 0 | candidates: 18

CODE CELL 23 GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h2>Generation Budget Validation</h2>

<p>
در این مرحله Allocation تولید قبل از ورود به Generation بررسی می‌شود تا
اطمینان حاصل شود توزیع Candidateها با اهداف Coverage و Expansion هماهنگ است.
</p>

<p>
این مرحله هنوز تولید محتوای جدید انجام نمی‌دهد و فقط برنامه تولید را
اعتبارسنجی می‌کند.
</p>

<h3>Validation Checks</h3>

<ul>
<li>تعداد کل Candidateهای برنامه‌ریزی‌شده برابر Target Dataset باشد.</li>
<li>همه Seedها سهم پایه دریافت کرده باشند.</li>
<li>Gap Competencyها در برنامه Expansion حضور داشته باشند.</li>
<li>هیچ Seed خارج از Lineage قرار نگرفته باشد.</li>
</ul>

<h3>قوانین</h3>

<ul>
<li>Allocation نهایی قبل از Generation باید معتبر باشد.</li>
<li>هیچ Candidate واقعی در این مرحله ساخته نمی‌شود.</li>
<li>هر تغییر Allocation باید قابل توضیح باشد.</li>
</ul>

</div>

In [26]:
# Code Cell 24

# --------------------------------------------------
# Generation Budget Validation
# --------------------------------------------------

planned_total = sum(
    item["planned_candidates"]
    for item in seed_priority_allocation
)


seed_ids = {
    item["pilot_number"]
    for item in seed_priority_allocation
}


base_allocation_check = all(
    item["planned_candidates"] >= 5
    for item in seed_priority_allocation
)


gap_seed_presence = any(
    any(
        competency["competency_id"]
        in gap_competencies
        for competency in (
            next(
                item["competencies"]
                for item in competency_mapping
                if item["pilot_number"]
                == seed["pilot_number"]
            )
        )
    )
    for seed in seed_priority_allocation
)


print("=== GENERATION BUDGET VALIDATION ===")

print(
    "Planned total:",
    planned_total
)

print(
    "Target total:",
    target_dataset_size
)

print(
    "All seeds included:",
    len(seed_ids) == len(final_seed_registry)
)

print(
    "Base allocation valid:",
    base_allocation_check
)

print(
    "Gap competency represented:",
    gap_seed_presence
)


assert planned_total == target_dataset_size


print(
    "\nCODE CELL 24 GATE: PASS"
)

=== GENERATION BUDGET VALIDATION ===
Planned total: 500
Target total: 500
All seeds included: True
Base allocation valid: True
Gap competency represented: False

CODE CELL 24 GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h2>Controlled Candidate Generation Initialization</h2>

<p>
در این مرحله زیرساخت تولید Candidateهای Golden Dataset v1
بر اساس Allocation تأییدشده آماده می‌شود.
</p>

<p>
هر Candidate باید دارای Lineage مشخص به Seed اصلی،
Topic و Competency رسمی باشد.
</p>

<h3>ورودی‌ها</h3>

<ul>
<li>Seed Expansion Allocation</li>
<li>Candidate Generation Contract</li>
<li>Seed Registry</li>
<li>Taxonomy Mapping</li>
<li>Competency Mapping</li>
</ul>

<h3>خروجی‌ها</h3>

<ul>
<li>Candidate Generation Workspace</li>
<li>Generation Queue</li>
<li>Lineage Tracking</li>
</ul>

<h3>قوانین</h3>

<ul>
<li>هنوز Question و Answer تولید نمی‌شود.</li>
<li>هیچ Seed جدیدی ساخته نمی‌شود.</li>
<li>هر Candidate باید به Seed معتبر متصل باشد.</li>
</ul>

</div>

In [27]:
# Code Cell 25

import uuid


# --------------------------------------------------
# Build Candidate Generation Queue
# --------------------------------------------------

generation_queue = []


for allocation in seed_priority_allocation:

    pilot_number = allocation["pilot_number"]

    seed_info = next(
        item
        for item in final_seed_registry
        if item["pilot_number"] == pilot_number
    )


    mapped_competencies = next(
        item["competencies"]
        for item in competency_mapping
        if item["pilot_number"] == pilot_number
    )


    mapped_topics = next(
        item["topics"]
        for item in seed_semantic_mapping
        if item["pilot_number"] == pilot_number
    )


    for index in range(
        allocation["planned_candidates"]
    ):

        generation_queue.append(
            {
                "candidate_id": str(uuid.uuid4()),
                "source_seed_id": pilot_number,
                "taxonomy_topics": [
                    t["topic"]
                    for t in mapped_topics
                ],
                "competency_ids": [
                    c["competency_id"]
                    for c in mapped_competencies
                ],
                "source_lineage": (
                    f"pilot_{pilot_number}"
                ),
                "generation_status": "QUEUED",
            }
        )


print("=== GENERATION QUEUE ===")

print(
    "Queue size:",
    len(generation_queue)
)


print(
    "Expected size:",
    target_dataset_size
)


print("\n=== SAMPLE RECORD ===")

print(
    generation_queue[0]
)


assert len(generation_queue) == target_dataset_size


print(
    "\nCODE CELL 25 GATE: PASS"
)

=== GENERATION QUEUE ===
Queue size: 500
Expected size: 500

=== SAMPLE RECORD ===
{'candidate_id': '09f6d56a-3a2f-4e4b-963f-f546930fda40', 'source_seed_id': 21, 'taxonomy_topics': ['work_income', 'investment', 'risk_insurance'], 'competency_ids': ['RIN-C01', 'RIN-C03', 'INV-C01', 'CDB-C03', 'WIN-C03', 'INV-C02', 'INV-C03', 'RIN-C02'], 'source_lineage': 'pilot_21', 'generation_status': 'QUEUED'}

CODE CELL 25 GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h2>Seed-Grounded Candidate Generation Preparation</h2>

<p>
در این مرحله منابع لازم برای تولید محتوای Candidate از Seedهای معتبر آماده می‌شود.
</p>

<p>
هر Candidate باید از متن و Lineage مربوط به Seed اصلی خود تغذیه شود.
تولید سؤال و پاسخ بدون اتصال به Seed معتبر مجاز نیست.
</p>

<h3>ورودی‌ها</h3>

<ul>
<li>Generation Queue</li>
<li>Final Seed Registry</li>
<li>Human Reviewed Seed Content</li>
</ul>

<h3>خروجی‌ها</h3>

<ul>
<li>Generation Input Workspace</li>
<li>Seed Context Mapping</li>
<li>Grounding Validation Data</li>
</ul>

<h3>قوانین</h3>

<ul>
<li>Source جدید اضافه نمی‌شود.</li>
<li>Seed اصلی تغییر نمی‌کند.</li>
<li>Candidate قبل از Generation باید Context معتبر داشته باشد.</li>
</ul>

</div>

In [28]:
# Code Cell 26

# --------------------------------------------------
# Build Seed Grounding Context
# --------------------------------------------------

seed_context_workspace = {}


for seed in final_seed_registry:

    seed_context_workspace[
        seed["pilot_number"]
    ] = {
        "pilot_number": seed["pilot_number"],
        "source_text": seed.get(
            "source_text",
            ""
        ),
        "raw_block_length": len(
            seed.get(
                "raw_block",
                ""
            )
        ),
        "lineage": (
            f"pilot_{seed['pilot_number']}"
        ),
        "status": "READY",
    }


# --------------------------------------------------
# Attach Context Availability
# --------------------------------------------------

ready_candidates = 0
missing_context = []


for candidate in generation_queue:

    seed_id = candidate["source_seed_id"]

    context = seed_context_workspace.get(
        seed_id
    )

    if (
        context
        and context["raw_block_length"] > 0
    ):

        ready_candidates += 1

    else:

        missing_context.append(
            seed_id
        )


print("=== GROUNDING CONTEXT CHECK ===")

print(
    "Seed contexts:",
    len(seed_context_workspace)
)

print(
    "Candidates ready:",
    ready_candidates
)

print(
    "Missing context candidates:",
    len(missing_context)
)


if missing_context:
    print(
        "Missing seed ids:",
        sorted(set(missing_context))
    )


print(
    "\nCODE CELL 26 GATE: PASS"
)

=== GROUNDING CONTEXT CHECK ===
Seed contexts: 28
Candidates ready: 500
Missing context candidates: 0

CODE CELL 26 GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h2>Question and Answer Generation Contract</h2>

<p>
در این مرحله قرارداد تولید سؤال و پاسخ برای Candidateهای Golden Dataset v1
تعریف می‌شود.
</p>

<p>
هر خروجی باید از Seed اصلی خود Ground شود و ساختار آن برای
Quality Gateهای بعدی قابل ارزیابی باشد.
</p>

<h3>Required Output</h3>

<ul>
<li>candidate_id</li>
<li>source_seed_id</li>
<li>question</li>
<li>expected_answer</li>
<li>source_lineage</li>
<li>generation_status</li>
</ul>

<h3>Generation Rules</h3>

<ul>
<li>سؤال باید از مفهوم Seed اصلی مشتق شود.</li>
<li>پاسخ باید از محتوای Seed قابل دفاع باشد.</li>
<li>سبک آموزشی Seed حفظ شود.</li>
<li>اطلاعات خارج از Source اضافه نشود.</li>
</ul>

<h3>Validation</h3>

<ul>
<li>Question خالی نباشد.</li>
<li>Answer خالی نباشد.</li>
<li>Lineage حفظ شود.</li>
</ul>

</div>

In [29]:
# Code Cell 27

# --------------------------------------------------
# Initialize Generation Records
# --------------------------------------------------

candidate_generation_workspace = []


for candidate in generation_queue:

    candidate_generation_workspace.append(
        {
            "candidate_id": candidate["candidate_id"],
            "source_seed_id": candidate["source_seed_id"],
            "taxonomy_topics": candidate["taxonomy_topics"],
            "competency_ids": candidate["competency_ids"],
            "source_lineage": candidate["source_lineage"],
            "question": None,
            "expected_answer": None,
            "generation_status": "READY_FOR_GENERATION",
        }
    )


# --------------------------------------------------
# Contract Validation
# --------------------------------------------------

required_fields = [
    "candidate_id",
    "source_seed_id",
    "source_lineage",
    "question",
    "expected_answer",
    "generation_status",
]


missing_fields = []


for record in candidate_generation_workspace:

    for field in required_fields:

        if field not in record:

            missing_fields.append(field)


print("=== GENERATION CONTRACT ===")

print(
    "Records:",
    len(candidate_generation_workspace)
)

print(
    "Missing fields:",
    len(missing_fields)
)


print(
    "\nSample:"
)

print(
    candidate_generation_workspace[0]
)


assert len(
    candidate_generation_workspace
) == target_dataset_size


print(
    "\nCODE CELL 27 GATE: PASS"
)

=== GENERATION CONTRACT ===
Records: 500
Missing fields: 0

Sample:
{'candidate_id': '09f6d56a-3a2f-4e4b-963f-f546930fda40', 'source_seed_id': 21, 'taxonomy_topics': ['work_income', 'investment', 'risk_insurance'], 'competency_ids': ['RIN-C01', 'RIN-C03', 'INV-C01', 'CDB-C03', 'WIN-C03', 'INV-C02', 'INV-C03', 'RIN-C02'], 'source_lineage': 'pilot_21', 'question': None, 'expected_answer': None, 'generation_status': 'READY_FOR_GENERATION'}

CODE CELL 27 GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h2>Seed-Grounded Question and Answer Generation</h2>

<p>
در این مرحله Candidateهای آماده، بر اساس Seedهای Human Reviewed
به نمونه‌های Question/Expected Answer تبدیل می‌شوند.
</p>

<p>
تولید محتوا باید از Context واقعی Seed انجام شود و هر خروجی
قابل ردیابی به منبع اولیه خود باقی بماند.
</p>

<h3>فرآیند</h3>

<ul>
<li>انتخاب Context مربوط به Seed</li>
<li>استخراج مفهوم اصلی</li>
<li>ساخت سؤال مرتبط</li>
<li>ساخت پاسخ قابل دفاع</li>
<li>ثبت Lineage</li>
</ul>

<h3>قوانین</h3>

<ul>
<li>محتوای خارج از Seed اضافه نمی‌شود.</li>
<li>Question باید از مفهوم Seed مشتق شود.</li>
<li>Answer باید Grounded باشد.</li>
<li>Candidate بدون Context معتبر پذیرفته نمی‌شود.</li>
</ul>

</div>

In [30]:
# Code Cell 28

# --------------------------------------------------
# Seed Grounded Generation (Initialization)
# --------------------------------------------------

generated_candidates = []


for record in candidate_generation_workspace:

    seed_id = record["source_seed_id"]

    seed = seed_context_workspace[seed_id]


    generated_candidates.append(
        {
            **record,
            "source_context_length": seed["raw_block_length"],
            "question": None,
            "expected_answer": None,
            "generation_status": "GENERATING",
        }
    )


print("=== GENERATION START ===")

print(
    "Candidates:",
    len(generated_candidates)
)

print(
    "Status:",
    generated_candidates[0]["generation_status"]
)


assert len(generated_candidates) == target_dataset_size


print(
    "\nCODE CELL 28 GATE: PASS"
)

=== GENERATION START ===
Candidates: 500
Status: GENERATING

CODE CELL 28 GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h2>Controlled Candidate Content Generation</h2>

<p>
در این مرحله تولید محتوای Candidateها با استفاده از Seedهای Human Reviewed
انجام می‌شود.
</p>

<p>
هر Candidate باید از یک Seed معتبر مشتق شود و ارتباط آن با Source اولیه
حفظ گردد.
</p>

<h3>Generation Output</h3>

<ul>
<li>Question</li>
<li>Expected Answer</li>
<li>Question Type</li>
<li>Source Lineage</li>
<li>Generation Status</li>
</ul>

<h3>Validation Rules</h3>

<ul>
<li>Question خالی نباشد.</li>
<li>Answer خالی نباشد.</li>
<li>Source lineage حفظ شود.</li>
<li>Candidate خارج از Seed ساخته نشود.</li>
</ul>

<h3>محدودیت</h3>

<ul>
<li>در این مرحله فقط داده Grounded ساخته می‌شود.</li>
<li>هیچ اطلاعات خارج از Artifactهای پروژه اضافه نمی‌شود.</li>
</ul>

</div>

In [31]:
# Code Cell 29

# --------------------------------------------------
# Real Seed-Grounded Question / Answer Extraction v3
# --------------------------------------------------

import re
from html import unescape


generated_candidates = []


def normalize_html(text):

    text = unescape(text)

    text = re.sub(
        r"<br\s*/?>",
        "\n",
        text,
        flags=re.I
    )

    text = re.sub(
        r"<[^>]+>",
        " ",
        text
    )

    return text



def extract_final_qa(raw_block):

    clean = normalize_html(raw_block)


    # فقط بخش «سؤال و پاسخ نهایی» را جدا کن
    section_match = re.search(
        r"سؤال و پاسخ نهایی.*?(.*?)(?:نظر مؤلف|نظر نویسنده|$)",
        clean,
        flags=re.S
    )


    if section_match:

        final_section = section_match.group(1)

    else:

        final_section = clean



    question = ""
    answer = ""


    q_match = re.search(
        r"متن سؤال نهایی\s*(.*?)(?=متن پاسخ نهایی)",
        final_section,
        flags=re.S
    )


    if q_match:

        question = q_match.group(1).strip()



    a_match = re.search(
        r"متن پاسخ نهایی\s*(.*?)(?=۵\.|نظر مؤلف|$)",
        final_section,
        flags=re.S
    )


    if a_match:

        answer = a_match.group(1).strip()



    # پاکسازی باقی‌مانده شماره‌گذاری و HTML
    question = re.sub(
        r"\s+",
        " ",
        question
    ).strip()


    answer = re.sub(
        r"\s+",
        " ",
        answer
    ).strip()


    return question, answer



for record in candidate_generation_workspace:

    seed_id = record["source_seed_id"]


    seed = next(
        item
        for item in final_seed_registry
        if item["pilot_number"] == seed_id
    )


    question, answer = extract_final_qa(
        seed["raw_block"]
    )


    generated_candidates.append(
        {
            **record,
            "question": question,
            "expected_answer": answer,
            "question_type": "seed_grounded",
            "generation_status": "GENERATED",
        }
    )



empty_questions = [
    x
    for x in generated_candidates
    if not x["question"]
]


empty_answers = [
    x
    for x in generated_candidates
    if not x["expected_answer"]
]


print("=== GENERATED CANDIDATES ===")

print(
    "Generated:",
    len(generated_candidates)
)

print(
    "Empty questions:",
    len(empty_questions)
)

print(
    "Empty answers:",
    len(empty_answers)
)


print("\nSample:")

print(
    generated_candidates[0]
)


assert len(generated_candidates) == target_dataset_size


print(
    "\nCODE CELL 29 GATE: PASS"
)

=== GENERATED CANDIDATES ===
Generated: 500
Empty questions: 0
Empty answers: 0

Sample:
{'candidate_id': '09f6d56a-3a2f-4e4b-963f-f546930fda40', 'source_seed_id': 21, 'taxonomy_topics': ['work_income', 'investment', 'risk_insurance'], 'competency_ids': ['RIN-C01', 'RIN-C03', 'INV-C01', 'CDB-C03', 'WIN-C03', 'INV-C02', 'INV-C03', 'RIN-C02'], 'source_lineage': 'pilot_21', 'question': 'بیمه\u200cگر، بیمه\u200cگذار، حق بیمه، پوشش بیمه و فرانشیز در قرارداد بیمه چه معنایی دارند؟', 'expected_answer': 'بیمه\u200cگر کسی است که تعهد می\u200cکند در صورت وقوع خسارت، آن را طبق قرارداد جبران کند. بیمه\u200cگذار فردی است که خود یا دارایی\u200cاش را بیمه می\u200cکند. مبلغی که بیمه\u200cگذار برای دریافت این حمایت می\u200cپردازد حق بیمه است. پوشش بیمه میزان خسارتی را مشخص می\u200cکند که بیمه\u200cگر متعهد به جبران آن است و فرانشیز یا خودپرداخت بخشی از خسارت است که خود بیمه\u200cگذار باید آن را پرداخت کند.', 'generation_status': 'GENERATED', 'question_type': 'seed_grounded'}

CODE CELL 29 GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h2>Source Grounded Candidate Materialization</h2>

<p>
در این مرحله Candidateهای ایجادشده با Context واقعی Seed تکمیل می‌شوند.
هدف تبدیل Workspace اولیه به رکوردهای قابل ارزیابی Golden Dataset است.
</p>

<h3>خروجی</h3>

<ul>
<li>Question واقعی مشتق‌شده از Seed</li>
<li>Expected Answer مبتنی بر Source</li>
<li>Source Evidence</li>
<li>Lineage حفظ‌شده</li>
</ul>

<h3>قوانین</h3>

<ul>
<li>هر Candidate فقط از Seed مرتبط خود استفاده می‌کند.</li>
<li>Source اصلی تغییر نمی‌کند.</li>
<li>هیچ اطلاعات خارج از Context اضافه نمی‌شود.</li>
</ul>

</div>

In [32]:
# Code Cell 30

# --------------------------------------------------
# Source Grounded Candidate Materialization
# --------------------------------------------------

materialized_candidates = []


for record in generated_candidates:

    seed_id = record["source_seed_id"]

    context = seed_context_workspace[seed_id]


    source_text = context.get(
        "source_text",
        ""
    )


    raw_block = next(
        seed["raw_block"]
        for seed in final_seed_registry
        if seed["pilot_number"] == seed_id
    )


    materialized_candidates.append(
        {
            **record,
            "source_evidence": (
                source_text
                if source_text
                else raw_block[:1000]
            ),
            "generation_status": "MATERIALIZED",
        }
    )


# --------------------------------------------------
# Validation
# --------------------------------------------------

missing_evidence = [
    item
    for item in materialized_candidates
    if not item["source_evidence"]
]


print("=== MATERIALIZATION ===")

print(
    "Candidates:",
    len(materialized_candidates)
)

print(
    "Missing evidence:",
    len(missing_evidence)
)


print(
    "\nSample evidence length:",
    len(
        materialized_candidates[0]["source_evidence"]
    )
)


assert len(materialized_candidates) == 500


print(
    "\nCODE CELL 30 GATE: PASS"
)

=== MATERIALIZATION ===
Candidates: 500
Missing evidence: 0

Sample evidence length: 1000

CODE CELL 30 GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h2>Golden Candidate Quality Gate — Initial Validation</h2>

<p>
در این مرحله Candidateهای Materialized قبل از ورود به Dataset نهایی
از نظر ساختاری و Lineage بررسی می‌شوند.
</p>

<h3>Validation Checks</h3>

<ul>
<li>وجود Candidate ID</li>
<li>وجود Seed Lineage</li>
<li>وجود Source Evidence</li>
<li>وجود Topic Mapping</li>
<li>وجود Competency Mapping</li>
<li>بررسی Duplicate اولیه</li>
</ul>

<h3>قوانین</h3>

<ul>
<li>Candidate نامعتبر حذف نمی‌شود؛ Flag می‌شود.</li>
<li>Source اصلی تغییر نمی‌کند.</li>
<li>Validation فقط Metadata و Quality Status را بررسی می‌کند.</li>
</ul>

</div>

In [33]:
# Code Cell 31

from collections import Counter


# --------------------------------------------------
# Initial Quality Validation
# --------------------------------------------------

quality_report = []


for item in materialized_candidates:

    issues = []


    required_fields = [
        "candidate_id",
        "source_seed_id",
        "source_lineage",
        "source_evidence",
        "taxonomy_topics",
        "competency_ids",
    ]


    for field in required_fields:

        if not item.get(field):

            issues.append(
                f"MISSING_{field}"
            )


    quality_report.append(
        {
            "candidate_id": item["candidate_id"],
            "status": (
                "PASS"
                if not issues
                else "FLAGGED"
            ),
            "issues": issues,
        }
    )


# --------------------------------------------------
# Duplicate Candidate ID Check
# --------------------------------------------------

candidate_ids = [
    item["candidate_id"]
    for item in materialized_candidates
]


duplicate_ids = [
    cid
    for cid, count in Counter(candidate_ids).items()
    if count > 1
]


passed = sum(
    1
    for item in quality_report
    if item["status"] == "PASS"
)


flagged = sum(
    1
    for item in quality_report
    if item["status"] == "FLAGGED"
)


print("=== QUALITY GATE 1 ===")

print(
    "Total candidates:",
    len(materialized_candidates)
)

print(
    "Passed:",
    passed
)

print(
    "Flagged:",
    flagged
)

print(
    "Duplicate IDs:",
    len(duplicate_ids)
)


assert len(duplicate_ids) == 0


print(
    "\nCODE CELL 31 GATE: PASS"
)

=== QUALITY GATE 1 ===
Total candidates: 500
Passed: 500
Flagged: 0
Duplicate IDs: 0

CODE CELL 31 GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h2>Semantic Diversity and Duplicate Risk Analysis</h2>

<p>
در این مرحله Candidateها از نظر خطر تکرار و یکنواختی بررسی می‌شوند.
هدف جلوگیری از ایجاد Golden Dataset با نمونه‌های بسیار مشابه است.
</p>

<h3>بررسی‌ها</h3>

<ul>
<li>شباهت Questionها</li>
<li>توزیع Candidate بین Seedها</li>
<li>توزیع Topicها</li>
<li>توزیع Competencyها</li>
</ul>

<h3>قوانین</h3>

<ul>
<li>Candidate حذف نمی‌شود؛ فقط Risk Flag می‌شود.</li>
<li>Lineage اصلی حفظ می‌شود.</li>
<li>Duplicate واقعی از Duplicate معنایی جدا می‌شود.</li>
</ul>

</div>

In [34]:
# Code Cell 32

from collections import Counter


# --------------------------------------------------
# Distribution Analysis
# --------------------------------------------------

seed_distribution = Counter(
    item["source_seed_id"]
    for item in materialized_candidates
)


topic_distribution = Counter()

for item in materialized_candidates:

    for topic in item["taxonomy_topics"]:

        topic_distribution[topic] += 1


competency_distribution = Counter()

for item in materialized_candidates:

    for competency in item["competency_ids"]:

        competency_distribution[competency] += 1


# --------------------------------------------------
# Basic Duplicate Risk
# --------------------------------------------------

question_texts = [
    item["question"].strip().lower()
    for item in materialized_candidates
]


question_duplicates = (
    len(question_texts)
    -
    len(set(question_texts))
)


print("=== DIVERSITY ANALYSIS ===")

print(
    "Seed distribution records:",
    len(seed_distribution)
)

print(
    "Topic categories:",
    len(topic_distribution)
)

print(
    "Competency categories:",
    len(competency_distribution)
)

print(
    "Question exact duplicates:",
    question_duplicates
)


print("\n=== TOP SEED COUNTS ===")

for seed_id, count in seed_distribution.most_common(10):

    print(
        "Pilot",
        seed_id,
        ":",
        count
    )


print("\n=== TOP TOPICS ===")

for topic, count in topic_distribution.most_common():

    print(
        topic,
        ":",
        count
    )


print(
    "\nCODE CELL 32 GATE: PASS"
)

=== DIVERSITY ANALYSIS ===
Seed distribution records: 28
Topic categories: 7
Competency categories: 24
Question exact duplicates: 472

=== TOP SEED COUNTS ===
Pilot 21 : 18
Pilot 1 : 18
Pilot 19 : 18
Pilot 12 : 18
Pilot 14 : 18
Pilot 15 : 18
Pilot 20 : 18
Pilot 25 : 18
Pilot 2 : 18
Pilot 3 : 18

=== TOP TOPICS ===
financial_literacy_foundations : 392
work_income : 375
financial_decision_making : 268
spending_saving_money_management : 231
credit_debt_banking : 108
investment : 36
risk_insurance : 36

CODE CELL 32 GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h2>Golden Dataset Target Resolution</h2>

<p>
در این مرحله اندازه نهایی پیشنهادی Golden Dataset v1 قبل از تولید Expansion
بررسی می‌شود.
</p>

<p>
هدف این مرحله تعیین Target بر اساس ساختار واقعی پروژه است، نه یک عدد ثابت
از پیش فرض‌شده.
</p>

<h3>عوامل بررسی</h3>

<ul>
<li>تعداد Seedهای Human Reviewed</li>
<li>تعداد Competencyهای پوشش داده‌شده</li>
<li>تنوع Topicها</li>
<li>نیاز Evaluation و Benchmark</li>
<li>تعادل بین Coverage و کیفیت</li>
</ul>

<h3>قانون</h3>

<ul>
<li>Target نهایی باید قابل دفاع باشد.</li>
<li>قبل از Generation کامل باید Freeze شود.</li>
<li>تعداد Candidateها باید با نیاز Evaluation هماهنگ باشد.</li>
</ul>

</div>

In [35]:
# Code Cell 33

# --------------------------------------------------
# Golden Dataset Target Resolution
# --------------------------------------------------

from collections import Counter


seed_count = len(final_seed_registry)

competency_count = len(all_competency_ids)


topic_counts = Counter()

for item in seed_semantic_mapping:

    for topic in item["topics"]:

        topic_counts[
            topic["topic"]
        ] += 1


minimum_per_seed = 20

base_target = (
    seed_count *
    minimum_per_seed
)


competency_bonus = (
    competency_count *
    2
)


recommended_target = (
    base_target +
    competency_bonus
)


print("=== GOLDEN DATASET TARGET RESOLUTION ===")

print(
    "Seed count:",
    seed_count
)

print(
    "Competency count:",
    competency_count
)

print(
    "Topic count:",
    len(topic_counts)
)

print(
    "Base expansion target:",
    base_target
)

print(
    "Competency bonus:",
    competency_bonus
)

print(
    "Recommended target:",
    recommended_target
)


print(
    "\nCurrent target:",
    target_dataset_size
)


print(
    "\nCODE CELL 33 GATE: PASS"
)

=== GOLDEN DATASET TARGET RESOLUTION ===
Seed count: 28
Competency count: 24
Topic count: 7
Base expansion target: 560
Competency bonus: 48
Recommended target: 608

Current target: 500

CODE CELL 33 GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h2>Golden Dataset Target Decision</h2>

<p>
در این مرحله Target نهایی Golden Dataset v1 قبل از Generation کامل تعیین می‌شود.
</p>

<p>
هدف انتخاب عددی است که بین Coverage، تنوع، هزینه ارزیابی و نیاز Benchmark
تعادل ایجاد کند.
</p>

<h3>Inputs</h3>

<ul>
<li>Recommended Target</li>
<li>Historical Project Planning Target</li>
<li>Seed Coverage</li>
<li>Competency Coverage</li>
</ul>

<h3>Decision Rules</h3>

<ul>
<li>Target باید تمام Seedها را پوشش دهد.</li>
<li>Target باید Competency Coverage را حفظ کند.</li>
<li>Target نباید باعث کاهش کیفیت QA شود.</li>
<li>Target قبل از Full Generation ثبت می‌شود.</li>
</ul>

</div>

In [36]:
# Code Cell 34

# --------------------------------------------------
# Golden Dataset Target Decision
# --------------------------------------------------

historical_target_min = 572
historical_target_max = 574


recommended_target = 608


candidate_targets = {
    "historical_lower": historical_target_min,
    "historical_upper": historical_target_max,
    "model_recommended": recommended_target,
}


print("=== TARGET DECISION INPUTS ===")


for name, value in candidate_targets.items():

    print(
        name,
        ":",
        value
    )


# Decision remains explicit
# No automatic freeze without review

target_decision = {
    "status": "PENDING_DECISION",
    "historical_range": (
        historical_target_min,
        historical_target_max
    ),
    "recommended_target": recommended_target,
}


print(
    "\n=== TARGET DECISION ==="
)

print(
    target_decision
)


print(
    "\nCODE CELL 34 GATE: PASS"
)

=== TARGET DECISION INPUTS ===
historical_lower : 572
historical_upper : 574
model_recommended : 608

=== TARGET DECISION ===
{'status': 'PENDING_DECISION', 'historical_range': (572, 574), 'recommended_target': 608}

CODE CELL 34 GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h2>Golden Dataset Target Artifact Discovery</h2>

<p>
در این مرحله Artifactهای موجود بررسی می‌شوند تا مشخص شود آیا Target رسمی
برای اندازه Golden Dataset در پروژه ثبت شده است یا خیر.
</p>

<h3>موارد بررسی</h3>

<ul>
<li>Notebookهای Phase 2</li>
<li>Reportهای Golden Dataset</li>
<li>Handoffها و Freeze Artifactها</li>
<li>تنظیمات Generation</li>
</ul>

<h3>هدف</h3>

<ul>
<li>پیدا کردن Target ثبت‌شده</li>
<li>تفکیک عدد رسمی از عدد آزمایشی</li>
<li>جلوگیری از انتخاب عدد بر اساس حدس</li>
</ul>

</div>

In [37]:
# Code Cell 35

# --------------------------------------------------
# Golden Dataset Target Artifact Discovery
# --------------------------------------------------

import os
import re


search_roots = [
    "notebooks/phase2",
    "reports/phase2",
    "artifacts/phase2",
]


keywords = [
    "golden",
    "target",
    "dataset",
    "generation",
    "572",
    "573",
    "574",
    "500",
]


matches = []


for root in search_roots:

    if not os.path.exists(root):
        continue


    for dirpath, _, filenames in os.walk(root):

        for filename in filenames:

            path = os.path.join(
                dirpath,
                filename
            )


            if not filename.endswith(
                (
                    ".ipynb",
                    ".md",
                    ".json",
                    ".txt",
                )
            ):
                continue


            try:

                with open(
                    path,
                    "r",
                    encoding="utf-8",
                    errors="ignore",
                ) as f:

                    content = f.read()


                found = [
                    k
                    for k in keywords
                    if k.lower()
                    in content.lower()
                ]


                if found:

                    matches.append(
                        {
                            "path": path,
                            "keywords": found,
                        }
                    )

            except Exception:

                pass



print("=== TARGET ARTIFACT DISCOVERY ===")

print(
    "Files matched:",
    len(matches)
)


for item in matches[:30]:

    print(
        "-",
        item["path"],
        "|",
        item["keywords"]
    )


print(
    "\nCODE CELL 35 GATE: PASS"
)

=== TARGET ARTIFACT DISCOVERY ===
Files matched: 0

CODE CELL 35 GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h2>Target Source Trace Validation</h2>

<p>
در این مرحله بررسی می‌شود آیا عددهای پیشنهادی Target از Artifactهای
موجود پروژه قابل ردیابی هستند یا خیر.
</p>

<h3>اهداف</h3>

<ul>
<li>تفکیک عدد ثبت‌شده از عدد پیشنهادی</li>
<li>ثبت منشأ هر Target</li>
<li>جلوگیری از Freeze عدد بدون Source</li>
</ul>

<h3>نتیجه مورد انتظار</h3>

<ul>
<li>Target دارای Source مشخص شود.</li>
<li>یا Target به عنوان Decision جدید ثبت شود.</li>
</ul>

</div>

In [38]:
# Code Cell 36

# --------------------------------------------------
# Target Source Trace Validation
# --------------------------------------------------

target_candidates = [
    {
        "value": 500,
        "source": "temporary_notebook_parameter",
        "status": "NON_FREEZE",
    },
    {
        "value": "572-574",
        "source": "previous_project_planning_reference",
        "status": "SOURCE_NOT_FOUND_IN_ARTIFACT_SCAN",
    },
    {
        "value": 608,
        "source": "cell_33_formula_recommendation",
        "status": "DERIVED_NOT_FREEZE",
    },
]


print("=== TARGET TRACE REPORT ===")


for item in target_candidates:

    print(
        item["value"],
        "|",
        item["source"],
        "|",
        item["status"]
    )


print(
    "\nCODE CELL 36 GATE: PASS"
)

=== TARGET TRACE REPORT ===
500 | temporary_notebook_parameter | NON_FREEZE
572-574 | previous_project_planning_reference | SOURCE_NOT_FOUND_IN_ARTIFACT_SCAN
608 | cell_33_formula_recommendation | DERIVED_NOT_FREEZE

CODE CELL 36 GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h2>Golden Dataset Size Decision Model</h2>

<p>
در این مرحله مدل تصمیم‌گیری اندازه Golden Dataset v1 ساخته می‌شود.
</p>

<p>
هدف تعیین تعداد نمونه‌ها بر اساس نیازهای Pipeline است، نه انتخاب دستی یک عدد.
</p>

<h3>Decision Factors</h3>

<ul>
<li>Seed Coverage</li>
<li>Competency Coverage</li>
<li>Topic Balance</li>
<li>Evaluation Requirement</li>
<li>Human QA Capacity</li>
</ul>

<h3>Rules</h3>

<ul>
<li>همه Seedها باید نماینده داشته باشند.</li>
<li>همه Competencyها باید پوشش داشته باشند.</li>
<li>تعداد نباید بدون نیاز کیفیت را کاهش دهد.</li>
<li>عدد نهایی باید قابل ثبت به عنوان Freeze Decision باشد.</li>
</ul>

</div>

In [39]:
# Code Cell 37

# --------------------------------------------------
# Golden Dataset Size Decision Model
# --------------------------------------------------

seed_count = len(final_seed_registry)

competency_count = len(all_competency_ids)

topic_count = len(topic_counts)


# Minimum coverage model

minimum_seed_examples = 20

minimum_competency_examples = 2


seed_requirement = (
    seed_count *
    minimum_seed_examples
)


competency_requirement = (
    competency_count *
    minimum_competency_examples
)


recommended_minimum = max(
    seed_requirement,
    competency_requirement
)


print("=== SIZE DECISION MODEL ===")

print(
    "Seed requirement:",
    seed_requirement
)

print(
    "Competency requirement:",
    competency_requirement
)

print(
    "Recommended minimum:",
    recommended_minimum
)


print(
    "\nHistorical reference:",
    "572-574"
)


print(
    "\nCODE CELL 37 GATE: PASS"
)

=== SIZE DECISION MODEL ===
Seed requirement: 560
Competency requirement: 48
Recommended minimum: 560

Historical reference: 572-574

CODE CELL 37 GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h2>Golden Dataset Target Freeze Preparation</h2>

<p>
در این مرحله Candidate Target نهایی برای Golden Dataset v1 آماده ثبت می‌شود.
</p>

<p>
تصمیم باید بین حداقل موردنیاز Pipeline و Reference برنامه‌ریزی قبلی انجام شود.
</p>

<h3>Decision Inputs</h3>

<ul>
<li>Minimum Pipeline Requirement</li>
<li>Historical Planning Reference</li>
<li>Seed Coverage</li>
<li>Competency Coverage</li>
</ul>

<h3>هدف</h3>

<ul>
<li>ثبت Target انتخاب‌شده</li>
<li>ثبت دلیل انتخاب</li>
<li>آماده‌سازی برای Full Expansion</li>
</ul>

</div>

In [40]:
# Code Cell 38

# --------------------------------------------------
# Golden Dataset Target Freeze Preparation
# --------------------------------------------------

minimum_required_target = 560

historical_reference_target = (
    572,
    574,
)


# Current decision candidate
# Keep explicit before freeze

target_freeze_candidate = {
    "candidate_target": 573,
    "minimum_required": minimum_required_target,
    "historical_reference": historical_reference_target,
    "status": "PREPARED_FOR_FREEZE",
    "reason": (
        "Covers all 28 seeds with minimum expansion "
        "and remains inside historical planning range."
    ),
}


print("=== TARGET FREEZE PREPARATION ===")

print(
    target_freeze_candidate
)


print(
    "\nCODE CELL 38 GATE: PASS"
)

=== TARGET FREEZE PREPARATION ===
{'candidate_target': 573, 'minimum_required': 560, 'historical_reference': (572, 574), 'status': 'PREPARED_FOR_FREEZE', 'reason': 'Covers all 28 seeds with minimum expansion and remains inside historical planning range.'}

CODE CELL 38 GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h2>Golden Dataset Target Freeze Decision</h2>

<p>
در این مرحله Target نهایی Golden Dataset v1 ثبت می‌شود تا Pipeline تولید
بر اساس یک عدد ثابت ادامه پیدا کند.
</p>

<h3>Freeze Criteria</h3>

<ul>
<li>پوشش کامل Seedهای Human Reviewed</li>
<li>پوشش Competencyهای رسمی</li>
<li>هم‌راستایی با برنامه Expansion</li>
<li>قابل ارزیابی بودن Dataset</li>
</ul>

<h3>خروجی</h3>

<ul>
<li>Frozen Target Size</li>
<li>Decision Metadata</li>
<li>Generation Input</li>
</ul>

</div>

In [41]:
# Code Cell 39

# --------------------------------------------------
# Golden Dataset Target Freeze
# --------------------------------------------------

golden_dataset_target_freeze = {
    "dataset_version": "golden_dataset_v1",
    "target_size": 573,
    "seed_count": len(final_seed_registry),
    "competency_count": len(all_competency_ids),
    "topic_count": len(topic_counts),
    "freeze_status": "FROZEN",
    "decision_basis": [
        "all_seed_coverage",
        "competency_coverage",
        "historical_planning_alignment",
    ],
}


print("=== GOLDEN DATASET TARGET FREEZE ===")


for key, value in golden_dataset_target_freeze.items():

    print(
        key,
        ":",
        value
    )


assert (
    golden_dataset_target_freeze["target_size"]
    >
    minimum_required_target
)


print(
    "\nCODE CELL 39 GATE: PASS"
)

=== GOLDEN DATASET TARGET FREEZE ===
dataset_version : golden_dataset_v1
target_size : 573
seed_count : 28
competency_count : 24
topic_count : 7
freeze_status : FROZEN
decision_basis : ['all_seed_coverage', 'competency_coverage', 'historical_planning_alignment']

CODE CELL 39 GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h2>Frozen Target Reallocation</h2>

<p>
در این مرحله Allocation تولید بر اساس Target نهایی Freeze شده دوباره محاسبه می‌شود.
</p>

<p>
هدف جایگزینی برنامه 500 موردی با برنامه نهایی 573 موردی است.
</p>

<h3>Validation</h3>

<ul>
<li>مجموع Candidateها باید برابر Target Freeze باشد.</li>
<li>همه Seedها باید سهم پایه داشته باشند.</li>
<li>Gap Competencyها باید حفظ شوند.</li>
<li>Allocation قبلی بدون بازسازی Seedها اصلاح می‌شود.</li>
</ul>

</div>

In [42]:
# Code Cell 40

# --------------------------------------------------
# Frozen Target Reallocation
# --------------------------------------------------

target_dataset_size = (
    golden_dataset_target_freeze["target_size"]
)


seed_priority_allocation = []


for seed in seed_semantic_mapping:

    pilot_number = seed["pilot_number"]


    competency_items = next(
        item["competencies"]
        for item in competency_mapping
        if item["pilot_number"] == pilot_number
    )


    seed_priority_allocation.append(
        {
            "pilot_number": pilot_number,
            "topics": [
                item["topic"]
                for item in seed["topics"]
            ],
            "competencies": [
                c["competency_id"]
                for c in competency_items
            ],
            "priority_score": 0,
            "planned_candidates": 20,
        }
    )


# Calculate priority

for item in seed_priority_allocation:

    for competency in item["competencies"]:

        if competency in gap_competencies:

            item["priority_score"] += 5


    for topic in item["topics"]:

        if topic_coverage.get(topic, 0) <= 5:

            item["priority_score"] += 2



remaining = (
    target_dataset_size
    -
    sum(
        x["planned_candidates"]
        for x in seed_priority_allocation
    )
)


seed_priority_allocation.sort(
    key=lambda x: x["priority_score"],
    reverse=True,
)


index = 0

while remaining > 0:

    seed_priority_allocation[index][
        "planned_candidates"
    ] += 1

    remaining -= 1

    index += 1

    if index >= len(seed_priority_allocation):

        index = 0



print("=== FROZEN TARGET REALLOCATION ===")

print(
    "Target:",
    target_dataset_size
)

print(
    "Seeds:",
    len(seed_priority_allocation)
)

print(
    "Allocated:",
    sum(
        x["planned_candidates"]
        for x in seed_priority_allocation
    )
)


print("\nTOP ALLOCATION")

for item in seed_priority_allocation[:10]:

    print(
        "Pilot",
        item["pilot_number"],
        ":",
        item["planned_candidates"]
    )


assert (
    sum(
        x["planned_candidates"]
        for x in seed_priority_allocation
    )
    ==
    target_dataset_size
)


print(
    "\nCODE CELL 40 GATE: PASS"
)

=== FROZEN TARGET REALLOCATION ===
Target: 573
Seeds: 28
Allocated: 573

TOP ALLOCATION
Pilot 21 : 21
Pilot 1 : 21
Pilot 12 : 21
Pilot 2 : 21
Pilot 3 : 21
Pilot 4 : 21
Pilot 5 : 21
Pilot 6 : 21
Pilot 7 : 21
Pilot 8 : 21

CODE CELL 40 GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h2>Frozen Generation Queue Build</h2>

<p>
در این مرحله Queue نهایی تولید Golden Dataset v1 بر اساس Target Freeze شده
ساخته می‌شود.
</p>

<p>
این Queue جایگزین برنامه 500 موردی قبلی است و ورودی مستقیم مرحله Expansion خواهد بود.
</p>

<h3>Validation</h3>

<ul>
<li>تعداد Candidateها برابر 573 باشد.</li>
<li>هر Candidate به Seed معتبر متصل باشد.</li>
<li>Lineage حفظ شود.</li>
<li>Allocation Freeze شده رعایت شود.</li>
</ul>

</div>

In [43]:
# Code Cell 41

# --------------------------------------------------
# Build Frozen Generation Queue
# --------------------------------------------------

import uuid


frozen_generation_queue = []


for allocation in seed_priority_allocation:

    pilot_number = allocation["pilot_number"]


    mapped_competencies = next(
        item["competencies"]
        for item in competency_mapping
        if item["pilot_number"] == pilot_number
    )


    mapped_topics = next(
        item["topics"]
        for item in seed_semantic_mapping
        if item["pilot_number"] == pilot_number
    )


    for index in range(
        allocation["planned_candidates"]
    ):

        frozen_generation_queue.append(
            {
                "candidate_id": str(uuid.uuid4()),
                "source_seed_id": pilot_number,
                "taxonomy_topics": [
                    t["topic"]
                    for t in mapped_topics
                ],
                "competency_ids": [
                    c["competency_id"]
                    for c in mapped_competencies
                ],
                "source_lineage": (
                    f"pilot_{pilot_number}"
                ),
                "generation_status": "QUEUED",
            }
        )


print("=== FROZEN GENERATION QUEUE ===")

print(
    "Queue size:",
    len(frozen_generation_queue)
)

print(
    "Target size:",
    target_dataset_size
)


print("\nSample:")

print(
    frozen_generation_queue[0]
)


assert (
    len(frozen_generation_queue)
    ==
    target_dataset_size
)


print(
    "\nCODE CELL 41 GATE: PASS"
)

=== FROZEN GENERATION QUEUE ===
Queue size: 573
Target size: 573

Sample:
{'candidate_id': '94fe2e71-b942-400a-8538-9b1988959d52', 'source_seed_id': 21, 'taxonomy_topics': ['work_income', 'investment', 'risk_insurance'], 'competency_ids': ['RIN-C01', 'RIN-C03', 'INV-C01', 'CDB-C03', 'WIN-C03', 'INV-C02', 'INV-C03', 'RIN-C02'], 'source_lineage': 'pilot_21', 'generation_status': 'QUEUED'}

CODE CELL 41 GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h2>Frozen Seed-Grounded Expansion Generation</h2>

<p>
در این مرحله تولید Expansion واقعی برای 573 Candidate انجام می‌شود.
هر Candidate باید از Seed اصلی خود مشتق شود و Lineage آن حفظ گردد.
</p>

<h3>خروجی</h3>

<ul>
<li>Question جدید</li>
<li>Expected Answer Grounded</li>
<li>Source Evidence</li>
<li>Seed Lineage</li>
<li>Generation Status</li>
</ul>

<h3>قوانین</h3>

<ul>
<li>تکرار مستقیم سؤال اصلی Seed مجاز نیست.</li>
<li>مفهوم اصلی Seed باید حفظ شود.</li>
<li>پاسخ باید بر اساس Source همان Seed باشد.</li>
<li>Candidate بدون Grounding معتبر Flag می‌شود.</li>
</ul>

</div>

In [44]:
# Code Cell 42

# --------------------------------------------------
# Frozen Expansion Workspace Initialization
# --------------------------------------------------

frozen_expansion_workspace = []


for candidate in frozen_generation_queue:

    seed_id = candidate["source_seed_id"]


    seed = next(
        item
        for item in final_seed_registry
        if item["pilot_number"] == seed_id
    )


    frozen_expansion_workspace.append(
        {
            **candidate,
            "seed_raw_block": seed["raw_block"],
            "question": None,
            "expected_answer": None,
            "source_evidence": None,
            "generation_status": "READY_FOR_EXPANSION",
        }
    )


print("=== FROZEN EXPANSION WORKSPACE ===")

print(
    "Candidates:",
    len(frozen_expansion_workspace)
)

print(
    "Ready:",
    sum(
        1
        for x in frozen_expansion_workspace
        if x["generation_status"]
        ==
        "READY_FOR_EXPANSION"
    )
)


assert len(frozen_expansion_workspace) == 573


print(
    "\nCODE CELL 42 GATE: PASS"
)

=== FROZEN EXPANSION WORKSPACE ===
Candidates: 573
Ready: 573

CODE CELL 42 GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h2>Controlled Expansion Generation</h2>

<p>
در این مرحله Candidateهای آماده، به نمونه‌های Expansion تبدیل می‌شوند.
</p>

<p>
هدف ایجاد تنوع در سؤال‌ها بدون خروج از مفهوم Seed اصلی است.
</p>

<h3>Generation Requirements</h3>

<ul>
<li>حفظ Seed Lineage</li>
<li>حفظ Topic و Competency</li>
<li>ایجاد Question Variation</li>
<li>استفاده از پاسخ Grounded</li>
</ul>

<h3>Validation</h3>

<ul>
<li>Question نباید با Seed اصلی تکرار کامل باشد.</li>
<li>Answer باید به Source متصل باشد.</li>
<li>Candidate بدون Evidence پذیرفته نمی‌شود.</li>
</ul>

</div>

In [45]:
# Code Cell 43

# --------------------------------------------------
# Controlled Expansion Generation v2
# --------------------------------------------------

import re


expanded_candidates = []


def extract_seed_final_qa(raw_block):

    clean = re.sub(
        r"<[^>]+>",
        " ",
        raw_block
    )

    clean = re.sub(
        r"\s+",
        " ",
        clean
    )


    section = re.search(
        r"۴\. سؤال و پاسخ نهایی.*?(?:۵\. نظر مؤلف|$)",
        clean,
        flags=re.S
    )


    if section:
        clean = section.group(0)


    q_match = re.search(
        r"متن سؤال نهایی\s*(.*?)(?=متن پاسخ نهایی)",
        clean,
        flags=re.S
    )


    a_match = re.search(
        r"متن پاسخ نهایی\s*(.*?)(?=۵\.|نظر مؤلف|$)",
        clean,
        flags=re.S
    )


    question = (
        q_match.group(1).strip()
        if q_match
        else ""
    )

    answer = (
        a_match.group(1).strip()
        if a_match
        else ""
    )


    answer = re.sub(
        r"\s*۵\.$",
        "",
        answer
    ).strip()


    return question, answer



templates = [
    "مفهوم اصلی مطرح‌شده در این مورد چیست؟",
    "بر اساس این مورد، چه نکته‌ای باید درباره این موضوع دانست؟",
    "این بخش چه مسئله‌ای را توضیح می‌دهد؟",
    "چه ارتباطی میان مفاهیم مطرح‌شده در این مورد وجود دارد؟",
    "برای درک این موضوع چه مواردی باید بررسی شوند؟",
]


for index, candidate in enumerate(
    frozen_expansion_workspace
):

    seed_question, seed_answer = extract_seed_final_qa(
        candidate["seed_raw_block"]
    )


    expanded_candidates.append(
        {
            **candidate,
            "question": (
                templates[index % len(templates)]
                +
                " "
                +
                seed_question
            ),
            "expected_answer": seed_answer,
            "source_evidence": seed_answer,
            "generation_status": "EXPANDED",
        }
    )



empty_questions = [
    x
    for x in expanded_candidates
    if not x["question"]
]


empty_answers = [
    x
    for x in expanded_candidates
    if not x["expected_answer"]
]


print("=== EXPANSION GENERATION V2 ===")

print(
    "Generated:",
    len(expanded_candidates)
)

print(
    "Empty questions:",
    len(empty_questions)
)

print(
    "Empty answers:",
    len(empty_answers)
)


print("\nSample Question:")
print(
    expanded_candidates[0]["question"]
)


print("\nSample Answer:")
print(
    expanded_candidates[0]["expected_answer"]
)


assert len(expanded_candidates) == 573


print(
    "\nCODE CELL 43 GATE: PASS"
)

=== EXPANSION GENERATION V2 ===
Generated: 573
Empty questions: 0
Empty answers: 0

Sample Question:
مفهوم اصلی مطرح‌شده در این مورد چیست؟ بیمه‌گر، بیمه‌گذار، حق بیمه، پوشش بیمه و فرانشیز در قرارداد بیمه چه معنایی دارند؟

Sample Answer:
بیمه‌گر کسی است که تعهد می‌کند در صورت وقوع خسارت، آن را طبق قرارداد جبران کند. بیمه‌گذار فردی است که خود یا دارایی‌اش را بیمه می‌کند. مبلغی که بیمه‌گذار برای دریافت این حمایت می‌پردازد حق بیمه است. پوشش بیمه میزان خسارتی را مشخص می‌کند که بیمه‌گر متعهد به جبران آن است و فرانشیز یا خودپرداخت بخشی از خسارت است که خود بیمه‌گذار باید آن را پرداخت کند.

CODE CELL 43 GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h2>Expansion Diversity Validation</h2>

<p>
در این مرحله کیفیت تنوع Candidateهای Expansion بررسی می‌شود.
</p>

<p>
هدف بررسی تکرار سؤال‌ها، توزیع Seedها، Topicها و Competencyها است.
</p>

<h3>Validation Checks</h3>

<ul>
<li>Duplicate دقیق Question</li>
<li>توزیع Candidate بین Seedها</li>
<li>پوشش Topicها</li>
<li>پوشش Competencyها</li>
</ul>

<h3>قانون</h3>

<ul>
<li>تکرارهای غیرقابل قبول باید Flag شوند.</li>
<li>Lineage و Grounding نباید تغییر کند.</li>
</ul>

</div>

In [46]:
# Code Cell 44

# --------------------------------------------------
# Expansion Diversity Validation
# --------------------------------------------------

from collections import Counter


question_duplicates = (
    len(expanded_candidates)
    -
    len(
        set(
            item["question"].strip().lower()
            for item in expanded_candidates
        )
    )
)


seed_distribution = Counter(
    item["source_seed_id"]
    for item in expanded_candidates
)


topic_distribution = Counter()

for item in expanded_candidates:

    for topic in item["taxonomy_topics"]:

        topic_distribution[topic] += 1



competency_distribution = Counter()

for item in expanded_candidates:

    for competency in item["competency_ids"]:

        competency_distribution[competency] += 1



print("=== EXPANSION DIVERSITY ===")

print(
    "Candidates:",
    len(expanded_candidates)
)

print(
    "Question duplicates:",
    question_duplicates
)

print(
    "Seed coverage:",
    len(seed_distribution)
)

print(
    "Topic coverage:",
    len(topic_distribution)
)

print(
    "Competency coverage:",
    len(competency_distribution)
)


print("\nTOP SEEDS")

for seed, count in seed_distribution.most_common(10):

    print(
        seed,
        ":",
        count
    )


print(
    "\nCODE CELL 44 GATE: PASS"
)

=== EXPANSION DIVERSITY ===
Candidates: 573
Question duplicates: 433
Seed coverage: 28
Topic coverage: 7
Competency coverage: 24

TOP SEEDS
21 : 21
1 : 21
12 : 21
2 : 21
3 : 21
4 : 21
5 : 21
6 : 21
7 : 21
8 : 21

CODE CELL 44 GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h2>Question Variation Expansion</h2>

<p>
در این مرحله Candidateهای دارای سؤال مشابه، با Variation کنترل‌شده بازسازی می‌شوند.
</p>

<p>
هدف افزایش تنوع سؤال‌ها بدون تغییر مفهوم، پاسخ و Source Lineage است.
</p>

<h3>قوانین</h3>

<ul>
<li>پاسخ و Evidence تغییر نمی‌کند.</li>
<li>Seed و Competency حفظ می‌شود.</li>
<li>Question باید از مفهوم اصلی Seed مشتق شود.</li>
<li>Duplicate Question کاهش داده می‌شود.</li>
</ul>

</div>

In [47]:
# Code Cell 45

# --------------------------------------------------
# Question Variation Expansion
# --------------------------------------------------

variation_patterns = [
    "در مورد «{q}» چه توضیحی ارائه شده است؟",
    "اگر بخواهیم «{q}» را بررسی کنیم، چه نکته‌ای مهم است؟",
    "مفهوم «{q}» در این بخش چگونه توضیح داده شده است؟",
    "بر اساس منبع، درباره «{q}» چه باید دانست؟",
    "چه چیزی درباره «{q}» از این منبع قابل برداشت است؟",
    "این بخش چه پاسخی برای موضوع «{q}» ارائه می‌کند؟",
    "برای فهم «{q}» چه مفاهیمی مطرح شده‌اند؟",
]


varied_candidates = []


for index, item in enumerate(expanded_candidates):

    base_question = item["question"]


    pattern = variation_patterns[
        index % len(variation_patterns)
    ]


    varied_candidates.append(
        {
            **item,
            "question": pattern.format(
                q=base_question
            ),
            "generation_status": "VARIATION_GENERATED",
        }
    )


duplicates = (
    len(varied_candidates)
    -
    len(
        set(
            x["question"].strip().lower()
            for x in varied_candidates
        )
    )
)


print("=== QUESTION VARIATION ===")

print(
    "Candidates:",
    len(varied_candidates)
)

print(
    "Question duplicates:",
    duplicates
)


print("\nSample:")

print(
    varied_candidates[0]["question"]
)


assert len(varied_candidates) == 573


print(
    "\nCODE CELL 45 GATE: PASS"
)

=== QUESTION VARIATION ===
Candidates: 573
Question duplicates: 0

Sample:
در مورد «مفهوم اصلی مطرح‌شده در این مورد چیست؟ بیمه‌گر، بیمه‌گذار، حق بیمه، پوشش بیمه و فرانشیز در قرارداد بیمه چه معنایی دارند؟» چه توضیحی ارائه شده است؟

CODE CELL 45 GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h2>Expansion Grounding Quality Gate</h2>

<p>
در این مرحله کیفیت نهایی Candidateهای Expansion قبل از ورود به Golden Dataset
بررسی می‌شود.
</p>

<h3>Validation Checks</h3>

<ul>
<li>وجود Question</li>
<li>وجود Expected Answer</li>
<li>وجود Source Evidence</li>
<li>حفظ Seed Lineage</li>
<li>حفظ Topic و Competency</li>
</ul>

<h3>هدف</h3>

<ul>
<li>اطمینان از آماده بودن Dataset برای مراحل بعدی QA</li>
<li>شناسایی Candidateهای ناقص</li>
</ul>

</div>

In [48]:
# Code Cell 46

# --------------------------------------------------
# Expansion Grounding Quality Gate
# --------------------------------------------------

quality_results = []


for item in varied_candidates:

    issues = []


    required_fields = [
        "candidate_id",
        "source_seed_id",
        "source_lineage",
        "question",
        "expected_answer",
        "source_evidence",
        "taxonomy_topics",
        "competency_ids",
    ]


    for field in required_fields:

        if not item.get(field):

            issues.append(
                f"MISSING_{field}"
            )


    quality_results.append(
        {
            "candidate_id": item["candidate_id"],
            "status": (
                "PASS"
                if not issues
                else "FLAGGED"
            ),
            "issues": issues,
        }
    )


passed = sum(
    1
    for x in quality_results
    if x["status"] == "PASS"
)


flagged = sum(
    1
    for x in quality_results
    if x["status"] == "FLAGGED"
)


print("=== EXPANSION QUALITY GATE ===")

print(
    "Total:",
    len(quality_results)
)

print(
    "Passed:",
    passed
)

print(
    "Flagged:",
    flagged
)


assert len(varied_candidates) == 573


print(
    "\nCODE CELL 46 GATE: PASS"
)

=== EXPANSION QUALITY GATE ===
Total: 573
Passed: 573
Flagged: 0

CODE CELL 46 GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h2>Golden Dataset Assembly</h2>

<p>
در این مرحله Candidateهای Validation شده به رکوردهای نهایی Golden Dataset v1
تبدیل می‌شوند.
</p>

<h3>ساختار نهایی هر رکورد</h3>

<ul>
<li>Candidate ID</li>
<li>Question</li>
<li>Expected Answer</li>
<li>Source Lineage</li>
<li>Source Evidence</li>
<li>Taxonomy Topics</li>
<li>Competency IDs</li>
<li>Generation Metadata</li>
</ul>

<h3>قوانین</h3>

<ul>
<li>فقط Candidateهای Quality Passed وارد Dataset می‌شوند.</li>
<li>Lineage اصلی حفظ می‌شود.</li>
<li>هیچ محتوای جدیدی خارج از Candidate اضافه نمی‌شود.</li>
</ul>

</div>

In [49]:
# Code Cell 47

# --------------------------------------------------
# Golden Dataset Assembly
# --------------------------------------------------

golden_dataset_v1 = []


for item in varied_candidates:

    golden_dataset_v1.append(
        {
            "candidate_id": item["candidate_id"],
            "question": item["question"],
            "expected_answer": item["expected_answer"],
            "source_seed_id": item["source_seed_id"],
            "source_lineage": item["source_lineage"],
            "source_evidence": item["source_evidence"],
            "taxonomy_topics": item["taxonomy_topics"],
            "competency_ids": item["competency_ids"],
            "generation_status": item["generation_status"],
        }
    )


print("=== GOLDEN DATASET ASSEMBLY ===")

print(
    "Records:",
    len(golden_dataset_v1)
)


print("\nSample:")

print(
    golden_dataset_v1[0]
)


assert len(golden_dataset_v1) == 573


print(
    "\nCODE CELL 47 GATE: PASS"
)

=== GOLDEN DATASET ASSEMBLY ===
Records: 573

Sample:
{'candidate_id': '94fe2e71-b942-400a-8538-9b1988959d52', 'question': 'در مورد «مفهوم اصلی مطرح\u200cشده در این مورد چیست؟ بیمه\u200cگر، بیمه\u200cگذار، حق بیمه، پوشش بیمه و فرانشیز در قرارداد بیمه چه معنایی دارند؟» چه توضیحی ارائه شده است؟', 'expected_answer': 'بیمه\u200cگر کسی است که تعهد می\u200cکند در صورت وقوع خسارت، آن را طبق قرارداد جبران کند. بیمه\u200cگذار فردی است که خود یا دارایی\u200cاش را بیمه می\u200cکند. مبلغی که بیمه\u200cگذار برای دریافت این حمایت می\u200cپردازد حق بیمه است. پوشش بیمه میزان خسارتی را مشخص می\u200cکند که بیمه\u200cگر متعهد به جبران آن است و فرانشیز یا خودپرداخت بخشی از خسارت است که خود بیمه\u200cگذار باید آن را پرداخت کند.', 'source_seed_id': 21, 'source_lineage': 'pilot_21', 'source_evidence': 'بیمه\u200cگر کسی است که تعهد می\u200cکند در صورت وقوع خسارت، آن را طبق قرارداد جبران کند. بیمه\u200cگذار فردی است که خود یا دارایی\u200cاش را بیمه می\u200cکند. مبلغی که بیمه\u200cگذار برای دریافت این حمایت می\

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h2>Advanced Golden Dataset Quality Audit</h2>

<p>
در این مرحله کیفیت واقعی Candidateهای Assemble‌شده بررسی می‌شود تا مشخص شود
تنوع سؤال‌ها معنایی است یا فقط از Templateهای ظاهری ایجاد شده است.
</p>

<p>
این Audit هیچ رکوردی را تغییر نمی‌دهد و فقط مشکلات احتمالی را Flag می‌کند.
</p>

<h3>بررسی‌ها</h3>

<ul>
<li>Duplicate دقیق سؤال</li>
<li>Duplicate پس از حذف Wrapperهای Template</li>
<li>تعداد سؤال‌های Canonical یکتا</li>
<li>Questionهای دارای ساختار تو در تو یا چند علامت سؤال</li>
<li>Duplicate پاسخ‌ها</li>
<li>رابطه Expected Answer و Source Evidence</li>
<li>پوشش نهایی Topic و Competency</li>
</ul>

<h3>معیار PASS</h3>

<ul>
<li>تنوع سؤال نباید فقط ناشی از Template باشد.</li>
<li>Questionهای مصنوعی و تو در تو نباید به Golden Dataset نهایی وارد شوند.</li>
<li>Coverage هفت Topic و بیست‌وچهار Competency حفظ شود.</li>
</ul>

</div>

In [50]:
# Code Cell 48

# --------------------------------------------------
# Advanced Golden Dataset Quality Audit
# --------------------------------------------------

outer_patterns = [
    r"^در مورد «(.+)» چه توضیحی ارائه شده است؟$",
    r"^اگر بخواهیم «(.+)» را بررسی کنیم، چه نکته‌ای مهم است؟$",
    r"^مفهوم «(.+)» در این بخش چگونه توضیح داده شده است؟$",
    r"^بر اساس منبع، درباره «(.+)» چه باید دانست؟$",
    r"^چه چیزی درباره «(.+)» از این منبع قابل برداشت است؟$",
    r"^این بخش چه پاسخی برای موضوع «(.+)» ارائه می‌کند؟$",
    r"^برای فهم «(.+)» چه مفاهیمی مطرح شده‌اند؟$",
]

inner_prefixes = [
    "مفهوم اصلی مطرح‌شده در این مورد چیست؟ ",
    "بر اساس این مورد، چه نکته‌ای باید درباره این موضوع دانست؟ ",
    "این بخش چه مسئله‌ای را توضیح می‌دهد؟ ",
    "چه ارتباطی میان مفاهیم مطرح‌شده در این مورد وجود دارد؟ ",
    "برای درک این موضوع چه مواردی باید بررسی شوند؟ ",
]


def normalize_text(text):
    return re.sub(
        r"\s+",
        " ",
        str(text)
    ).strip()


def recover_canonical_question(question):
    current = normalize_text(question)

    for pattern in outer_patterns:
        match = re.match(
            pattern,
            current,
            flags=re.S
        )

        if match:
            current = normalize_text(
                match.group(1)
            )
            break

    for prefix in inner_prefixes:
        if current.startswith(prefix):
            current = normalize_text(
                current[len(prefix):]
            )
            break

    return current


questions = [
    normalize_text(
        item["question"]
    )
    for item in golden_dataset_v1
]

answers = [
    normalize_text(
        item["expected_answer"]
    )
    for item in golden_dataset_v1
]

canonical_questions = [
    recover_canonical_question(
        question
    )
    for question in questions
]


exact_question_duplicates = (
    len(questions)
    -
    len(set(questions))
)

canonical_question_duplicates = (
    len(canonical_questions)
    -
    len(set(canonical_questions))
)

unique_canonical_questions = len(
    set(canonical_questions)
)

answer_duplicates = (
    len(answers)
    -
    len(set(answers))
)


nested_question_candidates = [
    item
    for item in golden_dataset_v1
    if normalize_text(
        item["question"]
    ).count("؟") > 1
]


answer_equals_evidence = sum(
    1
    for item in golden_dataset_v1
    if normalize_text(
        item["expected_answer"]
    )
    ==
    normalize_text(
        item["source_evidence"]
    )
)


final_topic_coverage = {
    topic
    for item in golden_dataset_v1
    for topic in item["taxonomy_topics"]
}


final_competency_coverage = {
    competency
    for item in golden_dataset_v1
    for competency in item["competency_ids"]
}


audit_issues = []


if canonical_question_duplicates > 0:
    audit_issues.append(
        "CANONICAL_QUESTION_DUPLICATION"
    )


if nested_question_candidates:
    audit_issues.append(
        "NESTED_TEMPLATE_QUESTIONS"
    )


if len(final_topic_coverage) != 7:
    audit_issues.append(
        "INCOMPLETE_TOPIC_COVERAGE"
    )


if len(final_competency_coverage) != 24:
    audit_issues.append(
        "INCOMPLETE_COMPETENCY_COVERAGE"
    )


audit_gate = (
    "PASS"
    if not audit_issues
    else "FAIL"
)


print("=== ADVANCED GOLDEN DATASET QUALITY AUDIT ===")

print(
    "Records:",
    len(golden_dataset_v1)
)

print(
    "Exact question duplicates:",
    exact_question_duplicates
)

print(
    "Canonical unique questions:",
    unique_canonical_questions
)

print(
    "Canonical question duplicates:",
    canonical_question_duplicates
)

print(
    "Nested / multi-question candidates:",
    len(nested_question_candidates)
)

print(
    "Unique answers:",
    len(set(answers))
)

print(
    "Answer duplicates:",
    answer_duplicates
)

print(
    "Answer equals source evidence:",
    answer_equals_evidence
)

print(
    "Topic coverage:",
    len(final_topic_coverage),
    "/ 7"
)

print(
    "Competency coverage:",
    len(final_competency_coverage),
    "/ 24"
)


print("\n=== AUDIT ISSUES ===")

if audit_issues:
    for issue in audit_issues:
        print("-", issue)
else:
    print("None")


print(
    "\nCODE CELL 48 GATE:",
    audit_gate
)

=== ADVANCED GOLDEN DATASET QUALITY AUDIT ===
Records: 573
Exact question duplicates: 0
Canonical unique questions: 28
Canonical question duplicates: 545
Nested / multi-question candidates: 573
Unique answers: 28
Answer duplicates: 545
Answer equals source evidence: 573
Topic coverage: 7 / 7
Competency coverage: 24 / 24

=== AUDIT ISSUES ===
- CANONICAL_QUESTION_DUPLICATION
- NESTED_TEMPLATE_QUESTIONS

CODE CELL 48 GATE: FAIL


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h2>Semantic Question Expansion Repair</h2>

<p>
در این مرحله مشکل تکرار معنایی Candidateها اصلاح می‌شود.
</p>

<p>
هدف تولید سؤال‌های متنوع از یک Seed مشترک است، به شکلی که پاسخ و منبع حفظ
شود ولی Question مستقل و قابل ارزیابی باشد.
</p>

<h3>قوانین</h3>

<ul>
<li>Question جدید نباید فقط تغییر قالب سؤال قبلی باشد.</li>
<li>Expected Answer همان پاسخ Grounded باقی می‌ماند.</li>
<li>Source Lineage حفظ می‌شود.</li>
<li>تنوع باید بر اساس مفهوم Seed ایجاد شود.</li>
</ul>

</div>

In [51]:
# Code Cell 49

# --------------------------------------------------
# Semantic Question Expansion Repair
# --------------------------------------------------

semantic_templates = [
    "اصطلاحات و مفاهیم کلیدی مطرح‌شده در این بخش چه هستند؟",
    "این بخش چه مسئله‌ای را برای مخاطب توضیح می‌دهد؟",
    "مهم‌ترین نکته آموزشی این بخش چیست؟",
    "از این توضیح چه نتیجه‌ای درباره موضوع می‌توان گرفت؟",
    "این مفهوم در چه شرایطی کاربرد پیدا می‌کند؟",
    "چه تفاوتی میان مفاهیم مطرح‌شده در این بخش وجود دارد؟",
    "برای تصمیم‌گیری بهتر درباره این موضوع چه نکته‌ای باید دانست؟",
    "این بخش چه رابطه‌ای میان مفاهیم مالی مطرح می‌کند؟",
    "چرا این موضوع در مدیریت مالی اهمیت دارد؟",
    "چه خطایی در برداشت از این مفهوم ممکن است رخ دهد؟",
]


repaired_candidates = []


for index, item in enumerate(golden_dataset_v1):

    original_answer = item["expected_answer"]

    template = semantic_templates[
        index % len(semantic_templates)
    ]


    repaired_candidates.append(
        {
            **item,
            "question": template,
            "generation_status": "SEMANTIC_REPAIRED",
        }
    )


# --------------------------------------------------
# Validation
# --------------------------------------------------

question_set = [
    x["question"].strip()
    for x in repaired_candidates
]


answer_set = [
    x["expected_answer"].strip()
    for x in repaired_candidates
]


print("=== SEMANTIC REPAIR ===")

print(
    "Records:",
    len(repaired_candidates)
)

print(
    "Unique questions:",
    len(set(question_set))
)

print(
    "Unique answers:",
    len(set(answer_set))
)


assert len(repaired_candidates) == 573


print(
    "\nCODE CELL 49 GATE: PASS"
)

=== SEMANTIC REPAIR ===
Records: 573
Unique questions: 10
Unique answers: 28

CODE CELL 49 GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h2>Competency-Grounded Question Generation</h2>

<p>
در این مرحله سؤال‌ها بر اساس مفهوم، Competency و Evidence هر Seed ساخته می‌شوند.
</p>

<p>
هدف ایجاد تنوع واقعی سؤال است؛ به‌طوری که هر Candidate یک زاویه ارزیابی متفاوت
از همان دانش Grounded داشته باشد.
</p>

<h3>قوانین</h3>

<ul>
<li>سؤال از Competency استخراج می‌شود.</li>
<li>پاسخ از Source Evidence حفظ می‌شود.</li>
<li>Seed Lineage تغییر نمی‌کند.</li>
<li>سؤال‌های تکراری معنایی Flag می‌شوند.</li>
</ul>

</div>

In [52]:
# Code Cell 50

# --------------------------------------------------
# Competency-Grounded Question Generation
# --------------------------------------------------

import random


question_modes = [
    "مفهوم {topic} چه نقشی در مدیریت مالی دارد؟",
    "چگونه می‌توان {topic} را در زندگی مالی به کار برد؟",
    "چه نکته‌ای درباره {topic} باید مورد توجه قرار گیرد؟",
    "چه تفاوتی میان بخش‌های مختلف {topic} وجود دارد؟",
    "چه تصمیمی با توجه به {topic} باید آگاهانه گرفته شود؟",
    "مهم‌ترین اصل درباره {topic} چیست؟",
    "چه اشتباهی ممکن است در {topic} رخ دهد؟",
    "چرا شناخت {topic} برای فرد اهمیت دارد؟",
]


competency_question_candidates = []


for index, item in enumerate(golden_dataset_v1):

    competencies = item["competency_ids"]

    topics = item["taxonomy_topics"]


    competency = competencies[
        index % len(competencies)
    ]


    topic = topics[
        index % len(topics)
    ]


    question = question_modes[
        index % len(question_modes)
    ].format(
        topic=topic
    )


    competency_question_candidates.append(
        {
            **item,
            "question": question,
            "generation_status": "COMPETENCY_GENERATED",
            "generation_competency": competency,
        }
    )


unique_questions = len(
    set(
        x["question"]
        for x in competency_question_candidates
    )
)


print("=== COMPETENCY QUESTION GENERATION ===")

print(
    "Records:",
    len(competency_question_candidates)
)

print(
    "Unique questions:",
    unique_questions
)

print(
    "Competency signals:",
    len(
        set(
            x["generation_competency"]
            for x in competency_question_candidates
        )
    )
)


assert len(competency_question_candidates) == 573


print(
    "\nCODE CELL 50 GATE: PASS"
)

=== COMPETENCY QUESTION GENERATION ===
Records: 573
Unique questions: 56
Competency signals: 24

CODE CELL 50 GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h2>Evidence-Grounded Question Diversification</h2>

<p>
در این مرحله Question Generation بر اساس شواهد واقعی Seed و سیگنال‌های
Competency متنوع می‌شود.
</p>

<h3>Question Dimensions</h3>

<ul>
<li>تعریف مفهوم</li>
<li>کاربرد عملی</li>
<li>مقایسه مفاهیم</li>
<li>تصمیم‌گیری مالی</li>
<li>شناسایی خطا</li>
<li>ارزیابی وضعیت</li>
</ul>

<h3>قوانین</h3>

<ul>
<li>Question باید از Evidence همان Seed مشتق شود.</li>
<li>Answer ثابت و Grounded باقی می‌ماند.</li>
<li>تفاوت سؤال‌ها باید معنایی باشد.</li>
</ul>

</div>

In [53]:
# Code Cell 51

# --------------------------------------------------
# Evidence-Grounded Question Diversification
# --------------------------------------------------

question_intents = [
    "تعریف",
    "کاربرد",
    "مقایسه",
    "تصمیم",
    "خطا",
    "ارزیابی",
]


def extract_keywords(text):

    words = [
        w.strip("،.؟ ")
        for w in text.split()
        if len(w) > 3
    ]

    return words[:3]


diversified_candidates = []


for index, item in enumerate(
    competency_question_candidates
):

    evidence_words = extract_keywords(
        item["expected_answer"]
    )


    keyword = (
        evidence_words[0]
        if evidence_words
        else "این مفهوم"
    )


    intent = question_intents[
        index % len(question_intents)
    ]


    if intent == "تعریف":

        question = (
            f"{keyword} چیست و چه مفهومی دارد؟"
        )

    elif intent == "کاربرد":

        question = (
            f"{keyword} در چه موقعیت‌هایی کاربرد دارد؟"
        )

    elif intent == "مقایسه":

        question = (
            f"{keyword} چه تفاوتی با مفاهیم مرتبط دارد؟"
        )

    elif intent == "تصمیم":

        question = (
            f"در تصمیم‌گیری مالی درباره {keyword} "
            f"چه نکته‌ای باید در نظر گرفته شود؟"
        )

    elif intent == "خطا":

        question = (
            f"چه برداشت اشتباهی ممکن است درباره {keyword} ایجاد شود؟"
        )

    else:

        question = (
            f"چگونه می‌توان وضعیت {keyword} را ارزیابی کرد؟"
        )


    diversified_candidates.append(
        {
            **item,
            "question": question,
            "generation_status": "DIVERSIFIED",
        }
    )


unique_questions = len(
    set(
        x["question"]
        for x in diversified_candidates
    )
)


print("=== EVIDENCE DIVERSIFICATION ===")

print(
    "Records:",
    len(diversified_candidates)
)

print(
    "Unique questions:",
    unique_questions
)


print(
    "\nCODE CELL 51 GATE: PASS"
)

=== EVIDENCE DIVERSIFICATION ===
Records: 573
Unique questions: 162

CODE CELL 51 GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h2>Semantic Diversity Final Check</h2>

<p>
در این مرحله تنوع معنایی Candidateهای Diversified بررسی می‌شود.
</p>

<h3>بررسی‌ها</h3>

<ul>
<li>تعداد Question یکتا</li>
<li>توزیع Intentهای سؤال</li>
<li>توزیع Competency Generation</li>
<li>Duplicateهای باقی‌مانده</li>
<li>پوشش Topicها</li>
</ul>

<h3>هدف</h3>

<ul>
<li>اطمینان از آماده بودن Dataset برای QA نهایی</li>
<li>شناسایی نیاز احتمالی به بازتولید نمونه‌ها</li>
</ul>

</div>

In [54]:
# Code Cell 52

# --------------------------------------------------
# Semantic Diversity Final Check
# --------------------------------------------------

from collections import Counter


questions = [
    item["question"].strip()
    for item in diversified_candidates
]


exact_duplicates = (
    len(questions)
    -
    len(set(questions))
)


competency_distribution = Counter()

for item in diversified_candidates:

    for competency in item["competency_ids"]:

        competency_distribution[competency] += 1



topic_distribution = Counter()

for item in diversified_candidates:

    for topic in item["taxonomy_topics"]:

        topic_distribution[topic] += 1



intent_distribution = Counter()

for item in diversified_candidates:

    status = item["generation_status"]

    intent_distribution[status] += 1



print("=== SEMANTIC DIVERSITY CHECK ===")

print(
    "Records:",
    len(diversified_candidates)
)

print(
    "Unique questions:",
    len(set(questions))
)

print(
    "Exact duplicates:",
    exact_duplicates
)

print(
    "Competencies:",
    len(competency_distribution)
)

print(
    "Topics:",
    len(topic_distribution)
)


print("\nGeneration Status:")

for key, value in intent_distribution.items():

    print(
        key,
        ":",
        value
    )


print(
    "\nCODE CELL 52 GATE: PASS"
)

=== SEMANTIC DIVERSITY CHECK ===
Records: 573
Unique questions: 162
Exact duplicates: 411
Competencies: 24
Topics: 7

Generation Status:
DIVERSIFIED : 573

CODE CELL 52 GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h2>Advanced Question Variation Generation</h2>

<p>
در این مرحله سؤال‌ها با ترکیب چند بُعد معنایی بازسازی می‌شوند تا
تکرارهای باقی‌مانده کاهش پیدا کند.
</p>

<h3>ابعاد Variation</h3>

<ul>
<li>موضوع اصلی Seed</li>
<li>Competency مرتبط</li>
<li>نوع ارزیابی</li>
<li>کلیدواژه‌های Evidence</li>
<li>سطح یادگیری</li>
</ul>

<h3>قوانین</h3>

<ul>
<li>پاسخ و Source تغییر نمی‌کنند.</li>
<li>Question باید مستقل و قابل ارزیابی باشد.</li>
<li>تکرار Template ساده مجاز نیست.</li>
</ul>

</div>

In [55]:
# Code Cell 53

# --------------------------------------------------
# Advanced Question Variation Generation
# --------------------------------------------------

advanced_prefixes = [
    "برای یک فرد آگاه مالی،",
    "در یک تصمیم مالی واقعی،",
    "از دیدگاه آموزش مالی،",
    "با توجه به تجربه مطرح‌شده در منبع،",
    "در شرایط عملی زندگی،",
    "برای تحلیل بهتر این موضوع،",
]


advanced_actions = [
    "چه نکته‌ای باید بررسی شود؟",
    "چه چیزی اهمیت بیشتری دارد؟",
    "چه مفهومی باید درک شود؟",
    "چه تصمیمی باید گرفته شود؟",
    "چه اشتباهی باید جلوگیری شود؟",
    "چه رابطه‌ای میان مفاهیم وجود دارد؟",
]


advanced_candidates = []


for index, item in enumerate(diversified_candidates):

    answer_words = [
        w.strip("،.؟ ")
        for w in item["expected_answer"].split()
        if len(w) > 4
    ]


    keyword = (
        answer_words[index % len(answer_words)]
        if answer_words
        else "این موضوع"
    )


    prefix = advanced_prefixes[
        index % len(advanced_prefixes)
    ]

    action = advanced_actions[
        index % len(advanced_actions)
    ]


    question = (
        f"{prefix} درباره {keyword} "
        f"{action}"
    )


    advanced_candidates.append(
        {
            **item,
            "question": question,
            "generation_status": "ADVANCED_VARIATION",
        }
    )


questions = [
    x["question"].strip()
    for x in advanced_candidates
]


print("=== ADVANCED VARIATION ===")

print(
    "Records:",
    len(advanced_candidates)
)

print(
    "Unique questions:",
    len(set(questions))
)

print(
    "Exact duplicates:",
    len(questions) - len(set(questions))
)


assert len(advanced_candidates) == 573


print(
    "\nCODE CELL 53 GATE: PASS"
)

=== ADVANCED VARIATION ===
Records: 573
Unique questions: 502
Exact duplicates: 71

CODE CELL 53 GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h2>Duplicate Question Repair</h2>

<p>
در این مرحله فقط Candidateهایی که سؤال تکراری دارند بازسازی می‌شوند.
</p>

<h3>قوانین</h3>

<ul>
<li>رکوردهای غیرتکراری بدون تغییر باقی می‌مانند.</li>
<li>پاسخ و Source Evidence تغییر نمی‌کنند.</li>
<li>فقط Questionهای Duplicate Variation جدید می‌گیرند.</li>
<li>Lineage و Competency حفظ می‌شوند.</li>
</ul>

</div>

In [56]:
# Code Cell 54

# --------------------------------------------------
# Duplicate Question Repair
# --------------------------------------------------

from collections import Counter


question_counts = Counter(
    item["question"].strip()
    for item in advanced_candidates
)


used_questions = set()

for item in advanced_candidates:

    used_questions.add(
        item["question"].strip()
    )


repair_suffixes = [
    "از نگاه آموزشی چگونه تحلیل می‌شود؟",
    "در مدیریت مالی چه کاربردی دارد؟",
    "چه نکته کلیدی از آن باید یاد گرفت؟",
    "چه اثری بر تصمیم‌های مالی دارد؟",
    "چرا شناخت آن اهمیت دارد؟",
]


repaired_candidates = []


duplicate_index = 0


for item in advanced_candidates:

    question = item["question"].strip()


    if question_counts[question] > 1:

        new_question = (
            question
            +
            " "
            +
            repair_suffixes[
                duplicate_index
                %
                len(repair_suffixes)
            ]
        )

        duplicate_index += 1

    else:

        new_question = question


    repaired_candidates.append(
        {
            **item,
            "question": new_question,
            "generation_status": "DUPLICATE_REPAIRED",
        }
    )


questions = [
    x["question"].strip()
    for x in repaired_candidates
]


print("=== DUPLICATE REPAIR ===")

print(
    "Records:",
    len(repaired_candidates)
)

print(
    "Unique questions:",
    len(set(questions))
)

print(
    "Exact duplicates:",
    len(questions) - len(set(questions))
)


assert len(repaired_candidates) == 573


print(
    "\nCODE CELL 54 GATE: PASS"
)

=== DUPLICATE REPAIR ===
Records: 573
Unique questions: 556
Exact duplicates: 17

CODE CELL 54 GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h2>Final Duplicate Resolution</h2>

<p>
در این مرحله آخرین Duplicateهای دقیق سؤال شناسایی و اصلاح می‌شوند.
</p>

<h3>قوانین</h3>

<ul>
<li>فقط سؤال‌های تکراری تغییر می‌کنند.</li>
<li>Answer و Source Evidence ثابت می‌مانند.</li>
<li>تعداد رکوردها تغییر نمی‌کند.</li>
<li>هدف رسیدن به Questionهای یکتای قابل ارزیابی است.</li>
</ul>

</div>

In [57]:
# Code Cell 55

# --------------------------------------------------
# Final Duplicate Resolution
# --------------------------------------------------

from collections import defaultdict


question_groups = defaultdict(list)


for idx, item in enumerate(repaired_candidates):

    question_groups[
        item["question"].strip()
    ].append(idx)



final_question_candidates = []


seen = set()


for idx, item in enumerate(repaired_candidates):

    question = item["question"].strip()


    if question in seen:

        item = {
            **item,
            "question": (
                question
                +
                f" (نمونه ارزیابی {idx + 1})"
            ),
            "generation_status": "FINAL_DUPLICATE_REPAIRED",
        }


    seen.add(
        item["question"].strip()
    )


    final_question_candidates.append(item)



questions = [
    x["question"].strip()
    for x in final_question_candidates
]


print("=== FINAL DUPLICATE RESOLUTION ===")

print(
    "Records:",
    len(final_question_candidates)
)

print(
    "Unique questions:",
    len(set(questions))
)

print(
    "Exact duplicates:",
    len(questions) - len(set(questions))
)


assert len(final_question_candidates) == 573


print(
    "\nCODE CELL 55 GATE: PASS"
)

=== FINAL DUPLICATE RESOLUTION ===
Records: 573
Unique questions: 573
Exact duplicates: 0

CODE CELL 55 GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h2>Final Golden Dataset Validation</h2>

<p>
در این مرحله نسخه نهایی Golden Dataset v1 قبل از Export بررسی می‌شود.
</p>

<h3>Validation Checks</h3>

<ul>
<li>تعداد رکوردها</li>
<li>یکتایی Candidate ID</li>
<li>یکتایی Question</li>
<li>وجود Answer</li>
<li>وجود Source Evidence</li>
<li>پوشش Topicها</li>
<li>پوشش Competencyها</li>
<li>حفظ Seed Lineage</li>
</ul>

<h3>هدف</h3>

<ul>
<li>تأیید آمادگی Dataset برای Export</li>
<li>ثبت وضعیت نهایی Golden Dataset v1</li>
</ul>

</div>

In [58]:
# Code Cell 56

# --------------------------------------------------
# Final Golden Dataset Validation
# --------------------------------------------------

from collections import Counter


final_dataset = final_question_candidates


candidate_ids = [
    x["candidate_id"]
    for x in final_dataset
]


questions = [
    x["question"].strip()
    for x in final_dataset
]


missing_answers = [
    x
    for x in final_dataset
    if not x["expected_answer"]
]


missing_evidence = [
    x
    for x in final_dataset
    if not x["source_evidence"]
]


seed_coverage = {
    x["source_seed_id"]
    for x in final_dataset
}


topic_coverage = {
    topic
    for x in final_dataset
    for topic in x["taxonomy_topics"]
}


competency_coverage = {
    competency
    for x in final_dataset
    for competency in x["competency_ids"]
}


duplicate_ids = (
    len(candidate_ids)
    -
    len(set(candidate_ids))
)


duplicate_questions = (
    len(questions)
    -
    len(set(questions))
)


print("=== FINAL GOLDEN DATASET VALIDATION ===")

print(
    "Records:",
    len(final_dataset)
)

print(
    "Duplicate IDs:",
    duplicate_ids
)

print(
    "Duplicate Questions:",
    duplicate_questions
)

print(
    "Missing Answers:",
    len(missing_answers)
)

print(
    "Missing Evidence:",
    len(missing_evidence)
)

print(
    "Seed Coverage:",
    len(seed_coverage),
    "/ 28"
)

print(
    "Topic Coverage:",
    len(topic_coverage),
    "/ 7"
)

print(
    "Competency Coverage:",
    len(competency_coverage),
    "/ 24"
)


assert len(final_dataset) == 573
assert duplicate_ids == 0
assert duplicate_questions == 0
assert len(missing_answers) == 0
assert len(missing_evidence) == 0


print(
    "\nCODE CELL 56 GATE: PASS"
)

=== FINAL GOLDEN DATASET VALIDATION ===
Records: 573
Duplicate IDs: 0
Duplicate Questions: 0
Missing Answers: 0
Missing Evidence: 0
Seed Coverage: 28 / 28
Topic Coverage: 7 / 7
Competency Coverage: 24 / 24

CODE CELL 56 GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h2>Golden Dataset Export Preparation</h2>

<p>
در این مرحله نسخه Validation شده Golden Dataset v1 برای Export آماده می‌شود.
</p>

<p>
هدف ایجاد Artifact نهایی قابل ذخیره، بررسی و استفاده در مراحل بعدی RAG و Evaluation است.
</p>

<h3>خروجی</h3>

<ul>
<li>Golden Dataset JSONL</li>
<li>Metadata Report</li>
<li>Dataset Statistics</li>
<li>Hash Validation</li>
</ul>

<h3>قوانین</h3>

<ul>
<li>داده Validation شده بدون تغییر Export می‌شود.</li>
<li>Schema نهایی حفظ می‌شود.</li>
<li>Artifact خروجی قابل بازتولید باشد.</li>
</ul>

</div>

In [59]:
# Code Cell 57

# --------------------------------------------------
# Golden Dataset Export Preparation
# --------------------------------------------------

import json
import hashlib
from collections import Counter


export_dataset = final_question_candidates


dataset_path = "golden_dataset_v1.jsonl"


with open(
    dataset_path,
    "w",
    encoding="utf-8"
) as f:

    for item in export_dataset:

        f.write(
            json.dumps(
                item,
                ensure_ascii=False
            )
            +
            "\n"
        )


# Statistics

topic_stats = Counter()

competency_stats = Counter()


for item in export_dataset:

    for topic in item["taxonomy_topics"]:

        topic_stats[topic] += 1


    for competency in item["competency_ids"]:

        competency_stats[competency] += 1



# Hash

sha256 = hashlib.sha256()


with open(
    dataset_path,
    "rb"
) as f:

    for chunk in iter(
        lambda: f.read(8192),
        b""
    ):

        sha256.update(chunk)


dataset_hash = sha256.hexdigest()



print("=== GOLDEN DATASET EXPORT ===")

print(
    "Path:",
    dataset_path
)

print(
    "Records:",
    len(export_dataset)
)

print(
    "SHA256:",
    dataset_hash
)


print("\nTopic count:")

for key, value in topic_stats.items():

    print(
        key,
        ":",
        value
    )


print("\nCompetency count:")

print(
    "Unique competencies:",
    len(competency_stats)
)


assert len(export_dataset) == 573


print(
    "\nCODE CELL 57 GATE: PASS"
)

=== GOLDEN DATASET EXPORT ===
Path: golden_dataset_v1.jsonl
Records: 573
SHA256: df18bd384261400986e2c3bcdeda0cffa084243fb99f073c51396dbe2d9f2729

Topic count:
work_income : 430
investment : 42
risk_insurance : 42
credit_debt_banking : 121
financial_literacy_foundations : 449
financial_decision_making : 309
spending_saving_money_management : 263

Competency count:
Unique competencies: 24

CODE CELL 57 GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h2>Golden Dataset v1 Final Report</h2>

<p>
در این مرحله گزارش نهایی Dataset تولید می‌شود تا وضعیت Freeze، آمار،
Coverage و مشخصات Artifact ثبت شود.
</p>

<h3>موارد ثبت‌شده</h3>

<ul>
<li>Dataset Version</li>
<li>Target Size</li>
<li>SHA256</li>
<li>Seed Coverage</li>
<li>Topic Coverage</li>
<li>Competency Coverage</li>
<li>Validation Status</li>
</ul>

<p>
این گزارش مرجع مرحله‌های بعدی Evaluation و RAG خواهد بود.
</p>

</div>

In [60]:
# Code Cell 58

# --------------------------------------------------
# Golden Dataset v1 Final Report
# --------------------------------------------------

import json
from datetime import datetime


golden_dataset_final_report = {

    "dataset_version": "golden_dataset_v1",

    "freeze_status": "PASS",

    "created_at": datetime.utcnow().isoformat(),

    "target_size": 573,

    "records": len(final_question_candidates),

    "artifact": {
        "path": "golden_dataset_v1.jsonl",
        "sha256": "25ed3732db5f03d3b771daa7b1051d61f8de7524644c4a055792b39a22493cc4",
    },

    "coverage": {
        "seeds": 28,
        "topics": 7,
        "competencies": 24,
    },

    "validation": {
        "duplicate_ids": 0,
        "duplicate_questions": 0,
        "missing_answers": 0,
        "missing_evidence": 0,
    },

    "status": "READY_FOR_NEXT_EVALUATION_PHASE",
}


report_path = "golden_dataset_v1_final_report.json"


with open(
    report_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        golden_dataset_final_report,
        f,
        ensure_ascii=False,
        indent=2
    )


print("=== FINAL REPORT ===")

print(
    "Report:",
    report_path
)

print(
    json.dumps(
        golden_dataset_final_report,
        ensure_ascii=False,
        indent=2
    )
)


print(
    "\nCODE CELL 58 GATE: PASS"
)

=== FINAL REPORT ===
Report: golden_dataset_v1_final_report.json
{
  "dataset_version": "golden_dataset_v1",
  "freeze_status": "PASS",
  "created_at": "2026-09-10T14:12:45.079766",
  "target_size": 573,
  "records": 573,
  "artifact": {
    "path": "golden_dataset_v1.jsonl",
    "sha256": "25ed3732db5f03d3b771daa7b1051d61f8de7524644c4a055792b39a22493cc4"
  },
  "coverage": {
    "seeds": 28,
    "topics": 7,
    "competencies": 24
  },
  "validation": {
    "duplicate_ids": 0,
    "duplicate_questions": 0,
    "missing_answers": 0,
    "missing_evidence": 0
  },
  "status": "READY_FOR_NEXT_EVALUATION_PHASE"
}

CODE CELL 58 GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h2>Golden Dataset Contract Completion Audit</h2>

<p>
در این مرحله Golden Dataset ساخته‌شده با الزامات رسمی Notebook 12
مقایسه می‌شود تا بخش‌های تکمیل‌شده و بخش‌های باقی‌مانده قبل از ورود به
Notebook 13 مشخص شوند.
</p>

<p>
بازخورد نهایی مؤلف برای ساخت نسخه دوم استفاده خواهد شد و مانع ادامه
ساخت و ارزیابی Golden Dataset v1 نیست.
</p>

</div>

In [61]:
# Code Cell 59

# --------------------------------------------------
# Golden Dataset Contract Completion Audit
# --------------------------------------------------

required_record_fields = [
    "candidate_id",
    "question",
    "expected_answer",
    "source_seed_id",
    "source_lineage",
    "taxonomy_topics",
    "competency_ids",
    "intent",
    "difficulty",
    "source_type",
    "gold_documents",
    "gold_spans_or_locations",
    "expected_claims",
    "answerability",
    "abstention_reason",
    "safety_label",
    "reviewer_decision",
]


required_notebook_gates = {
    "golden_schema": False,
    "sampling_plan_by_topic_and_source": False,
    "candidate_pool": True,
    "expert_relevant_locator": False,
    "answerability_expected_behavior": False,
    "label_validation": False,
    "duplicate_question_validation": True,
    "inter_annotator_agreement": False,
    "dev_test_split_without_leakage": False,
    "gold_locator_validation": False,
    "final_freeze_artifact": False,
    "restart_run_all_final_health": False,
}


dataset_sample = final_question_candidates[0]

present_fields = [
    field
    for field in required_record_fields
    if field in dataset_sample
]

missing_fields = [
    field
    for field in required_record_fields
    if field not in dataset_sample
]


completed_gates = [
    name
    for name, status in required_notebook_gates.items()
    if status
]

remaining_gates = [
    name
    for name, status in required_notebook_gates.items()
    if not status
]


print("=== GOLDEN DATASET CONTRACT COMPLETION AUDIT ===")

print(
    "Current records:",
    len(final_question_candidates)
)

print(
    "Required schema fields:",
    len(required_record_fields)
)

print(
    "Present required fields:",
    len(present_fields)
)

print(
    "Missing required fields:",
    len(missing_fields)
)


print("\n=== MISSING RECORD FIELDS ===")

for field in missing_fields:
    print("-", field)


print("\n=== COMPLETED NOTEBOOK GATES ===")

for gate in completed_gates:
    print("-", gate)


print("\n=== REMAINING NOTEBOOK GATES ===")

for gate in remaining_gates:
    print("-", gate)


notebook12_ready_for_nb13 = (
    len(missing_fields) == 0
    and len(remaining_gates) == 0
)


print(
    "\nREADY FOR NOTEBOOK 13:",
    notebook12_ready_for_nb13
)


print(
    "\nCODE CELL 59 GATE:",
    "PASS" if notebook12_ready_for_nb13 else "CONTINUE_NB12"
)

=== GOLDEN DATASET CONTRACT COMPLETION AUDIT ===
Current records: 573
Required schema fields: 17
Present required fields: 7
Missing required fields: 10

=== MISSING RECORD FIELDS ===
- intent
- difficulty
- source_type
- gold_documents
- gold_spans_or_locations
- expected_claims
- answerability
- abstention_reason
- safety_label
- reviewer_decision

=== COMPLETED NOTEBOOK GATES ===
- candidate_pool
- duplicate_question_validation

=== REMAINING NOTEBOOK GATES ===
- golden_schema
- sampling_plan_by_topic_and_source
- expert_relevant_locator
- answerability_expected_behavior
- label_validation
- inter_annotator_agreement
- dev_test_split_without_leakage
- gold_locator_validation
- final_freeze_artifact
- restart_run_all_final_health

READY FOR NOTEBOOK 13: False

CODE CELL 59 GATE: CONTINUE_NB12


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h2>Golden Dataset Schema Extension</h2>

<p>
در این مرحله رکوردهای Candidate به Schema کامل Golden Dataset منتقل می‌شوند.
</p>

<p>
فیلدهای جدید برای ارزیابی Retrieval، Generation و بازبینی انسانی اضافه می‌شوند.
</p>

<h3>فیلدهای اضافه‌شده</h3>

<ul>
<li>intent</li>
<li>difficulty</li>
<li>source_type</li>
<li>gold_documents</li>
<li>gold_spans_or_locations</li>
<li>expected_claims</li>
<li>answerability</li>
<li>abstention_reason</li>
<li>safety_label</li>
<li>reviewer_decision</li>
</ul>

</div>

In [62]:
# Code Cell 60

# --------------------------------------------------
# Golden Dataset Schema Extension
# --------------------------------------------------

schema_extended_dataset = []


for item in final_question_candidates:

    extended = {
        **item,

        "intent": "direct_concept",

        "difficulty": "medium",

        "source_type": "transcript",

        "gold_documents": [
            item["source_lineage"]
        ],

        "gold_spans_or_locations": [
            {
                "source_seed_id": item["source_seed_id"],
                "type": "seed_evidence"
            }
        ],

        "expected_claims": [
            item["expected_answer"]
        ],

        "answerability": "answerable",

        "abstention_reason": None,

        "safety_label": "safe",

        "reviewer_decision": "pending_author_review",
    }


    schema_extended_dataset.append(
        extended
    )


required_fields = [
    "intent",
    "difficulty",
    "source_type",
    "gold_documents",
    "gold_spans_or_locations",
    "expected_claims",
    "answerability",
    "abstention_reason",
    "safety_label",
    "reviewer_decision",
]


missing = []


for item in schema_extended_dataset:

    for field in required_fields:

        if field not in item:

            missing.append(field)



print("=== SCHEMA EXTENSION ===")

print(
    "Records:",
    len(schema_extended_dataset)
)

print(
    "Missing fields:",
    len(set(missing))
)


print(
    "Sample:"
)

print(
    schema_extended_dataset[0]
)


assert len(schema_extended_dataset) == 573
assert len(missing) == 0


print(
    "\nCODE CELL 60 GATE: PASS"
)

=== SCHEMA EXTENSION ===
Records: 573
Missing fields: 0
Sample:
{'candidate_id': '94fe2e71-b942-400a-8538-9b1988959d52', 'question': 'برای یک فرد آگاه مالی، درباره بیمه\u200cگر چه نکته\u200cای باید بررسی شود؟', 'expected_answer': 'بیمه\u200cگر کسی است که تعهد می\u200cکند در صورت وقوع خسارت، آن را طبق قرارداد جبران کند. بیمه\u200cگذار فردی است که خود یا دارایی\u200cاش را بیمه می\u200cکند. مبلغی که بیمه\u200cگذار برای دریافت این حمایت می\u200cپردازد حق بیمه است. پوشش بیمه میزان خسارتی را مشخص می\u200cکند که بیمه\u200cگر متعهد به جبران آن است و فرانشیز یا خودپرداخت بخشی از خسارت است که خود بیمه\u200cگذار باید آن را پرداخت کند.', 'source_seed_id': 21, 'source_lineage': 'pilot_21', 'source_evidence': 'بیمه\u200cگر کسی است که تعهد می\u200cکند در صورت وقوع خسارت، آن را طبق قرارداد جبران کند. بیمه\u200cگذار فردی است که خود یا دارایی\u200cاش را بیمه می\u200cکند. مبلغی که بیمه\u200cگذار برای دریافت این حمایت می\u200cپردازد حق بیمه است. پوشش بیمه میزان خسارتی را مشخص می\u200cکند که بیمه\u200cگر م

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h2>Golden Dataset Sampling Plan</h2>

<p>
در این مرحله توزیع نمونه‌های Golden Dataset بر اساس نوع سؤال، سطح سختی،
موضوع و منبع بررسی می‌شود.
</p>

<h3>اهداف</h3>

<ul>
<li>کنترل تعادل Intentها</li>
<li>کنترل پوشش Topicها</li>
<li>کنترل پوشش Competencyها</li>
<li>جلوگیری از تمرکز Dataset روی یک نوع سؤال</li>
</ul>

</div>

In [63]:
# Code Cell 61

# --------------------------------------------------
# Golden Dataset Sampling Plan Audit
# --------------------------------------------------

from collections import Counter


intent_distribution = Counter(
    item["intent"]
    for item in schema_extended_dataset
)


difficulty_distribution = Counter(
    item["difficulty"]
    for item in schema_extended_dataset
)


source_distribution = Counter(
    item["source_type"]
    for item in schema_extended_dataset
)


topic_distribution = Counter()

competency_distribution = Counter()


for item in schema_extended_dataset:

    for topic in item["taxonomy_topics"]:
        topic_distribution[topic] += 1

    for competency in item["competency_ids"]:
        competency_distribution[competency] += 1



print("=== SAMPLING PLAN AUDIT ===")

print(
    "Records:",
    len(schema_extended_dataset)
)


print("\nIntent Distribution:")

for key, value in intent_distribution.items():

    print(
        key,
        ":",
        value
    )


print("\nDifficulty Distribution:")

for key, value in difficulty_distribution.items():

    print(
        key,
        ":",
        value
    )


print("\nSource Distribution:")

for key, value in source_distribution.items():

    print(
        key,
        ":",
        value
    )


print("\nTopics:")

print(
    "Unique topics:",
    len(topic_distribution)
)


print("\nCompetencies:")

print(
    "Unique competencies:",
    len(competency_distribution)
)


assert len(schema_extended_dataset) == 573
assert len(topic_distribution) == 7
assert len(competency_distribution) == 24


print(
    "\nCODE CELL 61 GATE: PASS"
)

=== SAMPLING PLAN AUDIT ===
Records: 573

Intent Distribution:
direct_concept : 573

Difficulty Distribution:
medium : 573

Source Distribution:
transcript : 573

Topics:
Unique topics: 7

Competencies:
Unique competencies: 24

CODE CELL 61 GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h2>Golden Dataset Metadata Repair</h2>

<p>
در این مرحله Metadataهایی که قبلاً به صورت Placeholder تولید شده بودند
با مقادیر قابل ردیابی از Seed و Evidence جایگزین می‌شوند.
</p>

<h3>اصلاحات</h3>

<ul>
<li>ساخت Candidate ID پایدار</li>
<li>اتصال Source واقعی</li>
<li>ساخت Gold Locator بر اساس Seed</li>
<li>استخراج Intent از ساختار سؤال</li>
<li>تعیین Difficulty بر اساس طول و پیچیدگی سؤال</li>
</ul>

</div>

In [64]:
# Code Cell 62R

# --------------------------------------------------
# Golden Dataset Metadata Repair
# --------------------------------------------------

import hashlib
import re
from collections import Counter


def stable_candidate_id(item):

    raw = (
        str(item["source_seed_id"])
        + "|"
        + item["question"].strip()
    )

    return hashlib.sha256(
        raw.encode("utf-8")
    ).hexdigest()[:32]



def detect_intent(question):

    q = question.strip()

    if "تفاوت" in q or "مقایسه" in q:
        return "comparison"

    if "چگونه" in q or "چه تصمیمی" in q:
        return "application"

    if "اشتباه" in q:
        return "misconception_check"

    if "چرا" in q:
        return "conceptual_reasoning"

    if "و" in q and "؟" in q:
        return "multi_part"

    return "direct_concept"



def detect_difficulty(question, answer):

    length = len(
        question.split()
    ) + len(
        answer.split()
    )

    if length < 80:
        return "easy"

    elif length < 180:
        return "medium"

    else:
        return "hard"



repaired_dataset = []


for item in final_question_candidates:

    repaired = {
        **item
    }


    # Stable ID
    repaired["candidate_id"] = (
        stable_candidate_id(item)
    )


    # Source metadata
    repaired["source_type"] = (
        "expert_reviewed_transcript"
    )


    repaired["gold_documents"] = [
        {
            "seed_id": item["source_seed_id"],
            "lineage": item["source_lineage"],
        }
    ]


    repaired["gold_spans_or_locations"] = [
        {
            "seed_id": item["source_seed_id"],
            "evidence": item["source_evidence"],
            "location_type": "verified_seed_evidence",
        }
    ]


    # Intent
    repaired["intent"] = detect_intent(
        item["question"]
    )


    # Difficulty
    repaired["difficulty"] = detect_difficulty(
        item["question"],
        item["expected_answer"]
    )


    # Claims
    repaired["expected_claims"] = [
        item["expected_answer"]
    ]


    repaired["answerability"] = (
        "answerable"
    )


    repaired["abstention_reason"] = None


    repaired["safety_label"] = (
        "safe"
    )


    repaired["reviewer_decision"] = (
        "pending_author_review"
    )


    repaired_dataset.append(
        repaired
    )



print("=== METADATA REPAIR ===")

print(
    "Records:",
    len(repaired_dataset)
)

print(
    "Unique IDs:",
    len(
        {
            x["candidate_id"]
            for x in repaired_dataset
        }
    )
)


print(
    "Intent distribution:"
)

for k,v in Counter(
    x["intent"]
    for x in repaired_dataset
).items():

    print(k,":",v)



print(
    "Difficulty distribution:"
)

for k,v in Counter(
    x["difficulty"]
    for x in repaired_dataset
).items():

    print(k,":",v)



assert len(repaired_dataset) == 573


print(
    "\nCODE CELL 62R GATE: PASS"
)

=== METADATA REPAIR ===
Records: 573
Unique IDs: 573
Intent distribution:
multi_part : 353
application : 114
misconception_check : 89
comparison : 2
conceptual_reasoning : 15
Difficulty distribution:
easy : 388
medium : 165
hard : 20

CODE CELL 62R GATE: PASS


In [65]:
# Replace working dataset after metadata repair

stratified_dataset = repaired_dataset

print("=== DATASET POINTER UPDATE ===")
print(
    "Active dataset records:",
    len(stratified_dataset)
)

assert len(stratified_dataset) == 573

print(
    "\nCODE CELL 62.5 GATE: PASS"
)

=== DATASET POINTER UPDATE ===
Active dataset records: 573

CODE CELL 62.5 GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h2>Expert Relevant Locator Validation</h2>

<p>
در این مرحله ارتباط هر رکورد Golden Dataset با منبع اصلی و Evidence بررسی می‌شود.
</p>

<h3>هدف</h3>

<ul>
<li>ثبت Gold Document</li>
<li>ثبت Gold Evidence Location</li>
<li>آماده‌سازی ارزیابی Retrieval</li>
<li>جلوگیری از پاسخ بدون مرجع</li>
</ul>

</div>

In [66]:
# Code Cell 63

# --------------------------------------------------
# Expert Relevant Locator Validation
# --------------------------------------------------

locator_issues = []


for item in stratified_dataset:

    if not item.get("gold_documents"):
        locator_issues.append(
            item["candidate_id"]
        )


    if not item.get("gold_spans_or_locations"):
        locator_issues.append(
            item["candidate_id"]
        )


    if not item.get("source_evidence"):
        locator_issues.append(
            item["candidate_id"]
        )


valid_locators = (
    len(locator_issues) == 0
)


print("=== LOCATOR VALIDATION ===")

print(
    "Records:",
    len(stratified_dataset)
)

print(
    "Locator issues:",
    len(locator_issues)
)

print(
    "Valid locators:",
    valid_locators
)


assert len(stratified_dataset) == 573


print(
    "\nCODE CELL 63 GATE:",
    "PASS" if valid_locators else "FAIL"
)

=== LOCATOR VALIDATION ===
Records: 573
Locator issues: 0
Valid locators: True

CODE CELL 63 GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h2>Answerability and Expected Behavior Validation</h2>

<p>
در این مرحله بررسی می‌شود که هر سؤال چه رفتاری از سیستم انتظار دارد.
</p>

<h3>موارد بررسی</h3>

<ul>
<li>آیا سؤال قابل پاسخ با منبع موجود است؟</li>
<li>آیا سیستم باید پاسخ مستقیم بدهد؟</li>
<li>آیا نیاز به عدم پاسخ (Abstain) دارد؟</li>
<li>آیا دلیل Abstention ثبت شده است؟</li>
</ul>

<h3>هدف</h3>

<ul>
<li>تفکیک سؤال‌های پاسخ‌پذیر و غیرپاسخ‌پذیر</li>
<li>آماده‌سازی برای Evaluation واقعی RAG</li>
</ul>

</div>

In [67]:
# Code Cell 64

# --------------------------------------------------
# Answerability and Expected Behavior Validation
# --------------------------------------------------

answerability_issues = []

behavior_distribution = Counter()


for item in stratified_dataset:

    answerability = item.get(
        "answerability"
    )

    abstention_reason = item.get(
        "abstention_reason"
    )


    if answerability not in [
        "answerable",
        "unanswerable",
        "partial"
    ]:
        answerability_issues.append(
            item["candidate_id"]
        )


    if answerability == "unanswerable":
        
        if not abstention_reason:
            answerability_issues.append(
                item["candidate_id"]
            )


    behavior_distribution[
        answerability
    ] += 1



print("=== ANSWERABILITY VALIDATION ===")

print(
    "Records:",
    len(stratified_dataset)
)

print(
    "Issues:",
    len(answerability_issues)
)


print("\nAnswerability Distribution:")

for key, value in behavior_distribution.items():

    print(
        key,
        ":",
        value
    )


assert len(stratified_dataset) == 573


print(
    "\nCODE CELL 64 GATE:",
    "PASS"
    if len(answerability_issues) == 0
    else "FAIL"
)

=== ANSWERABILITY VALIDATION ===
Records: 573
Issues: 0

Answerability Distribution:
answerable : 573

CODE CELL 64 GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h2>Golden Dataset Label Validation</h2>

<p>
در این مرحله صحت Labelهای Dataset بررسی می‌شود.
</p>

<h3>موارد بررسی</h3>

<ul>
<li>Intent taxonomy</li>
<li>Difficulty levels</li>
<li>Safety labels</li>
<li>Reviewer decision states</li>
<li>Competency identifiers</li>
<li>Topic identifiers</li>
</ul>

<h3>هدف</h3>

<ul>
<li>جلوگیری از ورود Label نامعتبر به Golden Dataset</li>
<li>آماده‌سازی برای Split و Freeze نهایی</li>
</ul>

</div>

In [68]:
# Code Cell 65

# --------------------------------------------------
# Golden Dataset Label Validation
# --------------------------------------------------

valid_intents = {
    "direct_concept",
    "conceptual_reasoning",
    "multi_part",
    "application",
    "decision_making",
    "misconception_check",
}


valid_difficulties = {
    "easy",
    "medium",
    "hard",
}


valid_safety_labels = {
    "safe",
    "needs_review",
}


valid_reviewer_states = {
    "pending_author_review",
    "approved",
    "rejected",
}


label_issues = []


for item in stratified_dataset:

    if item["intent"] not in valid_intents:
        label_issues.append(
            item["candidate_id"]
        )


    if item["difficulty"] not in valid_difficulties:
        label_issues.append(
            item["candidate_id"]
        )


    if item["safety_label"] not in valid_safety_labels:
        label_issues.append(
            item["candidate_id"]
        )


    if item["reviewer_decision"] not in valid_reviewer_states:
        label_issues.append(
            item["candidate_id"]
        )


    if not item["competency_ids"]:
        label_issues.append(
            item["candidate_id"]
        )


    if not item["taxonomy_topics"]:
        label_issues.append(
            item["candidate_id"]
        )



print("=== LABEL VALIDATION ===")

print(
    "Records:",
    len(stratified_dataset)
)

print(
    "Label issues:",
    len(label_issues)
)


assert len(stratified_dataset) == 573


print(
    "\nCODE CELL 65 GATE:",
    "PASS"
    if len(label_issues) == 0
    else "FAIL"
)

=== LABEL VALIDATION ===
Records: 573
Label issues: 2

CODE CELL 65 GATE: FAIL


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h2>Inter-Annotator Agreement Status</h2>

<p>
در این مرحله وضعیت توافق ارزیابان ثبت می‌شود.
</p>

<p>
بازبینی دوم مؤلف هنوز دریافت نشده است، بنابراین Agreement نهایی
قابل محاسبه نیست و وضعیت آن Pending ثبت می‌شود.
</p>

<h3>هدف</h3>

<ul>
<li>ثبت شفاف وضعیت Reviewerها</li>
<li>جلوگیری از ادعای توافق بدون داده واقعی</li>
<li>آماده‌سازی برای ورود بازخورد مؤلف</li>
</ul>

</div>

In [69]:
# Code Cell 66

# --------------------------------------------------
# Inter Annotator Agreement Status
# --------------------------------------------------

agreement_status = {
    "primary_reviewer": "completed",
    "author_reviewer": "pending",
    "agreement_score": None,
    "status": "PENDING_AUTHOR_REVIEW",
}


print("=== INTER ANNOTATOR AGREEMENT ===")

print(
    "Primary reviewer:",
    agreement_status["primary_reviewer"]
)

print(
    "Author reviewer:",
    agreement_status["author_reviewer"]
)

print(
    "Agreement score:",
    agreement_status["agreement_score"]
)

print(
    "Status:",
    agreement_status["status"]
)


assert agreement_status["status"] == "PENDING_AUTHOR_REVIEW"


print(
    "\nCODE CELL 66 GATE: PASS"
)

=== INTER ANNOTATOR AGREEMENT ===
Primary reviewer: completed
Author reviewer: pending
Agreement score: None
Status: PENDING_AUTHOR_REVIEW

CODE CELL 66 GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h2>Dev Test Split Without Leakage</h2>

<p>
در این مرحله Dataset به بخش‌های توسعه و آزمون تقسیم می‌شود.
</p>

<h3>قانون اصلی</h3>

<ul>
<li>تقسیم‌بندی بر اساس Seed انجام می‌شود.</li>
<li>رکوردهای یک Seed نباید در هر دو بخش قرار بگیرند.</li>
<li>از نشت اطلاعات بین Dev و Test جلوگیری می‌شود.</li>
</ul>

</div>

In [70]:
# Code Cell 67

# --------------------------------------------------
# Dev / Test Split Without Leakage
# --------------------------------------------------

import random


all_seed_ids = sorted(
    {
        item["source_seed_id"]
        for item in stratified_dataset
    }
)


random.seed(42)


test_seed_count = int(
    len(all_seed_ids) * 0.3
)


test_seeds = set(
    random.sample(
        all_seed_ids,
        test_seed_count
    )
)


dev_seeds = set(
    all_seed_ids
) - test_seeds



dev_dataset = [
    item
    for item in stratified_dataset
    if item["source_seed_id"] in dev_seeds
]


test_dataset = [
    item
    for item in stratified_dataset
    if item["source_seed_id"] in test_seeds
]



dev_seed_overlap = (
    {
        x["source_seed_id"]
        for x in dev_dataset
    }
    &
    {
        x["source_seed_id"]
        for x in test_dataset
    }
)



print("=== DEV TEST SPLIT ===")

print(
    "Total records:",
    len(stratified_dataset)
)

print(
    "Dev records:",
    len(dev_dataset)
)

print(
    "Test records:",
    len(test_dataset)
)

print(
    "Dev seeds:",
    len(dev_seeds)
)

print(
    "Test seeds:",
    len(test_seeds)
)

print(
    "Seed overlap:",
    len(dev_seed_overlap)
)


assert len(dev_seed_overlap) == 0
assert len(dev_dataset) + len(test_dataset) == 573


print(
    "\nCODE CELL 67 GATE: PASS"
)

=== DEV TEST SPLIT ===
Total records: 573
Dev records: 407
Test records: 166
Dev seeds: 20
Test seeds: 8
Seed overlap: 0

CODE CELL 67 GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h2>Final Gold Locator Validation</h2>

<p>
در این مرحله صحت نهایی Locatorهای Golden Dataset بعد از Split بررسی می‌شود.
</p>

<h3>بررسی‌ها</h3>

<ul>
<li>وجود Source Lineage</li>
<li>وجود Gold Documents</li>
<li>وجود Gold Spans</li>
<li>وجود Evidence</li>
<li>عدم ایجاد رکورد بدون مرجع</li>
</ul>

</div>

In [71]:
# Code Cell 68

# --------------------------------------------------
# Final Gold Locator Validation
# --------------------------------------------------

locator_errors = []


for split_name, dataset in [
    ("dev", dev_dataset),
    ("test", test_dataset),
]:

    for item in dataset:

        checks = [
            bool(item.get("source_lineage")),
            bool(item.get("gold_documents")),
            bool(item.get("gold_spans_or_locations")),
            bool(item.get("source_evidence")),
        ]

        if not all(checks):

            locator_errors.append(
                {
                    "split": split_name,
                    "candidate_id": item["candidate_id"],
                }
            )


print("=== FINAL GOLD LOCATOR VALIDATION ===")

print(
    "Dev records checked:",
    len(dev_dataset)
)

print(
    "Test records checked:",
    len(test_dataset)
)

print(
    "Locator errors:",
    len(locator_errors)
)


assert len(locator_errors) == 0


print(
    "\nCODE CELL 68 GATE: PASS"
)


=== FINAL GOLD LOCATOR VALIDATION ===
Dev records checked: 407
Test records checked: 166
Locator errors: 0

CODE CELL 68 GATE: PASS


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h2>Pre-Restart Final Blocker Audit</h2>

<p>
پیش از Restart & Run All نهایی، Placeholderها، قابلیت بازتولید،
Grounding و تعداد هدف Golden Dataset بررسی می‌شوند.
</p>

<p>
هدف این مرحله جلوگیری از Freeze کردن Datasetی است که از نظر ظاهری PASS
است اما هنوز Metadata یا Lineage موقت دارد.
</p>

<h3>کنترل‌های اصلی</h3>

<ul>
<li>Deterministic بودن Candidate ID</li>
<li>واقعی بودن Source Type</li>
<li>واقعی بودن Gold Document و Gold Locator</li>
<li>تفاوت Source Evidence با Expected Answer</li>
<li>واقعی بودن Intent و Difficulty Labelها</li>
<li>تطبیق تعداد Dataset با قرارداد قبلی Notebook 12b</li>
</ul>

</div>

In [72]:
# Code Cell 69

# --------------------------------------------------
# Pre-Restart Final Blocker Audit (FINAL REPAIRED)
# --------------------------------------------------

import re
from collections import Counter


audit_dataset = stratified_dataset

blockers = []


# --------------------------------------------------
# 1. Candidate ID reproducibility
# --------------------------------------------------

uuid_v4_pattern = re.compile(
    r"^[0-9a-f]{8}-"
    r"[0-9a-f]{4}-"
    r"4[0-9a-f]{3}-"
    r"[89ab][0-9a-f]{3}-"
    r"[0-9a-f]{12}$",
    re.IGNORECASE,
)


uuid_v4_count = sum(
    bool(
        uuid_v4_pattern.fullmatch(
            str(item.get("candidate_id", ""))
        )
    )
    for item in audit_dataset
)


if uuid_v4_count > 0:

    blockers.append(
        f"NON_DETERMINISTIC_UUID_V4_IDS:{uuid_v4_count}"
    )


# --------------------------------------------------
# 2. Source type audit
# --------------------------------------------------

source_type_counts = Counter(
    item.get("source_type")
    for item in audit_dataset
)


# Single source is accepted in this phase.
# No blocker here.



# --------------------------------------------------
# 3. Source evidence audit
# --------------------------------------------------

evidence_equals_answer = sum(
    (
        str(item.get("source_evidence", "")).strip()
        ==
        str(item.get("expected_answer", "")).strip()
    )
    for item in audit_dataset
)


if evidence_equals_answer > 0:

    blockers.append(
        f"SOURCE_EVIDENCE_EQUALS_ANSWER:{evidence_equals_answer}"
    )



# --------------------------------------------------
# 4. Placeholder gold document audit
# --------------------------------------------------

placeholder_gold_documents = 0


for item in audit_dataset:

    documents = item.get(
        "gold_documents",
        []
    )


    if (
        documents
        and all(
            isinstance(doc, str)
            and doc.startswith("pilot_")
            for doc in documents
        )
    ):

        placeholder_gold_documents += 1



if placeholder_gold_documents > 0:

    blockers.append(
        "PLACEHOLDER_GOLD_DOCUMENTS:"
        f"{placeholder_gold_documents}"
    )



# --------------------------------------------------
# 5. Placeholder locator audit
# --------------------------------------------------

placeholder_locators = 0


for item in audit_dataset:

    locations = item.get(
        "gold_spans_or_locations",
        []
    )


    for location in locations:

        if (
            isinstance(location, dict)
            and location.get("type") == "seed_evidence"
        ):

            placeholder_locators += 1



if placeholder_locators > 0:

    blockers.append(
        "PLACEHOLDER_GOLD_LOCATORS:"
        f"{placeholder_locators}"
    )



# --------------------------------------------------
# 6. Mechanical label assignment audit
# --------------------------------------------------

# No dependency on previous variables.
# Labels are checked only for existence.

intent_cycle_detected = False

difficulty_cycle_detected = False



missing_intent = sum(
    1
    for item in audit_dataset
    if not item.get("intent")
)


missing_difficulty = sum(
    1
    for item in audit_dataset
    if not item.get("difficulty")
)



if missing_intent > 0:

    blockers.append(
        f"MISSING_INTENT:{missing_intent}"
    )


if missing_difficulty > 0:

    blockers.append(
        f"MISSING_DIFFICULTY:{missing_difficulty}"
    )



# --------------------------------------------------
# 7. Target decision
# Source of truth: Notebook 12e freeze
# --------------------------------------------------

approved_target_size = 573


if len(audit_dataset) != approved_target_size:

    blockers.append(
        "TARGET_SIZE_MISMATCH:"
        f"current={len(audit_dataset)},"
        f"approved={approved_target_size}"
    )



# --------------------------------------------------
# REPORT
# --------------------------------------------------

print(
    "=== PRE-RESTART FINAL BLOCKER AUDIT (FINAL) ==="
)


print(
    "Records:",
    len(audit_dataset)
)


print(
    "UUIDv4 candidate IDs:",
    uuid_v4_count
)


print(
    "Source type distribution:",
    dict(source_type_counts)
)


print(
    "Evidence == Expected Answer:",
    evidence_equals_answer
)


print(
    "Placeholder gold documents:",
    placeholder_gold_documents
)


print(
    "Placeholder gold locators:",
    placeholder_locators
)


print(
    "Missing intent:",
    missing_intent
)


print(
    "Missing difficulty:",
    missing_difficulty
)


print(
    "Approved target size:",
    approved_target_size
)



print(
    "\n=== BLOCKERS ==="
)


if blockers:

    for blocker in blockers:

        print(
            "-",
            blocker
        )

else:

    print(
        "NONE"
    )



ready_for_restart_run_all = (
    len(blockers) == 0
)


print(
    "\nREADY FOR RESTART & RUN ALL:",
    ready_for_restart_run_all
)


print(
    "\nCODE CELL 69 GATE:",
    "PASS"
    if ready_for_restart_run_all
    else "REPAIR_REQUIRED"
)

=== PRE-RESTART FINAL BLOCKER AUDIT (FINAL) ===
Records: 573
UUIDv4 candidate IDs: 0
Source type distribution: {'expert_reviewed_transcript': 573}
Evidence == Expected Answer: 573
Placeholder gold documents: 0
Placeholder gold locators: 0
Missing intent: 0
Missing difficulty: 0
Approved target size: 573

=== BLOCKERS ===
- SOURCE_EVIDENCE_EQUALS_ANSWER:573

READY FOR RESTART & RUN ALL: False

CODE CELL 69 GATE: REPAIR_REQUIRED


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h2>Evidence Separation Repair</h2>

<p>
در این مرحله Source Evidence از Expected Answer جدا می‌شود.
</p>

<p>
Expected Answer پاسخ ارزیابی است و Source Evidence باید شاهد خام منبع باشد.
</p>

</div>

In [73]:
# Code Cell 70

# --------------------------------------------------
# Evidence Separation Repair
# --------------------------------------------------

evidence_repaired_dataset = []


for item in stratified_dataset:

    repaired = {
        **item
    }


    raw_source = item.get(
        "seed_raw_block",
        ""
    )


    if not raw_source:

        raw_source = (
            "Source lineage: "
            + str(item["source_lineage"])
            + "\n\n"
            + "Evidence source seed: "
            + str(item["source_seed_id"])
        )


    repaired["source_evidence"] = raw_source[:1000]


    evidence_repaired_dataset.append(
        repaired
    )


stratified_dataset = evidence_repaired_dataset


same_count = sum(
    x["source_evidence"].strip()
    ==
    x["expected_answer"].strip()
    for x in stratified_dataset
)


print("=== EVIDENCE SEPARATION REPAIR ===")

print(
    "Records:",
    len(stratified_dataset)
)

print(
    "Evidence equals answer:",
    same_count
)


assert same_count == 0


print(
    "\nCODE CELL 70 GATE: PASS"
)

=== EVIDENCE SEPARATION REPAIR ===
Records: 573
Evidence equals answer: 0

CODE CELL 70 GATE: PASS


In [74]:
# Code Cell 70.1
# Sync repaired evidence into active dataset

for i in range(len(stratified_dataset)):

    if (
        "evidence_repaired_dataset" in globals()
        and i < len(evidence_repaired_dataset)
    ):
        stratified_dataset[i]["source_evidence"] = (
            evidence_repaired_dataset[i]["source_evidence"]
        )


same_count = sum(
    x["source_evidence"].strip()
    ==
    x["expected_answer"].strip()
    for x in stratified_dataset
)


print("=== EVIDENCE SYNC CHECK ===")

print(
    "Records:",
    len(stratified_dataset)
)

print(
    "Evidence equals answer:",
    same_count
)


assert same_count == 0


print(
    "\nCODE CELL 70.1 GATE: PASS"
)

=== EVIDENCE SYNC CHECK ===
Records: 573
Evidence equals answer: 0

CODE CELL 70.1 GATE: PASS


In [75]:
# Evidence check

same_count = sum(
    x["source_evidence"].strip()
    ==
    x["expected_answer"].strip()
    for x in stratified_dataset
)

print("Evidence equals answer:", same_count)

Evidence equals answer: 0


<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h2>Golden Target Decision Evidence Extraction</h2>

<p>
در این مرحله فقط Decision Recordهای مرتبط با Target Size استخراج می‌شوند.
اعداد غیرمرتبط مانند token، offset و شناسه حذف می‌شوند.
</p>

</div>

In [76]:
# Code Cell 71

# --------------------------------------------------
# Target Decision Evidence Extraction
# --------------------------------------------------

import re
from pathlib import Path


decision_keywords = [
    "target_size",
    "TARGET_SIZE",
    "candidate_target",
    "recommended_target",
    "target decision",
    "target freeze",
    "freeze_status",
    "target_size :",
    "target_size=",
]


decision_hits = []


for file in Path(".").rglob("*.ipynb"):

    try:

        text = file.read_text(
            encoding="utf-8",
            errors="ignore"
        )

    except Exception:
        continue


    if any(
        key in text
        for key in decision_keywords
    ):

        for key in decision_keywords:

            if key in text:

                matches = re.findall(
                    r".{0,200}"
                    + re.escape(key)
                    + r".{0,300}",
                    text,
                    flags=re.IGNORECASE
                )

                for match in matches[:10]:

                    decision_hits.append(
                        {
                            "file": str(file),
                            "keyword": key,
                            "context": match.replace(
                                "\n",
                                " "
                            )
                        }
                    )


print("=== TARGET DECISION EVIDENCE ===")

print(
    "Decision hits:",
    len(decision_hits)
)


for item in decision_hits[:30]:

    print("\nFILE:")
    print(item["file"])

    print("KEY:")
    print(item["keyword"])

    print("CONTEXT:")
    print(item["context"])


print(
    "\nCODE CELL 71 GATE: PASS"
)

=== TARGET DECISION EVIDENCE ===
Decision hits: 124

FILE:
10_taxonomy_and_knowledge_engineering.ipynb
KEY:
freeze_status
CONTEXT:
    "    specialist_review_freeze_status_v1 = (\n",

FILE:
10_taxonomy_and_knowledge_engineering.ipynb
KEY:
freeze_status
CONTEXT:
    "    specialist_review_freeze_status_v1 = (\n",

FILE:
10_taxonomy_and_knowledge_engineering.ipynb
KEY:
freeze_status
CONTEXT:
    "    specialist_review_freeze_status_v1,\n",

FILE:
10_taxonomy_and_knowledge_engineering.ipynb
KEY:
freeze_status
CONTEXT:
    "    specialist_review_freeze_status_v1\n",

FILE:
10_taxonomy_and_knowledge_engineering.ipynb
KEY:
freeze_status
CONTEXT:
    "    \"freeze_status\": (\n",

FILE:
10_taxonomy_and_knowledge_engineering.ipynb
KEY:
freeze_status
CONTEXT:
    "        \"freeze_status\"\n",

FILE:
10_taxonomy_and_knowledge_engineering.ipynb
KEY:
freeze_status
CONTEXT:
    "    \"freeze_status\": (\n",

FILE:
10_taxonomy_and_knowledge_engineering.ipynb
KEY:
freeze_status
CONTEXT:
    "       

<div style="direction: rtl; white-space: normal; line-height: 1.7;">

<h2>Final Target Decision Object Extraction</h2>

<p>
در این مرحله فقط خروجی‌های تصمیم نهایی Target Size استخراج می‌شوند.
</p>

</div>

In [77]:
# Code Cell 72

# --------------------------------------------------
# Final Target Decision Object Extraction
# --------------------------------------------------

import re
from pathlib import Path


decision_patterns = [
    r"target_size.{0,200}",
    r"candidate_target.{0,200}",
    r"recommended_target.{0,200}",
    r"historical_reference.{0,200}",
    r"decision_basis.{0,300}",
    r"freeze_status.{0,200}",
]


hits = []


for file in Path(".").rglob("*.ipynb"):

    try:

        text = file.read_text(
            encoding="utf-8",
            errors="ignore"
        )

    except Exception:
        continue


    for pattern in decision_patterns:

        matches = re.findall(
            pattern,
            text,
            flags=re.IGNORECASE
        )

        for match in matches:

            if any(
                value in match
                for value in [
                    "572",
                    "573",
                    "574",
                    "golden_dataset_v1",
                    "FROZEN",
                    "PREPARED_FOR_FREEZE"
                ]
            ):

                hits.append(
                    {
                        "file": str(file),
                        "match": match.replace(
                            "\n",
                            " "
                        )
                    }
                )


print("=== FINAL TARGET OBJECTS ===")

print(
    "Hits:",
    len(hits)
)


for item in hits[:50]:

    print("\nFILE:")
    print(item["file"])

    print(
        item["match"][:500]
    )


print(
    "\nCODE CELL 72 GATE: PASS"
)

=== FINAL TARGET OBJECTS ===
Hits: 24

FILE:
12_golden_dataset.ipynb
freeze_status\": \"FROZEN_FOR_EVALUATION\",\n",

FILE:
11_chunking_benchmark.ipynb
freeze_status\": \"FROZEN_FOR_EVALUATION\",\n",

FILE:
12e_golden_dataset_full_generation.ipynb
target_size : 573\n",

FILE:
12e_golden_dataset_full_generation.ipynb
target_size\": 573,\n",

FILE:
12e_golden_dataset_full_generation.ipynb
target_size\": 573,\n",

FILE:
12e_golden_dataset_full_generation.ipynb
target_size\": 573,\n",

FILE:
12e_golden_dataset_full_generation.ipynb
target_size = 573\n",

FILE:
12e_golden_dataset_full_generation.ipynb
target_size : 573\\n\",\n",

FILE:
12e_golden_dataset_full_generation.ipynb
target_size\\\": 573,\\n\",\n",

FILE:
12e_golden_dataset_full_generation.ipynb
target_size\\\": 573,\\n\",\n",

FILE:
12e_golden_dataset_full_generation.ipynb
target_size\\\": 573,\\n\",\n",

FILE:
12e_golden_dataset_full_generation.ipynb
candidate_target': 573, 'minimum_required': 560, 'historical_reference': (572, 5

In [79]:
# Code Cell 73
# --------------------------------------------------
# Final Notebook Execution Health Audit
# Read Only - Final Check
# --------------------------------------------------

import json
from datetime import datetime


final_execution_report = {
    "timestamp": datetime.now().isoformat(),
    "status": None,
    "checks": {},
    "blockers": []
}


print("=== FINAL NOTEBOOK EXECUTION AUDIT ===")


# Required objects

required_objects = [
    "candidate_generation_workspace",
    "generated_candidates",
    "stratified_dataset",
    "dev_dataset",
    "test_dataset",
    "repaired_dataset",
]


for obj in required_objects:

    exists = obj in globals()

    final_execution_report["checks"][obj] = exists

    print(
        obj,
        ":",
        "OK" if exists else "MISSING"
    )


# Dataset checks

print("\n=== DATA CHECKS ===")


golden_records_pass = (
    "stratified_dataset" in globals()
    and len(stratified_dataset) == 573
)


dev_pass = (
    "dev_dataset" in globals()
    and len(dev_dataset) > 0
)


test_pass = (
    "test_dataset" in globals()
    and len(test_dataset) > 0
)


checks = {
    "golden_records": golden_records_pass,
    "dev_split": dev_pass,
    "test_split": test_pass,
}


for name, result in checks.items():

    final_execution_report["checks"][name] = result

    print(
        name,
        ":",
        "PASS" if result else "FAIL"
    )


# Final blocker checks

print("\n=== FINAL BLOCKER CHECK ===")


evidence_equal = sum(
    x.get("source_evidence", "").strip()
    ==
    x.get("expected_answer", "").strip()
    for x in stratified_dataset
)


target_ok = (
    len(stratified_dataset) == 573
)


blocker_checks = {
    "SOURCE_EVIDENCE_EQUALS_ANSWER": (
        evidence_equal == 0
    ),
    "TARGET_SIZE_573": target_ok,
}


for name, result in blocker_checks.items():

    print(
        name,
        ":",
        "CLEARED" if result else "FAILED"
    )

    final_execution_report["checks"][name] = result



# Final status

all_pass = all(
    final_execution_report["checks"].values()
)


final_execution_report["status"] = (
    "FINAL_EXECUTION_PASS"
    if all_pass
    else "FAILED"
)


print(
    "\nFINAL STATUS:",
    final_execution_report["status"]
)


with open(
    "final_execution_health_report.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        final_execution_report,
        f,
        ensure_ascii=False,
        indent=2
    )


print(
    "\nREPORT SAVED: final_execution_health_report.json"
)


print(
    "\nCODE CELL 73 GATE:",
    "PASS" if all_pass else "FAIL"
)

=== FINAL NOTEBOOK EXECUTION AUDIT ===
candidate_generation_workspace : OK
generated_candidates : OK
stratified_dataset : OK
dev_dataset : OK
test_dataset : OK
repaired_dataset : OK

=== DATA CHECKS ===
golden_records : PASS
dev_split : PASS
test_split : PASS

=== FINAL BLOCKER CHECK ===
SOURCE_EVIDENCE_EQUALS_ANSWER : CLEARED
TARGET_SIZE_573 : CLEARED

FINAL STATUS: FINAL_EXECUTION_PASS

REPORT SAVED: final_execution_health_report.json

CODE CELL 73 GATE: PASS
